# THz-TDS Lecture Notebook

This notebook is organized as truly section-local calculation and plotting blocks. Each section owns its own plotting code so you can customize one section without editing the others.

> Generated notebook notice
>
> This notebook is written by `docs/lecture_assets_v2.py` via `_write_lecture_notebook()`.
> If you want your plot or cell edits to persist, update the generator source instead of only editing this `.ipynb`.

## Imports

Only common imports and shared constants live here. The plotting logic itself is written inside each section.

In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from IPython import get_ipython
from IPython.display import display

_ip = get_ipython()
if _ip is not None:
    try:
        _ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass

repo_root = Path.cwd().resolve()
search_roots = [repo_root, *repo_root.parents]
for candidate in search_roots:
    if (candidate / 'docs' / 'generate_lecture_assets.py').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Could not locate the repo root containing docs/generate_lecture_assets.py')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from docs.generate_lecture_assets import build_notebook_demo_bundle, run_lecture_map_from_spec
from thzsim2 import (
    Drude,
    Fit,
    Layer,
    Measurement,
    ReferenceStandard,
    TraceData,
    drude_gamma_thz_from_tau_ps,
    drude_plasma_freq_thz_from_sigma_tau,
    prepare_reflection_first_peak_pair,
    prepare_trace_pair_for_fit,
    run_measured_fit,
    run_staged_measured_fit,
)
from thzsim2.core.fft import fft_t_to_w
from thzsim2.io.trace_csv import write_trace_csv

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#fbfaf7",
        "axes.edgecolor": "#2a2a2a",
        "axes.labelcolor": "#1f1f1f",
        "axes.titleweight": "bold",
        "axes.titlesize": 13.0,
        "axes.labelsize": 11.0,
        "xtick.labelsize": 10.0,
        "ytick.labelsize": 10.0,
        "grid.color": "#b8b8b8",
        "grid.alpha": 0.20,
        "grid.linestyle": "--",
        "legend.frameon": False,
        "font.family": "STIXGeneral",
        "mathtext.fontset": "stix",
        "savefig.bbox": "tight",
    }
)

VALUE_LABELS = {
    "recovery_error": r"$\mathcal{E}_{\mathrm{rec}}$",
    "weighted_data_fit": r"$\mathcal{J}_{w}$",
    "data_fit": r"$\mathcal{J}$",
    "relative_l2": r"$\mathcal{L}_{2,\mathrm{rel}}$",
    "residual_rms": r"$\mathrm{RMS}(r)$",
    "max_abs_residual": r"$\max |r|$",
}
PARAMETER_LABELS = {
    "film_thickness_um": r"$d$ ($\mu$m)",
    "film_plasma_freq_thz": r"$\omega_{p}$ (THz)",
    "film_gamma_thz": r"$\gamma$ (THz)",
    "epi_thickness_um": r"$d_{\mathrm{epi}}$ ($\mu$m)",
    "oxide_thickness_um": r"$d_{\mathrm{ox}}$ ($\mu$m)",
    "epi_plasma1_thz": r"$\omega_{p1}$ (THz)",
    "epi_gamma1_thz": r"$\gamma_1$ (THz)",
    "epi_plasma2_thz": r"$\omega_{p2}$ (THz)",
    "epi_gamma2_thz": r"$\gamma_2$ (THz)",
    "wafer_thickness_um": r"$d$ ($\mu$m)",
    "wafer_plasma_freq_thz": r"$\omega_p$ (THz)",
    "wafer_gamma_thz": r"$\gamma$ (THz)",
    "delta_t_ps": r"$\Delta t$ (ps)",
    "tau_ps": r"$\tau$ (ps)",
    "sigma_s_per_m": r"$\sigma$ (S/m)",
    "tau1_ps": r"$\tau_1$ (ps)",
    "tau2_ps": r"$\tau_2$ (ps)",
    "sigma1_s_per_m": r"$\sigma_1$ (S/m)",
    "sigma2_s_per_m": r"$\sigma_2$ (S/m)",
}

def metric_label(value_key):
    value_key = str(value_key)
    if value_key in VALUE_LABELS:
        return VALUE_LABELS[value_key]
    if value_key.startswith("signed_err_"):
        parameter_key = value_key[len("signed_err_") :]
        parameter_label = PARAMETER_LABELS.get(parameter_key, parameter_key)
        return rf"$({parameter_label})_{{\mathrm{{true}}}} - ({parameter_label})_{{\mathrm{{fit}}}}$"
    if value_key.startswith("abs_err_"):
        parameter_key = value_key[len("abs_err_") :]
        parameter_label = PARAMETER_LABELS.get(parameter_key, parameter_key)
        return rf"$|({parameter_label})_{{\mathrm{{true}}}} - ({parameter_label})_{{\mathrm{{fit}}}}|$"
    return value_key

def print_parameter_mapping(title, mapping):
    print(title)
    if not mapping:
        print("  (none)")
        return
    for key in sorted(mapping):
        value = mapping[key]
        if isinstance(value, (int, float, np.integer, np.floating)) and np.isfinite(float(value)):
            print(f"  {key}: {float(value):.6g}")
        else:
            print(f"  {key}: {value}")

## Smoke Build

Use this first if you want a compact end-to-end validation of the lecture asset pipeline.

In [ ]:
smoke_result = build_notebook_demo_bundle()
smoke_result['output_root']

## Measured Transmission Example

The next cell calculates the measured transmission example and saves the section data. The plotting cell after that contains the full figure code for this section only.

In [ ]:
measured_tx_output = repo_root / 'docs' / 'lecture_build' / 'notebook_measured_transmission'
measured_tx_output.mkdir(parents=True, exist_ok=True)
measured_tx_data_dir = measured_tx_output / 'data'
measured_tx_data_dir.mkdir(parents=True, exist_ok=True)

def _measured_tx_json_ready(value):
    if isinstance(value, dict):
        return {str(key): _measured_tx_json_ready(inner) for key, inner in value.items()}
    if isinstance(value, (list, tuple)):
        return [_measured_tx_json_ready(item) for item in value]
    if isinstance(value, Path):
        return value.resolve().as_posix()
    if isinstance(value, np.ndarray):
        return [_measured_tx_json_ready(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

measured_tx_root = repo_root / 'Test_data_for_fitter' / 'A11013460_transmission'
measured_tx_prepared = prepare_trace_pair_for_fit(
    measured_tx_root / 'REFERENCE.csv',
    measured_tx_root / 'SAMPLE1.csv',
    baseline_mode='auto_pre_pulse',
    baseline_window_samples=40,
    crop_mode='auto',
)

measured_tx_sigma_s_per_m = 100.0 / 67.0
measured_tx_tau_ps = 0.25
measured_tx_layers = [
    Layer(
        name='wafer',
        thickness_um=Fit(625.0, abs_min=600.0, abs_max=650.0, label='wafer_thickness_um'),
        material=Drude(
            eps_inf=11.7,
            plasma_freq_thz=Fit(
                drude_plasma_freq_thz_from_sigma_tau(measured_tx_sigma_s_per_m, measured_tx_tau_ps),
                abs_min=0.05,
                abs_max=3.5,
                label='wafer_plasma_freq_thz',
            ),
            gamma_thz=Fit(
                drude_gamma_thz_from_tau_ps(measured_tx_tau_ps),
                abs_min=0.1,
                abs_max=2.5,
                label='wafer_gamma_thz',
            ),
        ),
    )
]

measured_tx_staged = run_staged_measured_fit(
    measured_tx_prepared,
    measured_tx_layers,
    out_dir=measured_tx_output / 'runs' / 'measured_a11013460_tx',
    measurement=Measurement(
        mode='transmission',
        angle_deg=0.0,
        polarization='s',
        reference_standard=ReferenceStandard(kind='ambient_replacement'),
    ),
    weighting={'mode': 'trace_amplitude', 'floor': 0.03, 'power': 2.0, 'smooth_window_samples': 41},
    delay_options={'enabled': True, 'search_window_ps': 20.0},
    reflection_counts=(0, 2, 4, 8),
)

measured_tx_fit_result = measured_tx_staged['best_fit_result']
measured_tx_time_ps = np.asarray(measured_tx_prepared.processed_sample.time_ps, dtype=np.float64)
measured_tx_reference_trace = np.asarray(measured_tx_prepared.processed_reference.trace, dtype=np.float64)
measured_tx_observed_trace = np.asarray(measured_tx_prepared.processed_sample.trace, dtype=np.float64)
measured_tx_fit_trace = np.asarray(measured_tx_fit_result['fitted_simulation']['sample_trace'], dtype=np.float64)
measured_tx_residual_trace = np.asarray(measured_tx_fit_result['residual_trace'], dtype=np.float64)

write_trace_csv(
    measured_tx_data_dir / 'measured_transmission_reference_trace.csv',
    TraceData(
        time_ps=measured_tx_time_ps.copy(),
        trace=measured_tx_reference_trace.copy(),
        source_kind='processed_reference',
        metadata=_measured_tx_json_ready(measured_tx_prepared.processed_reference.metadata),
    ),
)
write_trace_csv(
    measured_tx_data_dir / 'measured_transmission_observed_trace.csv',
    TraceData(
        time_ps=measured_tx_time_ps.copy(),
        trace=measured_tx_observed_trace.copy(),
        source_kind='processed_sample',
        metadata=_measured_tx_json_ready(measured_tx_prepared.processed_sample.metadata),
    ),
)
write_trace_csv(
    measured_tx_data_dir / 'measured_transmission_fit_trace.csv',
    TraceData(
        time_ps=measured_tx_time_ps.copy(),
        trace=measured_tx_fit_trace.copy(),
        source_kind='fit_sample',
        metadata={},
    ),
)
write_trace_csv(
    measured_tx_data_dir / 'measured_transmission_residual_trace.csv',
    TraceData(
        time_ps=measured_tx_time_ps.copy(),
        trace=measured_tx_residual_trace.copy(),
        source_kind='fit_residual',
        metadata={},
    ),
)

measured_tx_corr_rows = []
measured_tx_corr_matrix = measured_tx_fit_result.get('parameter_correlation')
measured_tx_parameter_names = list(measured_tx_fit_result.get('parameter_names', []))
if measured_tx_corr_matrix is not None and measured_tx_parameter_names:
    measured_tx_corr_matrix = np.asarray(measured_tx_corr_matrix, dtype=np.float64)
    for measured_tx_i, measured_tx_name_i in enumerate(measured_tx_parameter_names):
        for measured_tx_j, measured_tx_name_j in enumerate(measured_tx_parameter_names):
            measured_tx_corr_rows.append(
                {
                    'param_i': str(measured_tx_name_i),
                    'param_j': str(measured_tx_name_j),
                    'correlation': float(measured_tx_corr_matrix[measured_tx_i, measured_tx_j]),
                }
            )

with (measured_tx_data_dir / 'measured_transmission_correlation_rows.csv').open('w', newline='', encoding='utf-8') as measured_tx_handle:
    measured_tx_writer = csv.DictWriter(measured_tx_handle, fieldnames=['param_i', 'param_j', 'correlation'])
    measured_tx_writer.writeheader()
    for measured_tx_row in measured_tx_corr_rows:
        measured_tx_writer.writerow(measured_tx_row)

measured_tx_summary = {
    'dataset': 'A11013460_transmission',
    'sample': 'SAMPLE1.csv',
    'selection_reason': str(measured_tx_staged['selection_reason']),
    'selection_score': float(measured_tx_staged['selection_score']),
    'prepared_metadata': _measured_tx_json_ready(measured_tx_prepared.metadata),
    'residual_metrics': {
        key: float(value)
        for key, value in measured_tx_fit_result['residual_metrics'].items()
        if np.isfinite(float(value))
    },
    'recovered_parameters': {
        key: float(value)
        for key, value in measured_tx_fit_result['recovered_parameters'].items()
    },
}
(measured_tx_data_dir / 'measured_transmission_summary.json').write_text(
    json.dumps(measured_tx_summary, indent=2, ensure_ascii=True),
    encoding='utf-8',
)

measured_tx_bundle = {
    'output_root': measured_tx_output.resolve().as_posix(),
    'prepared_traces': measured_tx_prepared,
    'fit_result': measured_tx_fit_result,
    'selection_reason': measured_tx_staged['selection_reason'],
    'selection_score': measured_tx_staged['selection_score'],
    'data_paths': {
        'summary_json': (measured_tx_data_dir / 'measured_transmission_summary.json').resolve().as_posix(),
        'reference_trace_csv': (measured_tx_data_dir / 'measured_transmission_reference_trace.csv').resolve().as_posix(),
        'observed_trace_csv': (measured_tx_data_dir / 'measured_transmission_observed_trace.csv').resolve().as_posix(),
        'fit_trace_csv': (measured_tx_data_dir / 'measured_transmission_fit_trace.csv').resolve().as_posix(),
        'residual_trace_csv': (measured_tx_data_dir / 'measured_transmission_residual_trace.csv').resolve().as_posix(),
        'correlation_rows_csv': (measured_tx_data_dir / 'measured_transmission_correlation_rows.csv').resolve().as_posix(),
    },
}
print_parameter_mapping('Measured transmission recovered parameters', measured_tx_fit_result.get('recovered_parameters', {}))
print()
print_parameter_mapping('Measured transmission fitted measurement', measured_tx_fit_result.get('fitted_measurement', {}))
measured_tx_bundle['data_paths']

Edit only this section's plotting cell if you want the measured transmission figures to look different from the other sections.

In [ ]:
measured_tx_plot_title = 'Measured transmission fit: A11013460'
measured_tx_corr_title = 'Measured transmission fit: local parameter correlation'
measured_tx_figure_root = Path(measured_tx_bundle['output_root']) / 'figures'
measured_tx_figure_root.mkdir(parents=True, exist_ok=True)

def _measured_tx_positive_spectrum(trace, time_ps):
    measured_tx_dt_s = float(np.median(np.diff(np.asarray(time_ps, dtype=np.float64)))) * 1e-12
    measured_tx_t0_s = float(np.asarray(time_ps, dtype=np.float64)[0]) * 1e-12
    measured_tx_omega, measured_tx_spectrum = fft_t_to_w(np.asarray(trace, dtype=np.float64), dt=measured_tx_dt_s, t0=measured_tx_t0_s)
    measured_tx_freq_thz = measured_tx_omega / (2.0 * np.pi * 1e12)
    measured_tx_mask = measured_tx_freq_thz > 0.0
    return np.asarray(measured_tx_freq_thz[measured_tx_mask], dtype=np.float64), np.asarray(measured_tx_spectrum[measured_tx_mask], dtype=np.complex128)

def _measured_tx_relative_db(values, floor_db=-110.0):
    measured_tx_values = np.asarray(values, dtype=np.float64)
    measured_tx_reference = max(float(np.max(measured_tx_values)), 1e-30)
    measured_tx_floor = 10.0 ** (float(floor_db) / 20.0)
    return 20.0 * np.log10(np.maximum(measured_tx_values / measured_tx_reference, measured_tx_floor))

measured_tx_observed = np.asarray(measured_tx_bundle['prepared_traces'].processed_sample.trace, dtype=np.float64)
measured_tx_fitted = np.asarray(measured_tx_bundle['fit_result']['fitted_simulation']['sample_trace'], dtype=np.float64)
measured_tx_residual = np.asarray(measured_tx_bundle['fit_result']['residual_trace'], dtype=np.float64)
measured_tx_time_ps = np.asarray(measured_tx_bundle['prepared_traces'].processed_sample.time_ps, dtype=np.float64)

measured_tx_freq_thz, measured_tx_observed_spec = _measured_tx_positive_spectrum(measured_tx_observed, measured_tx_time_ps)
_, measured_tx_fitted_spec = _measured_tx_positive_spectrum(measured_tx_fitted, measured_tx_time_ps)
measured_tx_observed_amp = _measured_tx_relative_db(np.abs(measured_tx_observed_spec))
measured_tx_fitted_amp = _measured_tx_relative_db(np.abs(measured_tx_fitted_spec))
measured_tx_observed_phase = np.unwrap(np.angle(measured_tx_observed_spec))
measured_tx_fitted_phase = np.unwrap(np.angle(measured_tx_fitted_spec))

measured_tx_overview_fig, measured_tx_overview_axes = plt.subplots(2, 2, figsize=(12.2, 8.2))
measured_tx_overview_axes[0, 0].plot(measured_tx_time_ps, measured_tx_observed, linewidth=1.9, color='#0f4c81', label='Observed')
measured_tx_overview_axes[0, 0].plot(measured_tx_time_ps, measured_tx_fitted, linewidth=1.55, color='#d1495b', label='Fit')
measured_tx_overview_axes[0, 0].set_title('Time-domain waveform')
measured_tx_overview_axes[0, 0].set_xlabel(r'$t$ (ps)')
measured_tx_overview_axes[0, 0].set_ylabel(r'$E(t)$')
measured_tx_overview_axes[0, 0].legend(loc='upper left')

measured_tx_overview_axes[0, 1].plot(measured_tx_time_ps, measured_tx_residual, linewidth=1.45, color='#3b7d3a')
measured_tx_overview_axes[0, 1].axhline(0.0, color='#555555', linewidth=0.9, alpha=0.7)
measured_tx_overview_axes[0, 1].set_title('Residual trace')
measured_tx_overview_axes[0, 1].set_xlabel(r'$t$ (ps)')
measured_tx_overview_axes[0, 1].set_ylabel(r'$r(t)=E_{fit}(t)-E_{data}(t)$')

measured_tx_overview_axes[1, 0].plot(measured_tx_freq_thz, measured_tx_observed_amp, linewidth=1.85, color='#0f4c81', label='Observed')
measured_tx_overview_axes[1, 0].plot(measured_tx_freq_thz, measured_tx_fitted_amp, linewidth=1.45, color='#d1495b', label='Fit')
measured_tx_overview_axes[1, 0].set_title('Spectral amplitude')
measured_tx_overview_axes[1, 0].set_xlabel(r'$f$ (THz)')
measured_tx_overview_axes[1, 0].set_ylabel('Amplitude (dB)')
measured_tx_overview_axes[1, 0].set_xlim(0.05, min(3.0, float(measured_tx_freq_thz[-1])))
measured_tx_overview_axes[1, 0].legend(loc='upper right')

measured_tx_overview_axes[1, 1].plot(measured_tx_freq_thz, measured_tx_observed_phase, linewidth=1.85, color='#0f4c81', label='Observed')
measured_tx_overview_axes[1, 1].plot(measured_tx_freq_thz, measured_tx_fitted_phase, linewidth=1.45, color='#d1495b', label='Fit')
measured_tx_overview_axes[1, 1].set_title('Unwrapped spectral phase')
measured_tx_overview_axes[1, 1].set_xlabel(r'$f$ (THz)')
measured_tx_overview_axes[1, 1].set_ylabel(r'$\phi(f)$ (rad)')
measured_tx_overview_axes[1, 1].set_xlim(0.05, min(3.0, float(measured_tx_freq_thz[-1])))
measured_tx_overview_axes[1, 1].legend(loc='upper right')

for measured_tx_axis in measured_tx_overview_axes.flat:
    measured_tx_axis.grid(True, alpha=0.18)

measured_tx_overview_fig.suptitle(measured_tx_plot_title, fontsize=15, fontweight='bold')
measured_tx_overview_fig.tight_layout()
measured_tx_overview_png = measured_tx_figure_root / 'measured_transmission_overview_section.png'
measured_tx_overview_pdf = measured_tx_figure_root / 'measured_transmission_overview_section.pdf'
measured_tx_overview_fig.savefig(measured_tx_overview_png, dpi=220)
measured_tx_overview_fig.savefig(measured_tx_overview_pdf)
display(measured_tx_overview_fig)
plt.close(measured_tx_overview_fig)

measured_tx_corr_names = list(measured_tx_bundle['fit_result'].get('parameter_names', []))
measured_tx_corr_matrix = measured_tx_bundle['fit_result'].get('parameter_correlation')
if measured_tx_corr_matrix is None or not measured_tx_corr_names:
    raise ValueError('No parameter correlation matrix is available for the measured transmission example.')
measured_tx_corr_matrix = np.asarray(measured_tx_corr_matrix, dtype=np.float64)

measured_tx_corr_fig, measured_tx_corr_ax = plt.subplots(figsize=(4.8, 4.2))
measured_tx_corr_image = measured_tx_corr_ax.imshow(measured_tx_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
measured_tx_corr_ax.set_title(measured_tx_corr_title)
measured_tx_corr_ax.set_xticks(range(len(measured_tx_corr_names)))
measured_tx_corr_ax.set_yticks(range(len(measured_tx_corr_names)))
measured_tx_corr_ax.set_xticklabels(
    [PARAMETER_LABELS.get(name, name) for name in measured_tx_corr_names],
    rotation=45,
    ha='right',
)
measured_tx_corr_ax.set_yticklabels([PARAMETER_LABELS.get(name, name) for name in measured_tx_corr_names])
measured_tx_corr_fig.colorbar(measured_tx_corr_image, ax=measured_tx_corr_ax, label=r'$\rho_{ij}$')
measured_tx_corr_fig.tight_layout()
measured_tx_corr_png = measured_tx_figure_root / 'measured_transmission_correlation_section.png'
measured_tx_corr_pdf = measured_tx_figure_root / 'measured_transmission_correlation_section.pdf'
measured_tx_corr_fig.savefig(measured_tx_corr_png, dpi=220)
measured_tx_corr_fig.savefig(measured_tx_corr_pdf)
display(measured_tx_corr_fig)
plt.close(measured_tx_corr_fig)

measured_tx_bundle['figure_paths'] = {
    'overview_png': measured_tx_overview_png.resolve().as_posix(),
    'overview_pdf': measured_tx_overview_pdf.resolve().as_posix(),
    'correlation_png': measured_tx_corr_png.resolve().as_posix(),
    'correlation_pdf': measured_tx_corr_pdf.resolve().as_posix(),
}
measured_tx_bundle['figure_paths']

## Measured Reflection Example

The next cell calculates the measured reflection example and saves the section data. The plotting cell after that contains the full figure code for this section only.

In [ ]:
measured_refl_output = repo_root / 'docs' / 'lecture_build' / 'notebook_measured_reflection'
measured_refl_output.mkdir(parents=True, exist_ok=True)
measured_refl_data_dir = measured_refl_output / 'data'
measured_refl_data_dir.mkdir(parents=True, exist_ok=True)

def _measured_refl_json_ready(value):
    if isinstance(value, dict):
        return {str(key): _measured_refl_json_ready(inner) for key, inner in value.items()}
    if isinstance(value, (list, tuple)):
        return [_measured_refl_json_ready(item) for item in value]
    if isinstance(value, Path):
        return value.resolve().as_posix()
    if isinstance(value, np.ndarray):
        return [_measured_refl_json_ready(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

measured_refl_reference_path = (
    repo_root
    / 'Test_data_for_fitter'
    / 'A11013460_reflection'
    / 'reflection_setup_ref_after_with_AuMirror_A11013460_avg600_onDryAir10min_int56.csv'
)
measured_refl_sample_path = (
    repo_root
    / 'Test_data_for_fitter'
    / 'A11013460_reflection'
    / 'reflection_setup_sample_A11013460_avg600_onDryAir10min_int30.csv'
)
measured_refl_prepared = prepare_trace_pair_for_fit(
    measured_refl_reference_path,
    measured_refl_sample_path,
    baseline_mode='auto_pre_pulse',
    baseline_window_samples=40,
    crop_mode='auto',
)

measured_refl_sigma_s_per_m = 100.0 / 67.0
measured_refl_tau_ps = 0.25
measured_refl_layers = [
    Layer(
        name='wafer',
        thickness_um=Fit(625.0, abs_min=575.0, abs_max=675.0, label='wafer_thickness_um'),
        material=Drude(
            eps_inf=Fit(11.7, abs_min=4.0, abs_max=20.0, label='wafer_eps_inf'),
            plasma_freq_thz=Fit(
                drude_plasma_freq_thz_from_sigma_tau(measured_refl_sigma_s_per_m, measured_refl_tau_ps),
                abs_min=0.01,
                abs_max=4.0,
                label='wafer_plasma_freq_thz',
            ),
            gamma_thz=Fit(
                drude_gamma_thz_from_tau_ps(measured_refl_tau_ps),
                abs_min=0.01,
                abs_max=4.0,
                label='wafer_gamma_thz',
            ),
        ),
    )
]

measured_refl_result = run_measured_fit(
    measured_refl_prepared,
    measured_refl_layers,
    out_dir=measured_refl_output / 'runs' / 'measured_a11013460_reflection',
    measurement=Measurement(
        mode='reflection',
        angle_deg=Fit(10.0, abs_min=0.0, abs_max=45.0, label='measurement_angle_deg'),
        polarization='mixed',
        polarization_mix=Fit(0.5, abs_min=0.0, abs_max=1.0, label='measurement_polarization_mix'),
        trace_scale=Fit(-1.0, abs_min=-2.5, abs_max=0.5, label='measurement_trace_scale'),
        trace_offset=Fit(0.0, abs_min=-1.0, abs_max=1.0, label='measurement_trace_offset'),
        reference_standard=ReferenceStandard(kind='identity'),
    ),
    optimizer={
        'method': 'L-BFGS-B',
        'options': {'maxiter': 140},
        'global_options': {'maxiter': 10, 'popsize': 10, 'seed': 123},
        'fd_rel_step': 1e-5,
    },
    max_internal_reflections=2,
    delay_options={'enabled': True, 'search_window_ps': 20.0, 'initial_ps': 0.0},
    weighting={'mode': 'trace_amplitude', 'floor': 0.03, 'power': 2.0, 'smooth_window_samples': 41},
    metric='weighted_data_fit',
)

measured_refl_fit_result = measured_refl_result.fit_result
measured_refl_time_ps = np.asarray(measured_refl_prepared.processed_sample.time_ps, dtype=np.float64)
measured_refl_reference_trace = np.asarray(measured_refl_prepared.processed_reference.trace, dtype=np.float64)
measured_refl_observed_trace = np.asarray(measured_refl_prepared.processed_sample.trace, dtype=np.float64)
measured_refl_fit_trace = np.asarray(measured_refl_fit_result['fitted_simulation']['sample_trace'], dtype=np.float64)
measured_refl_residual_trace = np.asarray(measured_refl_fit_result['residual_trace'], dtype=np.float64)

write_trace_csv(
    measured_refl_data_dir / 'measured_reflection_reference_trace.csv',
    TraceData(
        time_ps=measured_refl_time_ps.copy(),
        trace=measured_refl_reference_trace.copy(),
        source_kind='processed_reference',
        metadata=_measured_refl_json_ready(measured_refl_prepared.processed_reference.metadata),
    ),
)
write_trace_csv(
    measured_refl_data_dir / 'measured_reflection_observed_trace.csv',
    TraceData(
        time_ps=measured_refl_time_ps.copy(),
        trace=measured_refl_observed_trace.copy(),
        source_kind='processed_sample',
        metadata=_measured_refl_json_ready(measured_refl_prepared.processed_sample.metadata),
    ),
)
write_trace_csv(
    measured_refl_data_dir / 'measured_reflection_fit_trace.csv',
    TraceData(
        time_ps=measured_refl_time_ps.copy(),
        trace=measured_refl_fit_trace.copy(),
        source_kind='fit_sample',
        metadata={},
    ),
)
write_trace_csv(
    measured_refl_data_dir / 'measured_reflection_residual_trace.csv',
    TraceData(
        time_ps=measured_refl_time_ps.copy(),
        trace=measured_refl_residual_trace.copy(),
        source_kind='fit_residual',
        metadata={},
    ),
)

measured_refl_corr_rows = []
measured_refl_corr_matrix = measured_refl_fit_result.get('parameter_correlation')
measured_refl_parameter_names = list(measured_refl_fit_result.get('parameter_names', []))
if measured_refl_corr_matrix is not None and measured_refl_parameter_names:
    measured_refl_corr_matrix = np.asarray(measured_refl_corr_matrix, dtype=np.float64)
    for measured_refl_i, measured_refl_name_i in enumerate(measured_refl_parameter_names):
        for measured_refl_j, measured_refl_name_j in enumerate(measured_refl_parameter_names):
            measured_refl_corr_rows.append(
                {
                    'param_i': str(measured_refl_name_i),
                    'param_j': str(measured_refl_name_j),
                    'correlation': float(measured_refl_corr_matrix[measured_refl_i, measured_refl_j]),
                }
            )

with (measured_refl_data_dir / 'measured_reflection_correlation_rows.csv').open('w', newline='', encoding='utf-8') as measured_refl_handle:
    measured_refl_writer = csv.DictWriter(measured_refl_handle, fieldnames=['param_i', 'param_j', 'correlation'])
    measured_refl_writer.writeheader()
    for measured_refl_row in measured_refl_corr_rows:
        measured_refl_writer.writerow(measured_refl_row)

measured_refl_summary = {
    'dataset': 'A11013460_reflection',
    'reference': measured_refl_reference_path.name,
    'sample': measured_refl_sample_path.name,
    'construction': 'mirror_reference_pair',
    'prepared_metadata': _measured_refl_json_ready(measured_refl_prepared.metadata),
    'residual_metrics': {
        key: float(value)
        for key, value in measured_refl_fit_result['residual_metrics'].items()
        if np.isfinite(float(value))
    },
    'max_abs_residual': float(np.max(np.abs(measured_refl_residual_trace))),
    'fitted_measurement': {
        key: (float(value) if value is not None else None)
        for key, value in measured_refl_fit_result['fitted_measurement'].items()
        if key in {'angle_deg', 'polarization_mix', 'trace_scale', 'trace_offset'}
    },
    'recovered_parameters': {
        key: float(value)
        for key, value in measured_refl_fit_result['recovered_parameters'].items()
    },
}
(measured_refl_data_dir / 'measured_reflection_summary.json').write_text(
    json.dumps(measured_refl_summary, indent=2, ensure_ascii=True),
    encoding='utf-8',
)

measured_refl_bundle = {
    'output_root': measured_refl_output.resolve().as_posix(),
    'prepared_traces': measured_refl_prepared,
    'fit_result': measured_refl_fit_result,
    'measured_fit_result': measured_refl_result,
    'data_paths': {
        'summary_json': (measured_refl_data_dir / 'measured_reflection_summary.json').resolve().as_posix(),
        'reference_trace_csv': (measured_refl_data_dir / 'measured_reflection_reference_trace.csv').resolve().as_posix(),
        'observed_trace_csv': (measured_refl_data_dir / 'measured_reflection_observed_trace.csv').resolve().as_posix(),
        'fit_trace_csv': (measured_refl_data_dir / 'measured_reflection_fit_trace.csv').resolve().as_posix(),
        'residual_trace_csv': (measured_refl_data_dir / 'measured_reflection_residual_trace.csv').resolve().as_posix(),
        'correlation_rows_csv': (measured_refl_data_dir / 'measured_reflection_correlation_rows.csv').resolve().as_posix(),
    },
}
print_parameter_mapping('Measured reflection recovered parameters', measured_refl_fit_result.get('recovered_parameters', {}))
print()
print_parameter_mapping('Measured reflection fitted measurement', measured_refl_fit_result.get('fitted_measurement', {}))
measured_refl_bundle['data_paths']

Edit only this section's plotting cell if you want the measured reflection figures to look different from the other sections.

In [ ]:
measured_refl_plot_title = 'Measured reflection fit: A11013460 mirror-reference pair'
measured_refl_corr_title = 'Measured reflection fit: local parameter correlation'
measured_refl_figure_root = Path(measured_refl_bundle['output_root']) / 'figures'
measured_refl_figure_root.mkdir(parents=True, exist_ok=True)

def _measured_refl_positive_spectrum(trace, time_ps):
    measured_refl_dt_s = float(np.median(np.diff(np.asarray(time_ps, dtype=np.float64)))) * 1e-12
    measured_refl_t0_s = float(np.asarray(time_ps, dtype=np.float64)[0]) * 1e-12
    measured_refl_omega, measured_refl_spectrum = fft_t_to_w(np.asarray(trace, dtype=np.float64), dt=measured_refl_dt_s, t0=measured_refl_t0_s)
    measured_refl_freq_thz = measured_refl_omega / (2.0 * np.pi * 1e12)
    measured_refl_mask = measured_refl_freq_thz > 0.0
    return np.asarray(measured_refl_freq_thz[measured_refl_mask], dtype=np.float64), np.asarray(measured_refl_spectrum[measured_refl_mask], dtype=np.complex128)

def _measured_refl_relative_db(values, floor_db=-110.0):
    measured_refl_values = np.asarray(values, dtype=np.float64)
    measured_refl_reference = max(float(np.max(measured_refl_values)), 1e-30)
    measured_refl_floor = 10.0 ** (float(floor_db) / 20.0)
    return 20.0 * np.log10(np.maximum(measured_refl_values / measured_refl_reference, measured_refl_floor))

measured_refl_reference = np.asarray(measured_refl_bundle['prepared_traces'].processed_reference.trace, dtype=np.float64)
measured_refl_observed = np.asarray(measured_refl_bundle['prepared_traces'].processed_sample.trace, dtype=np.float64)
measured_refl_fitted = np.asarray(measured_refl_bundle['fit_result']['fitted_simulation']['sample_trace'], dtype=np.float64)
measured_refl_residual = np.asarray(measured_refl_bundle['fit_result']['residual_trace'], dtype=np.float64)
measured_refl_time_ps = np.asarray(measured_refl_bundle['prepared_traces'].processed_sample.time_ps, dtype=np.float64)

measured_refl_freq_thz, measured_refl_observed_spec = _measured_refl_positive_spectrum(measured_refl_observed, measured_refl_time_ps)
_, measured_refl_fitted_spec = _measured_refl_positive_spectrum(measured_refl_fitted, measured_refl_time_ps)
measured_refl_observed_amp = _measured_refl_relative_db(np.abs(measured_refl_observed_spec))
measured_refl_fitted_amp = _measured_refl_relative_db(np.abs(measured_refl_fitted_spec))

measured_refl_overview_fig, measured_refl_overview_axes = plt.subplots(2, 2, figsize=(12.2, 8.2))
measured_refl_overview_axes[0, 0].plot(measured_refl_time_ps, measured_refl_reference, linewidth=1.75, color='#5b84b1', label='Measured Au-mirror reference')
measured_refl_overview_axes[0, 0].plot(measured_refl_time_ps, measured_refl_observed, linewidth=1.45, color='#d1495b', label='Observed wafer reflection')
measured_refl_overview_axes[0, 0].set_title('Mirror-reference pair')
measured_refl_overview_axes[0, 0].set_xlabel(r'$t$ (ps)')
measured_refl_overview_axes[0, 0].set_ylabel(r'$E(t)$')
measured_refl_overview_axes[0, 0].legend(loc='upper left')

measured_refl_overview_axes[0, 1].plot(measured_refl_time_ps, measured_refl_observed, linewidth=1.8, color='#0f4c81', label='Observed')
measured_refl_overview_axes[0, 1].plot(measured_refl_time_ps, measured_refl_fitted, linewidth=1.45, color='#d1495b', label='Fit')
measured_refl_overview_axes[0, 1].set_title('Reflection fit')
measured_refl_overview_axes[0, 1].set_xlabel(r'$t$ (ps)')
measured_refl_overview_axes[0, 1].set_ylabel(r'$E(t)$')
measured_refl_overview_axes[0, 1].legend(loc='upper left')

measured_refl_overview_axes[1, 0].plot(measured_refl_time_ps, measured_refl_residual, linewidth=1.45, color='#3b7d3a')
measured_refl_overview_axes[1, 0].axhline(0.0, color='#555555', linewidth=0.9, alpha=0.7)
measured_refl_overview_axes[1, 0].set_title('Residual trace')
measured_refl_overview_axes[1, 0].set_xlabel(r'$t$ (ps)')
measured_refl_overview_axes[1, 0].set_ylabel(r'$r(t)=E_{fit}(t)-E_{data}(t)$')

measured_refl_overview_axes[1, 1].plot(measured_refl_freq_thz, measured_refl_observed_amp, linewidth=1.85, color='#0f4c81', label='Observed')
measured_refl_overview_axes[1, 1].plot(measured_refl_freq_thz, measured_refl_fitted_amp, linewidth=1.45, color='#d1495b', label='Fit')
measured_refl_overview_axes[1, 1].set_title('Spectral amplitude')
measured_refl_overview_axes[1, 1].set_xlabel(r'$f$ (THz)')
measured_refl_overview_axes[1, 1].set_ylabel('Amplitude (dB)')
measured_refl_overview_axes[1, 1].set_xlim(0.05, min(3.0, float(measured_refl_freq_thz[-1])))
measured_refl_overview_axes[1, 1].legend(loc='upper right')

for measured_refl_axis in measured_refl_overview_axes.flat:
    measured_refl_axis.grid(True, alpha=0.18)

measured_refl_overview_fig.suptitle(measured_refl_plot_title, fontsize=15, fontweight='bold')
measured_refl_overview_fig.tight_layout()
measured_refl_overview_png = measured_refl_figure_root / 'measured_reflection_overview_section.png'
measured_refl_overview_pdf = measured_refl_figure_root / 'measured_reflection_overview_section.pdf'
measured_refl_overview_fig.savefig(measured_refl_overview_png, dpi=220)
measured_refl_overview_fig.savefig(measured_refl_overview_pdf)
display(measured_refl_overview_fig)
plt.close(measured_refl_overview_fig)

measured_refl_corr_names = list(measured_refl_bundle['fit_result'].get('parameter_names', []))
measured_refl_corr_matrix = measured_refl_bundle['fit_result'].get('parameter_correlation')
if measured_refl_corr_matrix is None or not measured_refl_corr_names:
    raise ValueError('No parameter correlation matrix is available for the measured reflection example.')
measured_refl_corr_matrix = np.asarray(measured_refl_corr_matrix, dtype=np.float64)

measured_refl_corr_fig, measured_refl_corr_ax = plt.subplots(figsize=(4.8, 4.2))
measured_refl_corr_image = measured_refl_corr_ax.imshow(measured_refl_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
measured_refl_corr_ax.set_title(measured_refl_corr_title)
measured_refl_corr_ax.set_xticks(range(len(measured_refl_corr_names)))
measured_refl_corr_ax.set_yticks(range(len(measured_refl_corr_names)))
measured_refl_corr_ax.set_xticklabels(
    [PARAMETER_LABELS.get(name, name) for name in measured_refl_corr_names],
    rotation=45,
    ha='right',
)
measured_refl_corr_ax.set_yticklabels([PARAMETER_LABELS.get(name, name) for name in measured_refl_corr_names])
measured_refl_corr_fig.colorbar(measured_refl_corr_image, ax=measured_refl_corr_ax, label=r'$\rho_{ij}$')
measured_refl_corr_fig.tight_layout()
measured_refl_corr_png = measured_refl_figure_root / 'measured_reflection_correlation_section.png'
measured_refl_corr_pdf = measured_refl_figure_root / 'measured_reflection_correlation_section.pdf'
measured_refl_corr_fig.savefig(measured_refl_corr_png, dpi=220)
measured_refl_corr_fig.savefig(measured_refl_corr_pdf)
display(measured_refl_corr_fig)
plt.close(measured_refl_corr_fig)

measured_refl_bundle['figure_paths'] = {
    'overview_png': measured_refl_overview_png.resolve().as_posix(),
    'overview_pdf': measured_refl_overview_pdf.resolve().as_posix(),
    'correlation_png': measured_refl_corr_png.resolve().as_posix(),
    'correlation_pdf': measured_refl_corr_pdf.resolve().as_posix(),
}
measured_refl_bundle['figure_paths']

## Quick Study Demo Blocks

Each demo has its own explicit calculation cell and its own explicit plotting cell.

### Quick One-Layer Demo

In [ ]:
quick_one_layer_tx_tau_sigma_low_s_0deg_spec = {
  "slug": "one_layer_tx_tau_sigma_low_s_0deg",
  "title": "One-layer Drude transmission: $\\tau$ vs $\\sigma$ (0 deg, s-pol, $d=525\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 0.0,
  "polarization": "s",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    0.01,
    1.0
  ],
  "thickness_um": 525.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
quick_one_layer_tx_tau_sigma_low_s_0deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'quick_one_layer_tx_tau_sigma_low_s_0deg'
quick_one_layer_tx_tau_sigma_low_s_0deg_output_root.mkdir(parents=True, exist_ok=True)
quick_one_layer_tx_tau_sigma_low_s_0deg_result = run_lecture_map_from_spec(quick_one_layer_tx_tau_sigma_low_s_0deg_spec, output_root=quick_one_layer_tx_tau_sigma_low_s_0deg_output_root, profile='smoke')
quick_one_layer_tx_tau_sigma_low_s_0deg_result['study_dir']

The next cell contains the full plotting code for this section. Change it here without affecting the other study sections.

In [ ]:
quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir = Path(quick_one_layer_tx_tau_sigma_low_s_0deg_result['study_dir'])
quick_one_layer_tx_tau_sigma_low_s_0deg_plot_title = quick_one_layer_tx_tau_sigma_low_s_0deg_result['title']
quick_one_layer_tx_tau_sigma_low_s_0deg_x_label = "$\\tau$ (ps)"
quick_one_layer_tx_tau_sigma_low_s_0deg_y_label = "$\\sigma$ (S/m)"

with (quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as quick_one_layer_tx_tau_sigma_low_s_0deg_summary_handle:
    quick_one_layer_tx_tau_sigma_low_s_0deg_summary_meta = json.load(quick_one_layer_tx_tau_sigma_low_s_0deg_summary_handle)
with (quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as quick_one_layer_tx_tau_sigma_low_s_0deg_corr_handle:
    quick_one_layer_tx_tau_sigma_low_s_0deg_corr_meta = json.load(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_handle)

quick_one_layer_tx_tau_sigma_low_s_0deg_rows = []
with (quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as quick_one_layer_tx_tau_sigma_low_s_0deg_summary_csv:
    quick_one_layer_tx_tau_sigma_low_s_0deg_reader = csv.DictReader(quick_one_layer_tx_tau_sigma_low_s_0deg_summary_csv)
    for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_reader:
        quick_one_layer_tx_tau_sigma_low_s_0deg_parsed = {}
        for quick_one_layer_tx_tau_sigma_low_s_0deg_key, quick_one_layer_tx_tau_sigma_low_s_0deg_value in quick_one_layer_tx_tau_sigma_low_s_0deg_row.items():
            if quick_one_layer_tx_tau_sigma_low_s_0deg_value is None:
                quick_one_layer_tx_tau_sigma_low_s_0deg_parsed[quick_one_layer_tx_tau_sigma_low_s_0deg_key] = quick_one_layer_tx_tau_sigma_low_s_0deg_value
                continue
            quick_one_layer_tx_tau_sigma_low_s_0deg_text = str(quick_one_layer_tx_tau_sigma_low_s_0deg_value).strip()
            if quick_one_layer_tx_tau_sigma_low_s_0deg_text == '':
                quick_one_layer_tx_tau_sigma_low_s_0deg_parsed[quick_one_layer_tx_tau_sigma_low_s_0deg_key] = quick_one_layer_tx_tau_sigma_low_s_0deg_text
                continue
            try:
                quick_one_layer_tx_tau_sigma_low_s_0deg_parsed[quick_one_layer_tx_tau_sigma_low_s_0deg_key] = float(quick_one_layer_tx_tau_sigma_low_s_0deg_text)
            except ValueError:
                quick_one_layer_tx_tau_sigma_low_s_0deg_parsed[quick_one_layer_tx_tau_sigma_low_s_0deg_key] = quick_one_layer_tx_tau_sigma_low_s_0deg_text
        quick_one_layer_tx_tau_sigma_low_s_0deg_rows.append(quick_one_layer_tx_tau_sigma_low_s_0deg_parsed)

quick_one_layer_tx_tau_sigma_low_s_0deg_x_key, quick_one_layer_tx_tau_sigma_low_s_0deg_y_key = list(quick_one_layer_tx_tau_sigma_low_s_0deg_summary_meta['recovery_keys'])
quick_one_layer_tx_tau_sigma_low_s_0deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{quick_one_layer_tx_tau_sigma_low_s_0deg_x_key}_true_minus_fit': f'signed_err_{quick_one_layer_tx_tau_sigma_low_s_0deg_x_key}',
    f'{quick_one_layer_tx_tau_sigma_low_s_0deg_y_key}_true_minus_fit': f'signed_err_{quick_one_layer_tx_tau_sigma_low_s_0deg_y_key}',
    f'abs_{quick_one_layer_tx_tau_sigma_low_s_0deg_x_key}_error': f'abs_err_{quick_one_layer_tx_tau_sigma_low_s_0deg_x_key}',
    f'abs_{quick_one_layer_tx_tau_sigma_low_s_0deg_y_key}_error': f'abs_err_{quick_one_layer_tx_tau_sigma_low_s_0deg_y_key}',
}
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key = quick_one_layer_tx_tau_sigma_low_s_0deg_metric_options['data_fit']
quick_one_layer_tx_tau_sigma_low_s_0deg_log_value_key = quick_one_layer_tx_tau_sigma_low_s_0deg_metric_options['data_fit']
quick_one_layer_tx_tau_sigma_low_s_0deg_x_values = sorted({float(quick_one_layer_tx_tau_sigma_low_s_0deg_row[quick_one_layer_tx_tau_sigma_low_s_0deg_x_key]) for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_rows})
quick_one_layer_tx_tau_sigma_low_s_0deg_y_values = sorted({float(quick_one_layer_tx_tau_sigma_low_s_0deg_row[quick_one_layer_tx_tau_sigma_low_s_0deg_y_key]) for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_rows})

print('Available map metrics:')
for quick_one_layer_tx_tau_sigma_low_s_0deg_option_name, quick_one_layer_tx_tau_sigma_low_s_0deg_option_key in quick_one_layer_tx_tau_sigma_low_s_0deg_metric_options.items():
    print(f"  {quick_one_layer_tx_tau_sigma_low_s_0deg_option_name}: {metric_label(quick_one_layer_tx_tau_sigma_low_s_0deg_option_key)}")
print(f"Current linear map: {metric_label(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key)}")
print(f"Current log map: {metric_label(quick_one_layer_tx_tau_sigma_low_s_0deg_log_value_key)}")

def quick_one_layer_tx_tau_sigma_low_s_0deg_aggregate_grid(value_key):
    quick_one_layer_tx_tau_sigma_low_s_0deg_grid = np.full((len(quick_one_layer_tx_tau_sigma_low_s_0deg_y_values), len(quick_one_layer_tx_tau_sigma_low_s_0deg_x_values)), np.nan, dtype=np.float64)
    for quick_one_layer_tx_tau_sigma_low_s_0deg_iy, quick_one_layer_tx_tau_sigma_low_s_0deg_y_value in enumerate(quick_one_layer_tx_tau_sigma_low_s_0deg_y_values):
        for quick_one_layer_tx_tau_sigma_low_s_0deg_ix, quick_one_layer_tx_tau_sigma_low_s_0deg_x_value in enumerate(quick_one_layer_tx_tau_sigma_low_s_0deg_x_values):
            quick_one_layer_tx_tau_sigma_low_s_0deg_cell = [
                float(quick_one_layer_tx_tau_sigma_low_s_0deg_row[value_key])
                for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_rows
                if math.isclose(float(quick_one_layer_tx_tau_sigma_low_s_0deg_row[quick_one_layer_tx_tau_sigma_low_s_0deg_x_key]), quick_one_layer_tx_tau_sigma_low_s_0deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(quick_one_layer_tx_tau_sigma_low_s_0deg_row[quick_one_layer_tx_tau_sigma_low_s_0deg_y_key]), quick_one_layer_tx_tau_sigma_low_s_0deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(quick_one_layer_tx_tau_sigma_low_s_0deg_row[value_key]))
            ]
            if quick_one_layer_tx_tau_sigma_low_s_0deg_cell:
                quick_one_layer_tx_tau_sigma_low_s_0deg_grid[quick_one_layer_tx_tau_sigma_low_s_0deg_iy, quick_one_layer_tx_tau_sigma_low_s_0deg_ix] = float(np.mean(quick_one_layer_tx_tau_sigma_low_s_0deg_cell))
    return quick_one_layer_tx_tau_sigma_low_s_0deg_grid

def quick_one_layer_tx_tau_sigma_low_s_0deg_positive_grid_for_log(grid):
    quick_one_layer_tx_tau_sigma_low_s_0deg_grid = np.asarray(grid, dtype=np.float64).copy()
    quick_one_layer_tx_tau_sigma_low_s_0deg_finite = quick_one_layer_tx_tau_sigma_low_s_0deg_grid[np.isfinite(quick_one_layer_tx_tau_sigma_low_s_0deg_grid)]
    quick_one_layer_tx_tau_sigma_low_s_0deg_positive = quick_one_layer_tx_tau_sigma_low_s_0deg_finite[quick_one_layer_tx_tau_sigma_low_s_0deg_finite > 0.0]
    if quick_one_layer_tx_tau_sigma_low_s_0deg_positive.size == 0:
        quick_one_layer_tx_tau_sigma_low_s_0deg_grid[np.isfinite(quick_one_layer_tx_tau_sigma_low_s_0deg_grid)] = 1.0
        return quick_one_layer_tx_tau_sigma_low_s_0deg_grid
    quick_one_layer_tx_tau_sigma_low_s_0deg_floor = max(float(np.min(quick_one_layer_tx_tau_sigma_low_s_0deg_positive)) * 0.5, 1e-18)
    quick_one_layer_tx_tau_sigma_low_s_0deg_grid[np.isfinite(quick_one_layer_tx_tau_sigma_low_s_0deg_grid) & (quick_one_layer_tx_tau_sigma_low_s_0deg_grid <= 0.0)] = quick_one_layer_tx_tau_sigma_low_s_0deg_floor
    return quick_one_layer_tx_tau_sigma_low_s_0deg_grid

quick_one_layer_tx_tau_sigma_low_s_0deg_linear_grid = quick_one_layer_tx_tau_sigma_low_s_0deg_aggregate_grid(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key)
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig, quick_one_layer_tx_tau_sigma_low_s_0deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_finite = quick_one_layer_tx_tau_sigma_low_s_0deg_linear_grid[np.isfinite(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_grid)]
if str(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key).startswith('signed_err_'):
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_limit = max(float(np.max(np.abs(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_finite))), 1e-18)
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin = -quick_one_layer_tx_tau_sigma_low_s_0deg_linear_limit
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax = quick_one_layer_tx_tau_sigma_low_s_0deg_linear_limit
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_levels = np.linspace(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin, quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax, 256)
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_cmap = 'plasma'
else:
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin = float(np.min(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_finite))
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax = float(np.max(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_finite)) + 1e-12
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_levels = np.linspace(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin, quick_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax, 256)
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_cmap = 'plasma'
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_contour = quick_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.contourf(
    np.asarray(quick_one_layer_tx_tau_sigma_low_s_0deg_x_values, dtype=np.float64),
    np.asarray(quick_one_layer_tx_tau_sigma_low_s_0deg_y_values, dtype=np.float64),
    quick_one_layer_tx_tau_sigma_low_s_0deg_linear_grid,
    levels=quick_one_layer_tx_tau_sigma_low_s_0deg_linear_levels,
    cmap=quick_one_layer_tx_tau_sigma_low_s_0deg_linear_cmap,
    extend='both',
)
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.set_title(quick_one_layer_tx_tau_sigma_low_s_0deg_plot_title + ' [linear]')
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.set_xlabel(quick_one_layer_tx_tau_sigma_low_s_0deg_x_label)
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.set_ylabel(quick_one_layer_tx_tau_sigma_low_s_0deg_y_label)
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_cbar = quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.colorbar(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_contour, ax=quick_one_layer_tx_tau_sigma_low_s_0deg_linear_ax)
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_cbar.set_label(metric_label(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key))
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.tight_layout()
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_png = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_linear.png"
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_pdf = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_linear.pdf"
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_png, dpi=220)
quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_pdf)

if str(quick_one_layer_tx_tau_sigma_low_s_0deg_log_value_key).startswith('signed_err_'):
    quick_one_layer_tx_tau_sigma_low_s_0deg_log_source_key = 'abs_err_' + str(quick_one_layer_tx_tau_sigma_low_s_0deg_log_value_key)[len('signed_err_'):]
else:
    quick_one_layer_tx_tau_sigma_low_s_0deg_log_source_key = quick_one_layer_tx_tau_sigma_low_s_0deg_log_value_key
quick_one_layer_tx_tau_sigma_low_s_0deg_log_grid = quick_one_layer_tx_tau_sigma_low_s_0deg_positive_grid_for_log(quick_one_layer_tx_tau_sigma_low_s_0deg_aggregate_grid(quick_one_layer_tx_tau_sigma_low_s_0deg_log_source_key))
quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig, quick_one_layer_tx_tau_sigma_low_s_0deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
quick_one_layer_tx_tau_sigma_low_s_0deg_log_finite = quick_one_layer_tx_tau_sigma_low_s_0deg_log_grid[np.isfinite(quick_one_layer_tx_tau_sigma_low_s_0deg_log_grid)]
quick_one_layer_tx_tau_sigma_low_s_0deg_log_positive = quick_one_layer_tx_tau_sigma_low_s_0deg_log_finite[quick_one_layer_tx_tau_sigma_low_s_0deg_log_finite > 0.0]
quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmin = float(np.min(quick_one_layer_tx_tau_sigma_low_s_0deg_log_positive))
quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmax = float(np.max(quick_one_layer_tx_tau_sigma_low_s_0deg_log_positive))
if math.isclose(quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmin, quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmax):
    quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmax = quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmin * 1.01
quick_one_layer_tx_tau_sigma_low_s_0deg_log_levels = np.geomspace(quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmin, quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmax, 256)
quick_one_layer_tx_tau_sigma_low_s_0deg_log_contour = quick_one_layer_tx_tau_sigma_low_s_0deg_log_ax.contourf(
    np.asarray(quick_one_layer_tx_tau_sigma_low_s_0deg_x_values, dtype=np.float64),
    np.asarray(quick_one_layer_tx_tau_sigma_low_s_0deg_y_values, dtype=np.float64),
    quick_one_layer_tx_tau_sigma_low_s_0deg_log_grid,
    levels=quick_one_layer_tx_tau_sigma_low_s_0deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmin, vmax=quick_one_layer_tx_tau_sigma_low_s_0deg_log_vmax),
    extend='both',
)
quick_one_layer_tx_tau_sigma_low_s_0deg_log_ax.set_title(quick_one_layer_tx_tau_sigma_low_s_0deg_plot_title + ' [log]')
quick_one_layer_tx_tau_sigma_low_s_0deg_log_ax.set_xlabel(quick_one_layer_tx_tau_sigma_low_s_0deg_x_label)
quick_one_layer_tx_tau_sigma_low_s_0deg_log_ax.set_ylabel(quick_one_layer_tx_tau_sigma_low_s_0deg_y_label)
quick_one_layer_tx_tau_sigma_low_s_0deg_log_cbar = quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig.colorbar(quick_one_layer_tx_tau_sigma_low_s_0deg_log_contour, ax=quick_one_layer_tx_tau_sigma_low_s_0deg_log_ax)
quick_one_layer_tx_tau_sigma_low_s_0deg_log_cbar.set_label(metric_label(quick_one_layer_tx_tau_sigma_low_s_0deg_log_source_key))
quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig.tight_layout()
quick_one_layer_tx_tau_sigma_low_s_0deg_log_png = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_log.png"
quick_one_layer_tx_tau_sigma_low_s_0deg_log_pdf = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_log.pdf"
quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_log_png, dpi=220)
quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_log_pdf)

quick_one_layer_tx_tau_sigma_low_s_0deg_corr_rows = quick_one_layer_tx_tau_sigma_low_s_0deg_corr_meta['rows']
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels = sorted({str(quick_one_layer_tx_tau_sigma_low_s_0deg_row['param_i']) for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_corr_rows} | {str(quick_one_layer_tx_tau_sigma_low_s_0deg_row['param_j']) for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_corr_rows})
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_index = {quick_one_layer_tx_tau_sigma_low_s_0deg_label: quick_one_layer_tx_tau_sigma_low_s_0deg_idx for quick_one_layer_tx_tau_sigma_low_s_0deg_idx, quick_one_layer_tx_tau_sigma_low_s_0deg_label in enumerate(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)}
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix = np.full((len(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels), len(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)), np.nan, dtype=np.float64)
for quick_one_layer_tx_tau_sigma_low_s_0deg_row in quick_one_layer_tx_tau_sigma_low_s_0deg_corr_rows:
    quick_one_layer_tx_tau_sigma_low_s_0deg_i = quick_one_layer_tx_tau_sigma_low_s_0deg_corr_index[str(quick_one_layer_tx_tau_sigma_low_s_0deg_row['param_i'])]
    quick_one_layer_tx_tau_sigma_low_s_0deg_j = quick_one_layer_tx_tau_sigma_low_s_0deg_corr_index[str(quick_one_layer_tx_tau_sigma_low_s_0deg_row['param_j'])]
    quick_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix[quick_one_layer_tx_tau_sigma_low_s_0deg_i, quick_one_layer_tx_tau_sigma_low_s_0deg_j] = float(quick_one_layer_tx_tau_sigma_low_s_0deg_row['correlation'])
    quick_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix[quick_one_layer_tx_tau_sigma_low_s_0deg_j, quick_one_layer_tx_tau_sigma_low_s_0deg_i] = float(quick_one_layer_tx_tau_sigma_low_s_0deg_row['correlation'])
for quick_one_layer_tx_tau_sigma_low_s_0deg_diag in range(len(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)):
    quick_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix[quick_one_layer_tx_tau_sigma_low_s_0deg_diag, quick_one_layer_tx_tau_sigma_low_s_0deg_diag] = 1.0

quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig, quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_image = quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.imshow(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_title(quick_one_layer_tx_tau_sigma_low_s_0deg_plot_title + ' [correlation]')
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_xticks(range(len(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)))
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_yticks(range(len(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)))
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(quick_one_layer_tx_tau_sigma_low_s_0deg_label, quick_one_layer_tx_tau_sigma_low_s_0deg_label) for quick_one_layer_tx_tau_sigma_low_s_0deg_label in quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels], rotation=45, ha='right')
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(quick_one_layer_tx_tau_sigma_low_s_0deg_label, quick_one_layer_tx_tau_sigma_low_s_0deg_label) for quick_one_layer_tx_tau_sigma_low_s_0deg_label in quick_one_layer_tx_tau_sigma_low_s_0deg_corr_labels])
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.colorbar(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_image, ax=quick_one_layer_tx_tau_sigma_low_s_0deg_corr_ax, label=r'$\rho_{ij}$')
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.tight_layout()
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_png = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_corr.png"
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_pdf = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_corr.pdf"
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_png, dpi=220)
quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_pdf)

quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig, quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for quick_one_layer_tx_tau_sigma_low_s_0deg_axis, quick_one_layer_tx_tau_sigma_low_s_0deg_image_path, quick_one_layer_tx_tau_sigma_low_s_0deg_panel_title in zip(
    quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_axes,
    (quick_one_layer_tx_tau_sigma_low_s_0deg_linear_png, quick_one_layer_tx_tau_sigma_low_s_0deg_log_png, quick_one_layer_tx_tau_sigma_low_s_0deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    quick_one_layer_tx_tau_sigma_low_s_0deg_image = plt.imread(str(quick_one_layer_tx_tau_sigma_low_s_0deg_image_path))
    quick_one_layer_tx_tau_sigma_low_s_0deg_axis.imshow(quick_one_layer_tx_tau_sigma_low_s_0deg_image)
    quick_one_layer_tx_tau_sigma_low_s_0deg_axis.set_title(quick_one_layer_tx_tau_sigma_low_s_0deg_panel_title)
    quick_one_layer_tx_tau_sigma_low_s_0deg_axis.axis('off')
quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.suptitle(quick_one_layer_tx_tau_sigma_low_s_0deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.tight_layout()
quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_png = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_triptych.png"
quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_pdf = quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{quick_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_triptych.pdf"
quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_png, dpi=220)
quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.savefig(quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_pdf)

display(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig)
display(quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig)
display(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig)
display(quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig)
plt.close(quick_one_layer_tx_tau_sigma_low_s_0deg_linear_fig)
plt.close(quick_one_layer_tx_tau_sigma_low_s_0deg_log_fig)
plt.close(quick_one_layer_tx_tau_sigma_low_s_0deg_corr_fig)
plt.close(quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig)

quick_one_layer_tx_tau_sigma_low_s_0deg_plot_paths = {
    'linear_png': quick_one_layer_tx_tau_sigma_low_s_0deg_linear_png.resolve().as_posix(),
    'linear_pdf': quick_one_layer_tx_tau_sigma_low_s_0deg_linear_pdf.resolve().as_posix(),
    'log_png': quick_one_layer_tx_tau_sigma_low_s_0deg_log_png.resolve().as_posix(),
    'log_pdf': quick_one_layer_tx_tau_sigma_low_s_0deg_log_pdf.resolve().as_posix(),
    'corr_png': quick_one_layer_tx_tau_sigma_low_s_0deg_corr_png.resolve().as_posix(),
    'corr_pdf': quick_one_layer_tx_tau_sigma_low_s_0deg_corr_pdf.resolve().as_posix(),
    'triptych_png': quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': quick_one_layer_tx_tau_sigma_low_s_0deg_triptych_pdf.resolve().as_posix(),
}
quick_one_layer_tx_tau_sigma_low_s_0deg_plot_paths

### Quick Advanced Demo

In [ ]:
quick_advanced_tx_tau1_tau2_0deg_s_525um_spec = {
  "slug": "advanced_tx_tau1_tau2_0deg_s_525um",
  "title": "Two-Drude transmission: $\\tau_1$ vs $\\tau_2$ (0 deg, s-pol, $d_{\\mathrm{sub}}=525\\,\\mu$m, $d_{\\mathrm{epi}}=10\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 0.0,
  "polarization": "s",
  "map_kind": "tau",
  "substrate_thickness_um": 525.0,
  "epi_thickness_um": 10.0,
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_fixed": 0.1,
  "x_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "y_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ]
}
quick_advanced_tx_tau1_tau2_0deg_s_525um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'quick_advanced_tx_tau1_tau2_0deg_s_525um'
quick_advanced_tx_tau1_tau2_0deg_s_525um_output_root.mkdir(parents=True, exist_ok=True)
quick_advanced_tx_tau1_tau2_0deg_s_525um_result = run_lecture_map_from_spec(quick_advanced_tx_tau1_tau2_0deg_s_525um_spec, output_root=quick_advanced_tx_tau1_tau2_0deg_s_525um_output_root, profile='smoke')
quick_advanced_tx_tau1_tau2_0deg_s_525um_result['study_dir']

The next cell contains the full plotting code for this section. Change it here without affecting the other study sections.

In [ ]:
quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir = Path(quick_advanced_tx_tau1_tau2_0deg_s_525um_result['study_dir'])
quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_title = quick_advanced_tx_tau1_tau2_0deg_s_525um_result['title']
quick_advanced_tx_tau1_tau2_0deg_s_525um_x_label = "$\\tau_1$ (ps)"
quick_advanced_tx_tau1_tau2_0deg_s_525um_y_label = "$\\tau_2$ (ps)"

with (quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as quick_advanced_tx_tau1_tau2_0deg_s_525um_summary_handle:
    quick_advanced_tx_tau1_tau2_0deg_s_525um_summary_meta = json.load(quick_advanced_tx_tau1_tau2_0deg_s_525um_summary_handle)
with (quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_handle:
    quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_meta = json.load(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_handle)

quick_advanced_tx_tau1_tau2_0deg_s_525um_rows = []
with (quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as quick_advanced_tx_tau1_tau2_0deg_s_525um_summary_csv:
    quick_advanced_tx_tau1_tau2_0deg_s_525um_reader = csv.DictReader(quick_advanced_tx_tau1_tau2_0deg_s_525um_summary_csv)
    for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_reader:
        quick_advanced_tx_tau1_tau2_0deg_s_525um_parsed = {}
        for quick_advanced_tx_tau1_tau2_0deg_s_525um_key, quick_advanced_tx_tau1_tau2_0deg_s_525um_value in quick_advanced_tx_tau1_tau2_0deg_s_525um_row.items():
            if quick_advanced_tx_tau1_tau2_0deg_s_525um_value is None:
                quick_advanced_tx_tau1_tau2_0deg_s_525um_parsed[quick_advanced_tx_tau1_tau2_0deg_s_525um_key] = quick_advanced_tx_tau1_tau2_0deg_s_525um_value
                continue
            quick_advanced_tx_tau1_tau2_0deg_s_525um_text = str(quick_advanced_tx_tau1_tau2_0deg_s_525um_value).strip()
            if quick_advanced_tx_tau1_tau2_0deg_s_525um_text == '':
                quick_advanced_tx_tau1_tau2_0deg_s_525um_parsed[quick_advanced_tx_tau1_tau2_0deg_s_525um_key] = quick_advanced_tx_tau1_tau2_0deg_s_525um_text
                continue
            try:
                quick_advanced_tx_tau1_tau2_0deg_s_525um_parsed[quick_advanced_tx_tau1_tau2_0deg_s_525um_key] = float(quick_advanced_tx_tau1_tau2_0deg_s_525um_text)
            except ValueError:
                quick_advanced_tx_tau1_tau2_0deg_s_525um_parsed[quick_advanced_tx_tau1_tau2_0deg_s_525um_key] = quick_advanced_tx_tau1_tau2_0deg_s_525um_text
        quick_advanced_tx_tau1_tau2_0deg_s_525um_rows.append(quick_advanced_tx_tau1_tau2_0deg_s_525um_parsed)

quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key, quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key = list(quick_advanced_tx_tau1_tau2_0deg_s_525um_summary_meta['recovery_keys'])
quick_advanced_tx_tau1_tau2_0deg_s_525um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key}_true_minus_fit': f'signed_err_{quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key}',
    f'{quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key}_true_minus_fit': f'signed_err_{quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key}',
    f'abs_{quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key}_error': f'abs_err_{quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key}',
    f'abs_{quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key}_error': f'abs_err_{quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key}',
}
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key = quick_advanced_tx_tau1_tau2_0deg_s_525um_metric_options['data_fit']
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key = quick_advanced_tx_tau1_tau2_0deg_s_525um_metric_options['data_fit']
quick_advanced_tx_tau1_tau2_0deg_s_525um_x_values = sorted({float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row[quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key]) for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_rows})
quick_advanced_tx_tau1_tau2_0deg_s_525um_y_values = sorted({float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row[quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key]) for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_rows})

print('Available map metrics:')
for quick_advanced_tx_tau1_tau2_0deg_s_525um_option_name, quick_advanced_tx_tau1_tau2_0deg_s_525um_option_key in quick_advanced_tx_tau1_tau2_0deg_s_525um_metric_options.items():
    print(f"  {quick_advanced_tx_tau1_tau2_0deg_s_525um_option_name}: {metric_label(quick_advanced_tx_tau1_tau2_0deg_s_525um_option_key)}")
print(f"Current linear map: {metric_label(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key)}")
print(f"Current log map: {metric_label(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key)}")

def quick_advanced_tx_tau1_tau2_0deg_s_525um_aggregate_grid(value_key):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_grid = np.full((len(quick_advanced_tx_tau1_tau2_0deg_s_525um_y_values), len(quick_advanced_tx_tau1_tau2_0deg_s_525um_x_values)), np.nan, dtype=np.float64)
    for quick_advanced_tx_tau1_tau2_0deg_s_525um_iy, quick_advanced_tx_tau1_tau2_0deg_s_525um_y_value in enumerate(quick_advanced_tx_tau1_tau2_0deg_s_525um_y_values):
        for quick_advanced_tx_tau1_tau2_0deg_s_525um_ix, quick_advanced_tx_tau1_tau2_0deg_s_525um_x_value in enumerate(quick_advanced_tx_tau1_tau2_0deg_s_525um_x_values):
            quick_advanced_tx_tau1_tau2_0deg_s_525um_cell = [
                float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row[value_key])
                for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_rows
                if math.isclose(float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row[quick_advanced_tx_tau1_tau2_0deg_s_525um_x_key]), quick_advanced_tx_tau1_tau2_0deg_s_525um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row[quick_advanced_tx_tau1_tau2_0deg_s_525um_y_key]), quick_advanced_tx_tau1_tau2_0deg_s_525um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row[value_key]))
            ]
            if quick_advanced_tx_tau1_tau2_0deg_s_525um_cell:
                quick_advanced_tx_tau1_tau2_0deg_s_525um_grid[quick_advanced_tx_tau1_tau2_0deg_s_525um_iy, quick_advanced_tx_tau1_tau2_0deg_s_525um_ix] = float(np.mean(quick_advanced_tx_tau1_tau2_0deg_s_525um_cell))
    return quick_advanced_tx_tau1_tau2_0deg_s_525um_grid

def quick_advanced_tx_tau1_tau2_0deg_s_525um_positive_grid_for_log(grid):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_grid = np.asarray(grid, dtype=np.float64).copy()
    quick_advanced_tx_tau1_tau2_0deg_s_525um_finite = quick_advanced_tx_tau1_tau2_0deg_s_525um_grid[np.isfinite(quick_advanced_tx_tau1_tau2_0deg_s_525um_grid)]
    quick_advanced_tx_tau1_tau2_0deg_s_525um_positive = quick_advanced_tx_tau1_tau2_0deg_s_525um_finite[quick_advanced_tx_tau1_tau2_0deg_s_525um_finite > 0.0]
    if quick_advanced_tx_tau1_tau2_0deg_s_525um_positive.size == 0:
        quick_advanced_tx_tau1_tau2_0deg_s_525um_grid[np.isfinite(quick_advanced_tx_tau1_tau2_0deg_s_525um_grid)] = 1.0
        return quick_advanced_tx_tau1_tau2_0deg_s_525um_grid
    quick_advanced_tx_tau1_tau2_0deg_s_525um_floor = max(float(np.min(quick_advanced_tx_tau1_tau2_0deg_s_525um_positive)) * 0.5, 1e-18)
    quick_advanced_tx_tau1_tau2_0deg_s_525um_grid[np.isfinite(quick_advanced_tx_tau1_tau2_0deg_s_525um_grid) & (quick_advanced_tx_tau1_tau2_0deg_s_525um_grid <= 0.0)] = quick_advanced_tx_tau1_tau2_0deg_s_525um_floor
    return quick_advanced_tx_tau1_tau2_0deg_s_525um_grid

quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid = quick_advanced_tx_tau1_tau2_0deg_s_525um_aggregate_grid(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key)
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig, quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite = quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid[np.isfinite(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid)]
if str(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key).startswith('signed_err_'):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_limit = max(float(np.max(np.abs(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite))), 1e-18)
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin = -quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_limit
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax = quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_limit
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_levels = np.linspace(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin, quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax, 256)
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_cmap = 'plasma'
else:
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin = float(np.min(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite))
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax = float(np.max(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite)) + 1e-12
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_levels = np.linspace(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin, quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax, 256)
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_cmap = 'plasma'
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_contour = quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.contourf(
    np.asarray(quick_advanced_tx_tau1_tau2_0deg_s_525um_x_values, dtype=np.float64),
    np.asarray(quick_advanced_tx_tau1_tau2_0deg_s_525um_y_values, dtype=np.float64),
    quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid,
    levels=quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_levels,
    cmap=quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_cmap,
    extend='both',
)
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.set_title(quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_title + ' [linear]')
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.set_xlabel(quick_advanced_tx_tau1_tau2_0deg_s_525um_x_label)
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.set_ylabel(quick_advanced_tx_tau1_tau2_0deg_s_525um_y_label)
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_cbar = quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.colorbar(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_contour, ax=quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax)
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_cbar.set_label(metric_label(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key))
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.tight_layout()
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_png = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_linear.png"
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_pdf = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_linear.pdf"
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_png, dpi=220)
quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_pdf)

if str(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key).startswith('signed_err_'):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key = 'abs_err_' + str(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key)[len('signed_err_'):]
else:
    quick_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key = quick_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_grid = quick_advanced_tx_tau1_tau2_0deg_s_525um_positive_grid_for_log(quick_advanced_tx_tau1_tau2_0deg_s_525um_aggregate_grid(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key))
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig, quick_advanced_tx_tau1_tau2_0deg_s_525um_log_ax = plt.subplots(figsize=(5.8, 4.6))
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_finite = quick_advanced_tx_tau1_tau2_0deg_s_525um_log_grid[np.isfinite(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_grid)]
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_positive = quick_advanced_tx_tau1_tau2_0deg_s_525um_log_finite[quick_advanced_tx_tau1_tau2_0deg_s_525um_log_finite > 0.0]
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin = float(np.min(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_positive))
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax = float(np.max(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_positive))
if math.isclose(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin, quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax = quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin * 1.01
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_levels = np.geomspace(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin, quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax, 256)
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_contour = quick_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.contourf(
    np.asarray(quick_advanced_tx_tau1_tau2_0deg_s_525um_x_values, dtype=np.float64),
    np.asarray(quick_advanced_tx_tau1_tau2_0deg_s_525um_y_values, dtype=np.float64),
    quick_advanced_tx_tau1_tau2_0deg_s_525um_log_grid,
    levels=quick_advanced_tx_tau1_tau2_0deg_s_525um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin, vmax=quick_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax),
    extend='both',
)
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.set_title(quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_title + ' [log]')
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.set_xlabel(quick_advanced_tx_tau1_tau2_0deg_s_525um_x_label)
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.set_ylabel(quick_advanced_tx_tau1_tau2_0deg_s_525um_y_label)
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_cbar = quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.colorbar(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_contour, ax=quick_advanced_tx_tau1_tau2_0deg_s_525um_log_ax)
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_cbar.set_label(metric_label(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key))
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.tight_layout()
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_png = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_log.png"
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_pdf = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_log.pdf"
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_png, dpi=220)
quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_pdf)

quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows = quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_meta['rows']
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels = sorted({str(quick_advanced_tx_tau1_tau2_0deg_s_525um_row['param_i']) for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows} | {str(quick_advanced_tx_tau1_tau2_0deg_s_525um_row['param_j']) for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows})
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_index = {quick_advanced_tx_tau1_tau2_0deg_s_525um_label: quick_advanced_tx_tau1_tau2_0deg_s_525um_idx for quick_advanced_tx_tau1_tau2_0deg_s_525um_idx, quick_advanced_tx_tau1_tau2_0deg_s_525um_label in enumerate(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)}
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix = np.full((len(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels), len(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)), np.nan, dtype=np.float64)
for quick_advanced_tx_tau1_tau2_0deg_s_525um_row in quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows:
    quick_advanced_tx_tau1_tau2_0deg_s_525um_i = quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_index[str(quick_advanced_tx_tau1_tau2_0deg_s_525um_row['param_i'])]
    quick_advanced_tx_tau1_tau2_0deg_s_525um_j = quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_index[str(quick_advanced_tx_tau1_tau2_0deg_s_525um_row['param_j'])]
    quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix[quick_advanced_tx_tau1_tau2_0deg_s_525um_i, quick_advanced_tx_tau1_tau2_0deg_s_525um_j] = float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row['correlation'])
    quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix[quick_advanced_tx_tau1_tau2_0deg_s_525um_j, quick_advanced_tx_tau1_tau2_0deg_s_525um_i] = float(quick_advanced_tx_tau1_tau2_0deg_s_525um_row['correlation'])
for quick_advanced_tx_tau1_tau2_0deg_s_525um_diag in range(len(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix[quick_advanced_tx_tau1_tau2_0deg_s_525um_diag, quick_advanced_tx_tau1_tau2_0deg_s_525um_diag] = 1.0

quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig, quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_image = quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.imshow(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_title(quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_title + ' [correlation]')
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_xticks(range(len(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)))
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_yticks(range(len(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)))
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(quick_advanced_tx_tau1_tau2_0deg_s_525um_label, quick_advanced_tx_tau1_tau2_0deg_s_525um_label) for quick_advanced_tx_tau1_tau2_0deg_s_525um_label in quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels], rotation=45, ha='right')
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(quick_advanced_tx_tau1_tau2_0deg_s_525um_label, quick_advanced_tx_tau1_tau2_0deg_s_525um_label) for quick_advanced_tx_tau1_tau2_0deg_s_525um_label in quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels])
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.colorbar(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_image, ax=quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax, label=r'$\rho_{ij}$')
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.tight_layout()
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_png = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_corr.png"
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_pdf = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_corr.pdf"
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_png, dpi=220)
quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_pdf)

quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig, quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for quick_advanced_tx_tau1_tau2_0deg_s_525um_axis, quick_advanced_tx_tau1_tau2_0deg_s_525um_image_path, quick_advanced_tx_tau1_tau2_0deg_s_525um_panel_title in zip(
    quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_axes,
    (quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_png, quick_advanced_tx_tau1_tau2_0deg_s_525um_log_png, quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    quick_advanced_tx_tau1_tau2_0deg_s_525um_image = plt.imread(str(quick_advanced_tx_tau1_tau2_0deg_s_525um_image_path))
    quick_advanced_tx_tau1_tau2_0deg_s_525um_axis.imshow(quick_advanced_tx_tau1_tau2_0deg_s_525um_image)
    quick_advanced_tx_tau1_tau2_0deg_s_525um_axis.set_title(quick_advanced_tx_tau1_tau2_0deg_s_525um_panel_title)
    quick_advanced_tx_tau1_tau2_0deg_s_525um_axis.axis('off')
quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.suptitle(quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_title, fontsize=14, fontweight='bold', y=1.02)
quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.tight_layout()
quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_png = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_triptych.png"
quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_pdf = quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{quick_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_triptych.pdf"
quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_png, dpi=220)
quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.savefig(quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_pdf)

display(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig)
display(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig)
display(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig)
display(quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig)
plt.close(quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig)
plt.close(quick_advanced_tx_tau1_tau2_0deg_s_525um_log_fig)
plt.close(quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig)
plt.close(quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig)

quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_paths = {
    'linear_png': quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_png.resolve().as_posix(),
    'linear_pdf': quick_advanced_tx_tau1_tau2_0deg_s_525um_linear_pdf.resolve().as_posix(),
    'log_png': quick_advanced_tx_tau1_tau2_0deg_s_525um_log_png.resolve().as_posix(),
    'log_pdf': quick_advanced_tx_tau1_tau2_0deg_s_525um_log_pdf.resolve().as_posix(),
    'corr_png': quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_png.resolve().as_posix(),
    'corr_pdf': quick_advanced_tx_tau1_tau2_0deg_s_525um_corr_pdf.resolve().as_posix(),
    'triptych_png': quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_png.resolve().as_posix(),
    'triptych_pdf': quick_advanced_tx_tau1_tau2_0deg_s_525um_triptych_pdf.resolve().as_posix(),
}
quick_advanced_tx_tau1_tau2_0deg_s_525um_plot_paths

## Full-Resolution One-Layer Study Blocks

Every block below contains its own calculation cell and its own complete plotting cell.

### One-layer Drude transmission: $\tau$ vs $\sigma$ (0 deg, s-pol, $d=525\,\mu$m)

In [ ]:
full_one_layer_tx_tau_sigma_low_s_0deg_spec = {
  "slug": "one_layer_tx_tau_sigma_low_s_0deg",
  "title": "One-layer Drude transmission: $\\tau$ vs $\\sigma$ (0 deg, s-pol, $d=525\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 0.0,
  "polarization": "s",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    0.01,
    1.0
  ],
  "thickness_um": 525.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
full_one_layer_tx_tau_sigma_low_s_0deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_tx_tau_sigma_low_s_0deg'
full_one_layer_tx_tau_sigma_low_s_0deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_tx_tau_sigma_low_s_0deg_result = run_lecture_map_from_spec(full_one_layer_tx_tau_sigma_low_s_0deg_spec, output_root=full_one_layer_tx_tau_sigma_low_s_0deg_output_root, profile='full')
full_one_layer_tx_tau_sigma_low_s_0deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_tx_tau_sigma_low_s_0deg_study_dir = Path(full_one_layer_tx_tau_sigma_low_s_0deg_result['study_dir'])
full_one_layer_tx_tau_sigma_low_s_0deg_plot_title = full_one_layer_tx_tau_sigma_low_s_0deg_result['title']
full_one_layer_tx_tau_sigma_low_s_0deg_x_label = "$\\tau$ (ps)"
full_one_layer_tx_tau_sigma_low_s_0deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_low_s_0deg_summary_handle:
    full_one_layer_tx_tau_sigma_low_s_0deg_summary_meta = json.load(full_one_layer_tx_tau_sigma_low_s_0deg_summary_handle)
with (full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_low_s_0deg_corr_handle:
    full_one_layer_tx_tau_sigma_low_s_0deg_corr_meta = json.load(full_one_layer_tx_tau_sigma_low_s_0deg_corr_handle)

full_one_layer_tx_tau_sigma_low_s_0deg_rows = []
with (full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_tx_tau_sigma_low_s_0deg_summary_csv:
    full_one_layer_tx_tau_sigma_low_s_0deg_reader = csv.DictReader(full_one_layer_tx_tau_sigma_low_s_0deg_summary_csv)
    for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_reader:
        full_one_layer_tx_tau_sigma_low_s_0deg_parsed = {}
        for full_one_layer_tx_tau_sigma_low_s_0deg_key, full_one_layer_tx_tau_sigma_low_s_0deg_value in full_one_layer_tx_tau_sigma_low_s_0deg_row.items():
            if full_one_layer_tx_tau_sigma_low_s_0deg_value is None:
                full_one_layer_tx_tau_sigma_low_s_0deg_parsed[full_one_layer_tx_tau_sigma_low_s_0deg_key] = full_one_layer_tx_tau_sigma_low_s_0deg_value
                continue
            full_one_layer_tx_tau_sigma_low_s_0deg_text = str(full_one_layer_tx_tau_sigma_low_s_0deg_value).strip()
            if full_one_layer_tx_tau_sigma_low_s_0deg_text == '':
                full_one_layer_tx_tau_sigma_low_s_0deg_parsed[full_one_layer_tx_tau_sigma_low_s_0deg_key] = full_one_layer_tx_tau_sigma_low_s_0deg_text
                continue
            try:
                full_one_layer_tx_tau_sigma_low_s_0deg_parsed[full_one_layer_tx_tau_sigma_low_s_0deg_key] = float(full_one_layer_tx_tau_sigma_low_s_0deg_text)
            except ValueError:
                full_one_layer_tx_tau_sigma_low_s_0deg_parsed[full_one_layer_tx_tau_sigma_low_s_0deg_key] = full_one_layer_tx_tau_sigma_low_s_0deg_text
        full_one_layer_tx_tau_sigma_low_s_0deg_rows.append(full_one_layer_tx_tau_sigma_low_s_0deg_parsed)

full_one_layer_tx_tau_sigma_low_s_0deg_x_key, full_one_layer_tx_tau_sigma_low_s_0deg_y_key = list(full_one_layer_tx_tau_sigma_low_s_0deg_summary_meta['recovery_keys'])
full_one_layer_tx_tau_sigma_low_s_0deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_tx_tau_sigma_low_s_0deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_low_s_0deg_x_key}',
    f'{full_one_layer_tx_tau_sigma_low_s_0deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_low_s_0deg_y_key}',
    f'abs_{full_one_layer_tx_tau_sigma_low_s_0deg_x_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_low_s_0deg_x_key}',
    f'abs_{full_one_layer_tx_tau_sigma_low_s_0deg_y_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_low_s_0deg_y_key}',
}
full_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key = full_one_layer_tx_tau_sigma_low_s_0deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_low_s_0deg_log_value_key = full_one_layer_tx_tau_sigma_low_s_0deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_low_s_0deg_x_values = sorted({float(full_one_layer_tx_tau_sigma_low_s_0deg_row[full_one_layer_tx_tau_sigma_low_s_0deg_x_key]) for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_rows})
full_one_layer_tx_tau_sigma_low_s_0deg_y_values = sorted({float(full_one_layer_tx_tau_sigma_low_s_0deg_row[full_one_layer_tx_tau_sigma_low_s_0deg_y_key]) for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_rows})

print('Available map metrics:')
for full_one_layer_tx_tau_sigma_low_s_0deg_option_name, full_one_layer_tx_tau_sigma_low_s_0deg_option_key in full_one_layer_tx_tau_sigma_low_s_0deg_metric_options.items():
    print(f"  {full_one_layer_tx_tau_sigma_low_s_0deg_option_name}: {metric_label(full_one_layer_tx_tau_sigma_low_s_0deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_tx_tau_sigma_low_s_0deg_log_value_key)}")

def full_one_layer_tx_tau_sigma_low_s_0deg_aggregate_grid(value_key):
    full_one_layer_tx_tau_sigma_low_s_0deg_grid = np.full((len(full_one_layer_tx_tau_sigma_low_s_0deg_y_values), len(full_one_layer_tx_tau_sigma_low_s_0deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_tx_tau_sigma_low_s_0deg_iy, full_one_layer_tx_tau_sigma_low_s_0deg_y_value in enumerate(full_one_layer_tx_tau_sigma_low_s_0deg_y_values):
        for full_one_layer_tx_tau_sigma_low_s_0deg_ix, full_one_layer_tx_tau_sigma_low_s_0deg_x_value in enumerate(full_one_layer_tx_tau_sigma_low_s_0deg_x_values):
            full_one_layer_tx_tau_sigma_low_s_0deg_cell = [
                float(full_one_layer_tx_tau_sigma_low_s_0deg_row[value_key])
                for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_rows
                if math.isclose(float(full_one_layer_tx_tau_sigma_low_s_0deg_row[full_one_layer_tx_tau_sigma_low_s_0deg_x_key]), full_one_layer_tx_tau_sigma_low_s_0deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_tx_tau_sigma_low_s_0deg_row[full_one_layer_tx_tau_sigma_low_s_0deg_y_key]), full_one_layer_tx_tau_sigma_low_s_0deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_tx_tau_sigma_low_s_0deg_row[value_key]))
            ]
            if full_one_layer_tx_tau_sigma_low_s_0deg_cell:
                full_one_layer_tx_tau_sigma_low_s_0deg_grid[full_one_layer_tx_tau_sigma_low_s_0deg_iy, full_one_layer_tx_tau_sigma_low_s_0deg_ix] = float(np.mean(full_one_layer_tx_tau_sigma_low_s_0deg_cell))
    return full_one_layer_tx_tau_sigma_low_s_0deg_grid

def full_one_layer_tx_tau_sigma_low_s_0deg_positive_grid_for_log(grid):
    full_one_layer_tx_tau_sigma_low_s_0deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_tx_tau_sigma_low_s_0deg_finite = full_one_layer_tx_tau_sigma_low_s_0deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_0deg_grid)]
    full_one_layer_tx_tau_sigma_low_s_0deg_positive = full_one_layer_tx_tau_sigma_low_s_0deg_finite[full_one_layer_tx_tau_sigma_low_s_0deg_finite > 0.0]
    if full_one_layer_tx_tau_sigma_low_s_0deg_positive.size == 0:
        full_one_layer_tx_tau_sigma_low_s_0deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_0deg_grid)] = 1.0
        return full_one_layer_tx_tau_sigma_low_s_0deg_grid
    full_one_layer_tx_tau_sigma_low_s_0deg_floor = max(float(np.min(full_one_layer_tx_tau_sigma_low_s_0deg_positive)) * 0.5, 1e-18)
    full_one_layer_tx_tau_sigma_low_s_0deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_0deg_grid) & (full_one_layer_tx_tau_sigma_low_s_0deg_grid <= 0.0)] = full_one_layer_tx_tau_sigma_low_s_0deg_floor
    return full_one_layer_tx_tau_sigma_low_s_0deg_grid

full_one_layer_tx_tau_sigma_low_s_0deg_linear_grid = full_one_layer_tx_tau_sigma_low_s_0deg_aggregate_grid(full_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key)
full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig, full_one_layer_tx_tau_sigma_low_s_0deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_low_s_0deg_linear_finite = full_one_layer_tx_tau_sigma_low_s_0deg_linear_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_0deg_linear_grid)]
if str(full_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_limit = max(float(np.max(np.abs(full_one_layer_tx_tau_sigma_low_s_0deg_linear_finite))), 1e-18)
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin = -full_one_layer_tx_tau_sigma_low_s_0deg_linear_limit
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax = full_one_layer_tx_tau_sigma_low_s_0deg_linear_limit
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin, full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_cmap = 'plasma'
else:
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin = float(np.min(full_one_layer_tx_tau_sigma_low_s_0deg_linear_finite))
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax = float(np.max(full_one_layer_tx_tau_sigma_low_s_0deg_linear_finite)) + 1e-12
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmin, full_one_layer_tx_tau_sigma_low_s_0deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_cmap = 'plasma'
full_one_layer_tx_tau_sigma_low_s_0deg_linear_contour = full_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_low_s_0deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_low_s_0deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_low_s_0deg_linear_grid,
    levels=full_one_layer_tx_tau_sigma_low_s_0deg_linear_levels,
    cmap=full_one_layer_tx_tau_sigma_low_s_0deg_linear_cmap,
    extend='both',
)
full_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.set_title(full_one_layer_tx_tau_sigma_low_s_0deg_plot_title + ' [linear]')
full_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.set_xlabel(full_one_layer_tx_tau_sigma_low_s_0deg_x_label)
full_one_layer_tx_tau_sigma_low_s_0deg_linear_ax.set_ylabel(full_one_layer_tx_tau_sigma_low_s_0deg_y_label)
full_one_layer_tx_tau_sigma_low_s_0deg_linear_cbar = full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.colorbar(full_one_layer_tx_tau_sigma_low_s_0deg_linear_contour, ax=full_one_layer_tx_tau_sigma_low_s_0deg_linear_ax)
full_one_layer_tx_tau_sigma_low_s_0deg_linear_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_low_s_0deg_linear_value_key))
full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_0deg_linear_png = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_linear.png"
full_one_layer_tx_tau_sigma_low_s_0deg_linear_pdf = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_linear.pdf"
full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_linear_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_linear_pdf)

if str(full_one_layer_tx_tau_sigma_low_s_0deg_log_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_low_s_0deg_log_source_key = 'abs_err_' + str(full_one_layer_tx_tau_sigma_low_s_0deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_tx_tau_sigma_low_s_0deg_log_source_key = full_one_layer_tx_tau_sigma_low_s_0deg_log_value_key
full_one_layer_tx_tau_sigma_low_s_0deg_log_grid = full_one_layer_tx_tau_sigma_low_s_0deg_positive_grid_for_log(full_one_layer_tx_tau_sigma_low_s_0deg_aggregate_grid(full_one_layer_tx_tau_sigma_low_s_0deg_log_source_key))
full_one_layer_tx_tau_sigma_low_s_0deg_log_fig, full_one_layer_tx_tau_sigma_low_s_0deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_low_s_0deg_log_finite = full_one_layer_tx_tau_sigma_low_s_0deg_log_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_0deg_log_grid)]
full_one_layer_tx_tau_sigma_low_s_0deg_log_positive = full_one_layer_tx_tau_sigma_low_s_0deg_log_finite[full_one_layer_tx_tau_sigma_low_s_0deg_log_finite > 0.0]
full_one_layer_tx_tau_sigma_low_s_0deg_log_vmin = float(np.min(full_one_layer_tx_tau_sigma_low_s_0deg_log_positive))
full_one_layer_tx_tau_sigma_low_s_0deg_log_vmax = float(np.max(full_one_layer_tx_tau_sigma_low_s_0deg_log_positive))
if math.isclose(full_one_layer_tx_tau_sigma_low_s_0deg_log_vmin, full_one_layer_tx_tau_sigma_low_s_0deg_log_vmax):
    full_one_layer_tx_tau_sigma_low_s_0deg_log_vmax = full_one_layer_tx_tau_sigma_low_s_0deg_log_vmin * 1.01
full_one_layer_tx_tau_sigma_low_s_0deg_log_levels = np.geomspace(full_one_layer_tx_tau_sigma_low_s_0deg_log_vmin, full_one_layer_tx_tau_sigma_low_s_0deg_log_vmax, 256)
full_one_layer_tx_tau_sigma_low_s_0deg_log_contour = full_one_layer_tx_tau_sigma_low_s_0deg_log_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_low_s_0deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_low_s_0deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_low_s_0deg_log_grid,
    levels=full_one_layer_tx_tau_sigma_low_s_0deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_tx_tau_sigma_low_s_0deg_log_vmin, vmax=full_one_layer_tx_tau_sigma_low_s_0deg_log_vmax),
    extend='both',
)
full_one_layer_tx_tau_sigma_low_s_0deg_log_ax.set_title(full_one_layer_tx_tau_sigma_low_s_0deg_plot_title + ' [log]')
full_one_layer_tx_tau_sigma_low_s_0deg_log_ax.set_xlabel(full_one_layer_tx_tau_sigma_low_s_0deg_x_label)
full_one_layer_tx_tau_sigma_low_s_0deg_log_ax.set_ylabel(full_one_layer_tx_tau_sigma_low_s_0deg_y_label)
full_one_layer_tx_tau_sigma_low_s_0deg_log_cbar = full_one_layer_tx_tau_sigma_low_s_0deg_log_fig.colorbar(full_one_layer_tx_tau_sigma_low_s_0deg_log_contour, ax=full_one_layer_tx_tau_sigma_low_s_0deg_log_ax)
full_one_layer_tx_tau_sigma_low_s_0deg_log_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_low_s_0deg_log_source_key))
full_one_layer_tx_tau_sigma_low_s_0deg_log_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_0deg_log_png = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_log.png"
full_one_layer_tx_tau_sigma_low_s_0deg_log_pdf = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_log.pdf"
full_one_layer_tx_tau_sigma_low_s_0deg_log_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_log_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_0deg_log_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_log_pdf)

full_one_layer_tx_tau_sigma_low_s_0deg_corr_rows = full_one_layer_tx_tau_sigma_low_s_0deg_corr_meta['rows']
full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels = sorted({str(full_one_layer_tx_tau_sigma_low_s_0deg_row['param_i']) for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_corr_rows} | {str(full_one_layer_tx_tau_sigma_low_s_0deg_row['param_j']) for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_corr_rows})
full_one_layer_tx_tau_sigma_low_s_0deg_corr_index = {full_one_layer_tx_tau_sigma_low_s_0deg_label: full_one_layer_tx_tau_sigma_low_s_0deg_idx for full_one_layer_tx_tau_sigma_low_s_0deg_idx, full_one_layer_tx_tau_sigma_low_s_0deg_label in enumerate(full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)}
full_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix = np.full((len(full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels), len(full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_tx_tau_sigma_low_s_0deg_row in full_one_layer_tx_tau_sigma_low_s_0deg_corr_rows:
    full_one_layer_tx_tau_sigma_low_s_0deg_i = full_one_layer_tx_tau_sigma_low_s_0deg_corr_index[str(full_one_layer_tx_tau_sigma_low_s_0deg_row['param_i'])]
    full_one_layer_tx_tau_sigma_low_s_0deg_j = full_one_layer_tx_tau_sigma_low_s_0deg_corr_index[str(full_one_layer_tx_tau_sigma_low_s_0deg_row['param_j'])]
    full_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix[full_one_layer_tx_tau_sigma_low_s_0deg_i, full_one_layer_tx_tau_sigma_low_s_0deg_j] = float(full_one_layer_tx_tau_sigma_low_s_0deg_row['correlation'])
    full_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix[full_one_layer_tx_tau_sigma_low_s_0deg_j, full_one_layer_tx_tau_sigma_low_s_0deg_i] = float(full_one_layer_tx_tau_sigma_low_s_0deg_row['correlation'])
for full_one_layer_tx_tau_sigma_low_s_0deg_diag in range(len(full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)):
    full_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix[full_one_layer_tx_tau_sigma_low_s_0deg_diag, full_one_layer_tx_tau_sigma_low_s_0deg_diag] = 1.0

full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig, full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_tx_tau_sigma_low_s_0deg_corr_image = full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.imshow(full_one_layer_tx_tau_sigma_low_s_0deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_title(full_one_layer_tx_tau_sigma_low_s_0deg_plot_title + ' [correlation]')
full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_xticks(range(len(full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)))
full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_yticks(range(len(full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels)))
full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_low_s_0deg_label, full_one_layer_tx_tau_sigma_low_s_0deg_label) for full_one_layer_tx_tau_sigma_low_s_0deg_label in full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels], rotation=45, ha='right')
full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_low_s_0deg_label, full_one_layer_tx_tau_sigma_low_s_0deg_label) for full_one_layer_tx_tau_sigma_low_s_0deg_label in full_one_layer_tx_tau_sigma_low_s_0deg_corr_labels])
full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.colorbar(full_one_layer_tx_tau_sigma_low_s_0deg_corr_image, ax=full_one_layer_tx_tau_sigma_low_s_0deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_0deg_corr_png = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_corr.png"
full_one_layer_tx_tau_sigma_low_s_0deg_corr_pdf = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_corr.pdf"
full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_corr_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_corr_pdf)

full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig, full_one_layer_tx_tau_sigma_low_s_0deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_tx_tau_sigma_low_s_0deg_axis, full_one_layer_tx_tau_sigma_low_s_0deg_image_path, full_one_layer_tx_tau_sigma_low_s_0deg_panel_title in zip(
    full_one_layer_tx_tau_sigma_low_s_0deg_triptych_axes,
    (full_one_layer_tx_tau_sigma_low_s_0deg_linear_png, full_one_layer_tx_tau_sigma_low_s_0deg_log_png, full_one_layer_tx_tau_sigma_low_s_0deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_tx_tau_sigma_low_s_0deg_image = plt.imread(str(full_one_layer_tx_tau_sigma_low_s_0deg_image_path))
    full_one_layer_tx_tau_sigma_low_s_0deg_axis.imshow(full_one_layer_tx_tau_sigma_low_s_0deg_image)
    full_one_layer_tx_tau_sigma_low_s_0deg_axis.set_title(full_one_layer_tx_tau_sigma_low_s_0deg_panel_title)
    full_one_layer_tx_tau_sigma_low_s_0deg_axis.axis('off')
full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.suptitle(full_one_layer_tx_tau_sigma_low_s_0deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_0deg_triptych_png = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_triptych.png"
full_one_layer_tx_tau_sigma_low_s_0deg_triptych_pdf = full_one_layer_tx_tau_sigma_low_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_0deg_study_dir.name}__section_triptych.pdf"
full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_triptych_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_low_s_0deg_triptych_pdf)

display(full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig)
display(full_one_layer_tx_tau_sigma_low_s_0deg_log_fig)
display(full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig)
display(full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_0deg_linear_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_0deg_log_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_0deg_corr_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_0deg_triptych_fig)

full_one_layer_tx_tau_sigma_low_s_0deg_plot_paths = {
    'linear_png': full_one_layer_tx_tau_sigma_low_s_0deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_tx_tau_sigma_low_s_0deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_tx_tau_sigma_low_s_0deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_tx_tau_sigma_low_s_0deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_tx_tau_sigma_low_s_0deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_tx_tau_sigma_low_s_0deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_tx_tau_sigma_low_s_0deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_tx_tau_sigma_low_s_0deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_tx_tau_sigma_low_s_0deg_plot_paths

### One-layer Drude transmission: $\tau$ vs $\sigma$ (45 deg, s-pol, $d=525\,\mu$m)

In [ ]:
full_one_layer_tx_tau_sigma_low_s_45deg_spec = {
  "slug": "one_layer_tx_tau_sigma_low_s_45deg",
  "title": "One-layer Drude transmission: $\\tau$ vs $\\sigma$ (45 deg, s-pol, $d=525\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 45.0,
  "polarization": "s",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    0.01,
    1.0
  ],
  "thickness_um": 525.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
full_one_layer_tx_tau_sigma_low_s_45deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_tx_tau_sigma_low_s_45deg'
full_one_layer_tx_tau_sigma_low_s_45deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_tx_tau_sigma_low_s_45deg_result = run_lecture_map_from_spec(full_one_layer_tx_tau_sigma_low_s_45deg_spec, output_root=full_one_layer_tx_tau_sigma_low_s_45deg_output_root, profile='full')
full_one_layer_tx_tau_sigma_low_s_45deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_tx_tau_sigma_low_s_45deg_study_dir = Path(full_one_layer_tx_tau_sigma_low_s_45deg_result['study_dir'])
full_one_layer_tx_tau_sigma_low_s_45deg_plot_title = full_one_layer_tx_tau_sigma_low_s_45deg_result['title']
full_one_layer_tx_tau_sigma_low_s_45deg_x_label = "$\\tau$ (ps)"
full_one_layer_tx_tau_sigma_low_s_45deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_low_s_45deg_summary_handle:
    full_one_layer_tx_tau_sigma_low_s_45deg_summary_meta = json.load(full_one_layer_tx_tau_sigma_low_s_45deg_summary_handle)
with (full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_low_s_45deg_corr_handle:
    full_one_layer_tx_tau_sigma_low_s_45deg_corr_meta = json.load(full_one_layer_tx_tau_sigma_low_s_45deg_corr_handle)

full_one_layer_tx_tau_sigma_low_s_45deg_rows = []
with (full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_tx_tau_sigma_low_s_45deg_summary_csv:
    full_one_layer_tx_tau_sigma_low_s_45deg_reader = csv.DictReader(full_one_layer_tx_tau_sigma_low_s_45deg_summary_csv)
    for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_reader:
        full_one_layer_tx_tau_sigma_low_s_45deg_parsed = {}
        for full_one_layer_tx_tau_sigma_low_s_45deg_key, full_one_layer_tx_tau_sigma_low_s_45deg_value in full_one_layer_tx_tau_sigma_low_s_45deg_row.items():
            if full_one_layer_tx_tau_sigma_low_s_45deg_value is None:
                full_one_layer_tx_tau_sigma_low_s_45deg_parsed[full_one_layer_tx_tau_sigma_low_s_45deg_key] = full_one_layer_tx_tau_sigma_low_s_45deg_value
                continue
            full_one_layer_tx_tau_sigma_low_s_45deg_text = str(full_one_layer_tx_tau_sigma_low_s_45deg_value).strip()
            if full_one_layer_tx_tau_sigma_low_s_45deg_text == '':
                full_one_layer_tx_tau_sigma_low_s_45deg_parsed[full_one_layer_tx_tau_sigma_low_s_45deg_key] = full_one_layer_tx_tau_sigma_low_s_45deg_text
                continue
            try:
                full_one_layer_tx_tau_sigma_low_s_45deg_parsed[full_one_layer_tx_tau_sigma_low_s_45deg_key] = float(full_one_layer_tx_tau_sigma_low_s_45deg_text)
            except ValueError:
                full_one_layer_tx_tau_sigma_low_s_45deg_parsed[full_one_layer_tx_tau_sigma_low_s_45deg_key] = full_one_layer_tx_tau_sigma_low_s_45deg_text
        full_one_layer_tx_tau_sigma_low_s_45deg_rows.append(full_one_layer_tx_tau_sigma_low_s_45deg_parsed)

full_one_layer_tx_tau_sigma_low_s_45deg_x_key, full_one_layer_tx_tau_sigma_low_s_45deg_y_key = list(full_one_layer_tx_tau_sigma_low_s_45deg_summary_meta['recovery_keys'])
full_one_layer_tx_tau_sigma_low_s_45deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_tx_tau_sigma_low_s_45deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_low_s_45deg_x_key}',
    f'{full_one_layer_tx_tau_sigma_low_s_45deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_low_s_45deg_y_key}',
    f'abs_{full_one_layer_tx_tau_sigma_low_s_45deg_x_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_low_s_45deg_x_key}',
    f'abs_{full_one_layer_tx_tau_sigma_low_s_45deg_y_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_low_s_45deg_y_key}',
}
full_one_layer_tx_tau_sigma_low_s_45deg_linear_value_key = full_one_layer_tx_tau_sigma_low_s_45deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_low_s_45deg_log_value_key = full_one_layer_tx_tau_sigma_low_s_45deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_low_s_45deg_x_values = sorted({float(full_one_layer_tx_tau_sigma_low_s_45deg_row[full_one_layer_tx_tau_sigma_low_s_45deg_x_key]) for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_rows})
full_one_layer_tx_tau_sigma_low_s_45deg_y_values = sorted({float(full_one_layer_tx_tau_sigma_low_s_45deg_row[full_one_layer_tx_tau_sigma_low_s_45deg_y_key]) for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_rows})

print('Available map metrics:')
for full_one_layer_tx_tau_sigma_low_s_45deg_option_name, full_one_layer_tx_tau_sigma_low_s_45deg_option_key in full_one_layer_tx_tau_sigma_low_s_45deg_metric_options.items():
    print(f"  {full_one_layer_tx_tau_sigma_low_s_45deg_option_name}: {metric_label(full_one_layer_tx_tau_sigma_low_s_45deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_tx_tau_sigma_low_s_45deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_tx_tau_sigma_low_s_45deg_log_value_key)}")

def full_one_layer_tx_tau_sigma_low_s_45deg_aggregate_grid(value_key):
    full_one_layer_tx_tau_sigma_low_s_45deg_grid = np.full((len(full_one_layer_tx_tau_sigma_low_s_45deg_y_values), len(full_one_layer_tx_tau_sigma_low_s_45deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_tx_tau_sigma_low_s_45deg_iy, full_one_layer_tx_tau_sigma_low_s_45deg_y_value in enumerate(full_one_layer_tx_tau_sigma_low_s_45deg_y_values):
        for full_one_layer_tx_tau_sigma_low_s_45deg_ix, full_one_layer_tx_tau_sigma_low_s_45deg_x_value in enumerate(full_one_layer_tx_tau_sigma_low_s_45deg_x_values):
            full_one_layer_tx_tau_sigma_low_s_45deg_cell = [
                float(full_one_layer_tx_tau_sigma_low_s_45deg_row[value_key])
                for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_rows
                if math.isclose(float(full_one_layer_tx_tau_sigma_low_s_45deg_row[full_one_layer_tx_tau_sigma_low_s_45deg_x_key]), full_one_layer_tx_tau_sigma_low_s_45deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_tx_tau_sigma_low_s_45deg_row[full_one_layer_tx_tau_sigma_low_s_45deg_y_key]), full_one_layer_tx_tau_sigma_low_s_45deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_tx_tau_sigma_low_s_45deg_row[value_key]))
            ]
            if full_one_layer_tx_tau_sigma_low_s_45deg_cell:
                full_one_layer_tx_tau_sigma_low_s_45deg_grid[full_one_layer_tx_tau_sigma_low_s_45deg_iy, full_one_layer_tx_tau_sigma_low_s_45deg_ix] = float(np.mean(full_one_layer_tx_tau_sigma_low_s_45deg_cell))
    return full_one_layer_tx_tau_sigma_low_s_45deg_grid

def full_one_layer_tx_tau_sigma_low_s_45deg_positive_grid_for_log(grid):
    full_one_layer_tx_tau_sigma_low_s_45deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_tx_tau_sigma_low_s_45deg_finite = full_one_layer_tx_tau_sigma_low_s_45deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_45deg_grid)]
    full_one_layer_tx_tau_sigma_low_s_45deg_positive = full_one_layer_tx_tau_sigma_low_s_45deg_finite[full_one_layer_tx_tau_sigma_low_s_45deg_finite > 0.0]
    if full_one_layer_tx_tau_sigma_low_s_45deg_positive.size == 0:
        full_one_layer_tx_tau_sigma_low_s_45deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_45deg_grid)] = 1.0
        return full_one_layer_tx_tau_sigma_low_s_45deg_grid
    full_one_layer_tx_tau_sigma_low_s_45deg_floor = max(float(np.min(full_one_layer_tx_tau_sigma_low_s_45deg_positive)) * 0.5, 1e-18)
    full_one_layer_tx_tau_sigma_low_s_45deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_45deg_grid) & (full_one_layer_tx_tau_sigma_low_s_45deg_grid <= 0.0)] = full_one_layer_tx_tau_sigma_low_s_45deg_floor
    return full_one_layer_tx_tau_sigma_low_s_45deg_grid

full_one_layer_tx_tau_sigma_low_s_45deg_linear_grid = full_one_layer_tx_tau_sigma_low_s_45deg_aggregate_grid(full_one_layer_tx_tau_sigma_low_s_45deg_linear_value_key)
full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig, full_one_layer_tx_tau_sigma_low_s_45deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_low_s_45deg_linear_finite = full_one_layer_tx_tau_sigma_low_s_45deg_linear_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_45deg_linear_grid)]
if str(full_one_layer_tx_tau_sigma_low_s_45deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_limit = max(float(np.max(np.abs(full_one_layer_tx_tau_sigma_low_s_45deg_linear_finite))), 1e-18)
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmin = -full_one_layer_tx_tau_sigma_low_s_45deg_linear_limit
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmax = full_one_layer_tx_tau_sigma_low_s_45deg_linear_limit
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmin, full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_cmap = 'plasma'
else:
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmin = float(np.min(full_one_layer_tx_tau_sigma_low_s_45deg_linear_finite))
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmax = float(np.max(full_one_layer_tx_tau_sigma_low_s_45deg_linear_finite)) + 1e-12
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmin, full_one_layer_tx_tau_sigma_low_s_45deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_cmap = 'plasma'
full_one_layer_tx_tau_sigma_low_s_45deg_linear_contour = full_one_layer_tx_tau_sigma_low_s_45deg_linear_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_low_s_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_low_s_45deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_low_s_45deg_linear_grid,
    levels=full_one_layer_tx_tau_sigma_low_s_45deg_linear_levels,
    cmap=full_one_layer_tx_tau_sigma_low_s_45deg_linear_cmap,
    extend='both',
)
full_one_layer_tx_tau_sigma_low_s_45deg_linear_ax.set_title(full_one_layer_tx_tau_sigma_low_s_45deg_plot_title + ' [linear]')
full_one_layer_tx_tau_sigma_low_s_45deg_linear_ax.set_xlabel(full_one_layer_tx_tau_sigma_low_s_45deg_x_label)
full_one_layer_tx_tau_sigma_low_s_45deg_linear_ax.set_ylabel(full_one_layer_tx_tau_sigma_low_s_45deg_y_label)
full_one_layer_tx_tau_sigma_low_s_45deg_linear_cbar = full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig.colorbar(full_one_layer_tx_tau_sigma_low_s_45deg_linear_contour, ax=full_one_layer_tx_tau_sigma_low_s_45deg_linear_ax)
full_one_layer_tx_tau_sigma_low_s_45deg_linear_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_low_s_45deg_linear_value_key))
full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_45deg_linear_png = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_linear.png"
full_one_layer_tx_tau_sigma_low_s_45deg_linear_pdf = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_linear.pdf"
full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_linear_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_linear_pdf)

if str(full_one_layer_tx_tau_sigma_low_s_45deg_log_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_low_s_45deg_log_source_key = 'abs_err_' + str(full_one_layer_tx_tau_sigma_low_s_45deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_tx_tau_sigma_low_s_45deg_log_source_key = full_one_layer_tx_tau_sigma_low_s_45deg_log_value_key
full_one_layer_tx_tau_sigma_low_s_45deg_log_grid = full_one_layer_tx_tau_sigma_low_s_45deg_positive_grid_for_log(full_one_layer_tx_tau_sigma_low_s_45deg_aggregate_grid(full_one_layer_tx_tau_sigma_low_s_45deg_log_source_key))
full_one_layer_tx_tau_sigma_low_s_45deg_log_fig, full_one_layer_tx_tau_sigma_low_s_45deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_low_s_45deg_log_finite = full_one_layer_tx_tau_sigma_low_s_45deg_log_grid[np.isfinite(full_one_layer_tx_tau_sigma_low_s_45deg_log_grid)]
full_one_layer_tx_tau_sigma_low_s_45deg_log_positive = full_one_layer_tx_tau_sigma_low_s_45deg_log_finite[full_one_layer_tx_tau_sigma_low_s_45deg_log_finite > 0.0]
full_one_layer_tx_tau_sigma_low_s_45deg_log_vmin = float(np.min(full_one_layer_tx_tau_sigma_low_s_45deg_log_positive))
full_one_layer_tx_tau_sigma_low_s_45deg_log_vmax = float(np.max(full_one_layer_tx_tau_sigma_low_s_45deg_log_positive))
if math.isclose(full_one_layer_tx_tau_sigma_low_s_45deg_log_vmin, full_one_layer_tx_tau_sigma_low_s_45deg_log_vmax):
    full_one_layer_tx_tau_sigma_low_s_45deg_log_vmax = full_one_layer_tx_tau_sigma_low_s_45deg_log_vmin * 1.01
full_one_layer_tx_tau_sigma_low_s_45deg_log_levels = np.geomspace(full_one_layer_tx_tau_sigma_low_s_45deg_log_vmin, full_one_layer_tx_tau_sigma_low_s_45deg_log_vmax, 256)
full_one_layer_tx_tau_sigma_low_s_45deg_log_contour = full_one_layer_tx_tau_sigma_low_s_45deg_log_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_low_s_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_low_s_45deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_low_s_45deg_log_grid,
    levels=full_one_layer_tx_tau_sigma_low_s_45deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_tx_tau_sigma_low_s_45deg_log_vmin, vmax=full_one_layer_tx_tau_sigma_low_s_45deg_log_vmax),
    extend='both',
)
full_one_layer_tx_tau_sigma_low_s_45deg_log_ax.set_title(full_one_layer_tx_tau_sigma_low_s_45deg_plot_title + ' [log]')
full_one_layer_tx_tau_sigma_low_s_45deg_log_ax.set_xlabel(full_one_layer_tx_tau_sigma_low_s_45deg_x_label)
full_one_layer_tx_tau_sigma_low_s_45deg_log_ax.set_ylabel(full_one_layer_tx_tau_sigma_low_s_45deg_y_label)
full_one_layer_tx_tau_sigma_low_s_45deg_log_cbar = full_one_layer_tx_tau_sigma_low_s_45deg_log_fig.colorbar(full_one_layer_tx_tau_sigma_low_s_45deg_log_contour, ax=full_one_layer_tx_tau_sigma_low_s_45deg_log_ax)
full_one_layer_tx_tau_sigma_low_s_45deg_log_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_low_s_45deg_log_source_key))
full_one_layer_tx_tau_sigma_low_s_45deg_log_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_45deg_log_png = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_log.png"
full_one_layer_tx_tau_sigma_low_s_45deg_log_pdf = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_log.pdf"
full_one_layer_tx_tau_sigma_low_s_45deg_log_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_log_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_45deg_log_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_log_pdf)

full_one_layer_tx_tau_sigma_low_s_45deg_corr_rows = full_one_layer_tx_tau_sigma_low_s_45deg_corr_meta['rows']
full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels = sorted({str(full_one_layer_tx_tau_sigma_low_s_45deg_row['param_i']) for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_corr_rows} | {str(full_one_layer_tx_tau_sigma_low_s_45deg_row['param_j']) for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_corr_rows})
full_one_layer_tx_tau_sigma_low_s_45deg_corr_index = {full_one_layer_tx_tau_sigma_low_s_45deg_label: full_one_layer_tx_tau_sigma_low_s_45deg_idx for full_one_layer_tx_tau_sigma_low_s_45deg_idx, full_one_layer_tx_tau_sigma_low_s_45deg_label in enumerate(full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels)}
full_one_layer_tx_tau_sigma_low_s_45deg_corr_matrix = np.full((len(full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels), len(full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_tx_tau_sigma_low_s_45deg_row in full_one_layer_tx_tau_sigma_low_s_45deg_corr_rows:
    full_one_layer_tx_tau_sigma_low_s_45deg_i = full_one_layer_tx_tau_sigma_low_s_45deg_corr_index[str(full_one_layer_tx_tau_sigma_low_s_45deg_row['param_i'])]
    full_one_layer_tx_tau_sigma_low_s_45deg_j = full_one_layer_tx_tau_sigma_low_s_45deg_corr_index[str(full_one_layer_tx_tau_sigma_low_s_45deg_row['param_j'])]
    full_one_layer_tx_tau_sigma_low_s_45deg_corr_matrix[full_one_layer_tx_tau_sigma_low_s_45deg_i, full_one_layer_tx_tau_sigma_low_s_45deg_j] = float(full_one_layer_tx_tau_sigma_low_s_45deg_row['correlation'])
    full_one_layer_tx_tau_sigma_low_s_45deg_corr_matrix[full_one_layer_tx_tau_sigma_low_s_45deg_j, full_one_layer_tx_tau_sigma_low_s_45deg_i] = float(full_one_layer_tx_tau_sigma_low_s_45deg_row['correlation'])
for full_one_layer_tx_tau_sigma_low_s_45deg_diag in range(len(full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels)):
    full_one_layer_tx_tau_sigma_low_s_45deg_corr_matrix[full_one_layer_tx_tau_sigma_low_s_45deg_diag, full_one_layer_tx_tau_sigma_low_s_45deg_diag] = 1.0

full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig, full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_tx_tau_sigma_low_s_45deg_corr_image = full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax.imshow(full_one_layer_tx_tau_sigma_low_s_45deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax.set_title(full_one_layer_tx_tau_sigma_low_s_45deg_plot_title + ' [correlation]')
full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax.set_xticks(range(len(full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels)))
full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax.set_yticks(range(len(full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels)))
full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_low_s_45deg_label, full_one_layer_tx_tau_sigma_low_s_45deg_label) for full_one_layer_tx_tau_sigma_low_s_45deg_label in full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels], rotation=45, ha='right')
full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_low_s_45deg_label, full_one_layer_tx_tau_sigma_low_s_45deg_label) for full_one_layer_tx_tau_sigma_low_s_45deg_label in full_one_layer_tx_tau_sigma_low_s_45deg_corr_labels])
full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig.colorbar(full_one_layer_tx_tau_sigma_low_s_45deg_corr_image, ax=full_one_layer_tx_tau_sigma_low_s_45deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_45deg_corr_png = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_corr.png"
full_one_layer_tx_tau_sigma_low_s_45deg_corr_pdf = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_corr.pdf"
full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_corr_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_corr_pdf)

full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig, full_one_layer_tx_tau_sigma_low_s_45deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_tx_tau_sigma_low_s_45deg_axis, full_one_layer_tx_tau_sigma_low_s_45deg_image_path, full_one_layer_tx_tau_sigma_low_s_45deg_panel_title in zip(
    full_one_layer_tx_tau_sigma_low_s_45deg_triptych_axes,
    (full_one_layer_tx_tau_sigma_low_s_45deg_linear_png, full_one_layer_tx_tau_sigma_low_s_45deg_log_png, full_one_layer_tx_tau_sigma_low_s_45deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_tx_tau_sigma_low_s_45deg_image = plt.imread(str(full_one_layer_tx_tau_sigma_low_s_45deg_image_path))
    full_one_layer_tx_tau_sigma_low_s_45deg_axis.imshow(full_one_layer_tx_tau_sigma_low_s_45deg_image)
    full_one_layer_tx_tau_sigma_low_s_45deg_axis.set_title(full_one_layer_tx_tau_sigma_low_s_45deg_panel_title)
    full_one_layer_tx_tau_sigma_low_s_45deg_axis.axis('off')
full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig.suptitle(full_one_layer_tx_tau_sigma_low_s_45deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig.tight_layout()
full_one_layer_tx_tau_sigma_low_s_45deg_triptych_png = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_triptych.png"
full_one_layer_tx_tau_sigma_low_s_45deg_triptych_pdf = full_one_layer_tx_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_low_s_45deg_study_dir.name}__section_triptych.pdf"
full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_triptych_png, dpi=220)
full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_low_s_45deg_triptych_pdf)

display(full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig)
display(full_one_layer_tx_tau_sigma_low_s_45deg_log_fig)
display(full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig)
display(full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_45deg_linear_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_45deg_log_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_45deg_corr_fig)
plt.close(full_one_layer_tx_tau_sigma_low_s_45deg_triptych_fig)

full_one_layer_tx_tau_sigma_low_s_45deg_plot_paths = {
    'linear_png': full_one_layer_tx_tau_sigma_low_s_45deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_tx_tau_sigma_low_s_45deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_tx_tau_sigma_low_s_45deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_tx_tau_sigma_low_s_45deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_tx_tau_sigma_low_s_45deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_tx_tau_sigma_low_s_45deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_tx_tau_sigma_low_s_45deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_tx_tau_sigma_low_s_45deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_tx_tau_sigma_low_s_45deg_plot_paths

### One-layer Drude transmission: $\tau$ vs $\sigma$ (0 deg, s-pol, $d=725\,\mu$m)

In [ ]:
full_one_layer_tx_tau_sigma_high_s_0deg_spec = {
  "slug": "one_layer_tx_tau_sigma_high_s_0deg",
  "title": "One-layer Drude transmission: $\\tau$ vs $\\sigma$ (0 deg, s-pol, $d=725\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 0.0,
  "polarization": "s",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    100.0,
    1000.0
  ],
  "thickness_um": 725.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ]
}
full_one_layer_tx_tau_sigma_high_s_0deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_tx_tau_sigma_high_s_0deg'
full_one_layer_tx_tau_sigma_high_s_0deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_tx_tau_sigma_high_s_0deg_result = run_lecture_map_from_spec(full_one_layer_tx_tau_sigma_high_s_0deg_spec, output_root=full_one_layer_tx_tau_sigma_high_s_0deg_output_root, profile='full')
full_one_layer_tx_tau_sigma_high_s_0deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_tx_tau_sigma_high_s_0deg_study_dir = Path(full_one_layer_tx_tau_sigma_high_s_0deg_result['study_dir'])
full_one_layer_tx_tau_sigma_high_s_0deg_plot_title = full_one_layer_tx_tau_sigma_high_s_0deg_result['title']
full_one_layer_tx_tau_sigma_high_s_0deg_x_label = "$\\tau$ (ps)"
full_one_layer_tx_tau_sigma_high_s_0deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_high_s_0deg_summary_handle:
    full_one_layer_tx_tau_sigma_high_s_0deg_summary_meta = json.load(full_one_layer_tx_tau_sigma_high_s_0deg_summary_handle)
with (full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_high_s_0deg_corr_handle:
    full_one_layer_tx_tau_sigma_high_s_0deg_corr_meta = json.load(full_one_layer_tx_tau_sigma_high_s_0deg_corr_handle)

full_one_layer_tx_tau_sigma_high_s_0deg_rows = []
with (full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_tx_tau_sigma_high_s_0deg_summary_csv:
    full_one_layer_tx_tau_sigma_high_s_0deg_reader = csv.DictReader(full_one_layer_tx_tau_sigma_high_s_0deg_summary_csv)
    for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_reader:
        full_one_layer_tx_tau_sigma_high_s_0deg_parsed = {}
        for full_one_layer_tx_tau_sigma_high_s_0deg_key, full_one_layer_tx_tau_sigma_high_s_0deg_value in full_one_layer_tx_tau_sigma_high_s_0deg_row.items():
            if full_one_layer_tx_tau_sigma_high_s_0deg_value is None:
                full_one_layer_tx_tau_sigma_high_s_0deg_parsed[full_one_layer_tx_tau_sigma_high_s_0deg_key] = full_one_layer_tx_tau_sigma_high_s_0deg_value
                continue
            full_one_layer_tx_tau_sigma_high_s_0deg_text = str(full_one_layer_tx_tau_sigma_high_s_0deg_value).strip()
            if full_one_layer_tx_tau_sigma_high_s_0deg_text == '':
                full_one_layer_tx_tau_sigma_high_s_0deg_parsed[full_one_layer_tx_tau_sigma_high_s_0deg_key] = full_one_layer_tx_tau_sigma_high_s_0deg_text
                continue
            try:
                full_one_layer_tx_tau_sigma_high_s_0deg_parsed[full_one_layer_tx_tau_sigma_high_s_0deg_key] = float(full_one_layer_tx_tau_sigma_high_s_0deg_text)
            except ValueError:
                full_one_layer_tx_tau_sigma_high_s_0deg_parsed[full_one_layer_tx_tau_sigma_high_s_0deg_key] = full_one_layer_tx_tau_sigma_high_s_0deg_text
        full_one_layer_tx_tau_sigma_high_s_0deg_rows.append(full_one_layer_tx_tau_sigma_high_s_0deg_parsed)

full_one_layer_tx_tau_sigma_high_s_0deg_x_key, full_one_layer_tx_tau_sigma_high_s_0deg_y_key = list(full_one_layer_tx_tau_sigma_high_s_0deg_summary_meta['recovery_keys'])
full_one_layer_tx_tau_sigma_high_s_0deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_tx_tau_sigma_high_s_0deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_high_s_0deg_x_key}',
    f'{full_one_layer_tx_tau_sigma_high_s_0deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_high_s_0deg_y_key}',
    f'abs_{full_one_layer_tx_tau_sigma_high_s_0deg_x_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_high_s_0deg_x_key}',
    f'abs_{full_one_layer_tx_tau_sigma_high_s_0deg_y_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_high_s_0deg_y_key}',
}
full_one_layer_tx_tau_sigma_high_s_0deg_linear_value_key = full_one_layer_tx_tau_sigma_high_s_0deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_high_s_0deg_log_value_key = full_one_layer_tx_tau_sigma_high_s_0deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_high_s_0deg_x_values = sorted({float(full_one_layer_tx_tau_sigma_high_s_0deg_row[full_one_layer_tx_tau_sigma_high_s_0deg_x_key]) for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_rows})
full_one_layer_tx_tau_sigma_high_s_0deg_y_values = sorted({float(full_one_layer_tx_tau_sigma_high_s_0deg_row[full_one_layer_tx_tau_sigma_high_s_0deg_y_key]) for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_rows})

print('Available map metrics:')
for full_one_layer_tx_tau_sigma_high_s_0deg_option_name, full_one_layer_tx_tau_sigma_high_s_0deg_option_key in full_one_layer_tx_tau_sigma_high_s_0deg_metric_options.items():
    print(f"  {full_one_layer_tx_tau_sigma_high_s_0deg_option_name}: {metric_label(full_one_layer_tx_tau_sigma_high_s_0deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_tx_tau_sigma_high_s_0deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_tx_tau_sigma_high_s_0deg_log_value_key)}")

def full_one_layer_tx_tau_sigma_high_s_0deg_aggregate_grid(value_key):
    full_one_layer_tx_tau_sigma_high_s_0deg_grid = np.full((len(full_one_layer_tx_tau_sigma_high_s_0deg_y_values), len(full_one_layer_tx_tau_sigma_high_s_0deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_tx_tau_sigma_high_s_0deg_iy, full_one_layer_tx_tau_sigma_high_s_0deg_y_value in enumerate(full_one_layer_tx_tau_sigma_high_s_0deg_y_values):
        for full_one_layer_tx_tau_sigma_high_s_0deg_ix, full_one_layer_tx_tau_sigma_high_s_0deg_x_value in enumerate(full_one_layer_tx_tau_sigma_high_s_0deg_x_values):
            full_one_layer_tx_tau_sigma_high_s_0deg_cell = [
                float(full_one_layer_tx_tau_sigma_high_s_0deg_row[value_key])
                for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_rows
                if math.isclose(float(full_one_layer_tx_tau_sigma_high_s_0deg_row[full_one_layer_tx_tau_sigma_high_s_0deg_x_key]), full_one_layer_tx_tau_sigma_high_s_0deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_tx_tau_sigma_high_s_0deg_row[full_one_layer_tx_tau_sigma_high_s_0deg_y_key]), full_one_layer_tx_tau_sigma_high_s_0deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_tx_tau_sigma_high_s_0deg_row[value_key]))
            ]
            if full_one_layer_tx_tau_sigma_high_s_0deg_cell:
                full_one_layer_tx_tau_sigma_high_s_0deg_grid[full_one_layer_tx_tau_sigma_high_s_0deg_iy, full_one_layer_tx_tau_sigma_high_s_0deg_ix] = float(np.mean(full_one_layer_tx_tau_sigma_high_s_0deg_cell))
    return full_one_layer_tx_tau_sigma_high_s_0deg_grid

def full_one_layer_tx_tau_sigma_high_s_0deg_positive_grid_for_log(grid):
    full_one_layer_tx_tau_sigma_high_s_0deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_tx_tau_sigma_high_s_0deg_finite = full_one_layer_tx_tau_sigma_high_s_0deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_s_0deg_grid)]
    full_one_layer_tx_tau_sigma_high_s_0deg_positive = full_one_layer_tx_tau_sigma_high_s_0deg_finite[full_one_layer_tx_tau_sigma_high_s_0deg_finite > 0.0]
    if full_one_layer_tx_tau_sigma_high_s_0deg_positive.size == 0:
        full_one_layer_tx_tau_sigma_high_s_0deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_s_0deg_grid)] = 1.0
        return full_one_layer_tx_tau_sigma_high_s_0deg_grid
    full_one_layer_tx_tau_sigma_high_s_0deg_floor = max(float(np.min(full_one_layer_tx_tau_sigma_high_s_0deg_positive)) * 0.5, 1e-18)
    full_one_layer_tx_tau_sigma_high_s_0deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_s_0deg_grid) & (full_one_layer_tx_tau_sigma_high_s_0deg_grid <= 0.0)] = full_one_layer_tx_tau_sigma_high_s_0deg_floor
    return full_one_layer_tx_tau_sigma_high_s_0deg_grid

full_one_layer_tx_tau_sigma_high_s_0deg_linear_grid = full_one_layer_tx_tau_sigma_high_s_0deg_aggregate_grid(full_one_layer_tx_tau_sigma_high_s_0deg_linear_value_key)
full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig, full_one_layer_tx_tau_sigma_high_s_0deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_high_s_0deg_linear_finite = full_one_layer_tx_tau_sigma_high_s_0deg_linear_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_s_0deg_linear_grid)]
if str(full_one_layer_tx_tau_sigma_high_s_0deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_limit = max(float(np.max(np.abs(full_one_layer_tx_tau_sigma_high_s_0deg_linear_finite))), 1e-18)
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmin = -full_one_layer_tx_tau_sigma_high_s_0deg_linear_limit
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmax = full_one_layer_tx_tau_sigma_high_s_0deg_linear_limit
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmin, full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_cmap = 'plasma'
else:
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmin = float(np.min(full_one_layer_tx_tau_sigma_high_s_0deg_linear_finite))
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmax = float(np.max(full_one_layer_tx_tau_sigma_high_s_0deg_linear_finite)) + 1e-12
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmin, full_one_layer_tx_tau_sigma_high_s_0deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_cmap = 'plasma'
full_one_layer_tx_tau_sigma_high_s_0deg_linear_contour = full_one_layer_tx_tau_sigma_high_s_0deg_linear_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_high_s_0deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_high_s_0deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_high_s_0deg_linear_grid,
    levels=full_one_layer_tx_tau_sigma_high_s_0deg_linear_levels,
    cmap=full_one_layer_tx_tau_sigma_high_s_0deg_linear_cmap,
    extend='both',
)
full_one_layer_tx_tau_sigma_high_s_0deg_linear_ax.set_title(full_one_layer_tx_tau_sigma_high_s_0deg_plot_title + ' [linear]')
full_one_layer_tx_tau_sigma_high_s_0deg_linear_ax.set_xlabel(full_one_layer_tx_tau_sigma_high_s_0deg_x_label)
full_one_layer_tx_tau_sigma_high_s_0deg_linear_ax.set_ylabel(full_one_layer_tx_tau_sigma_high_s_0deg_y_label)
full_one_layer_tx_tau_sigma_high_s_0deg_linear_cbar = full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig.colorbar(full_one_layer_tx_tau_sigma_high_s_0deg_linear_contour, ax=full_one_layer_tx_tau_sigma_high_s_0deg_linear_ax)
full_one_layer_tx_tau_sigma_high_s_0deg_linear_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_high_s_0deg_linear_value_key))
full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_s_0deg_linear_png = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_linear.png"
full_one_layer_tx_tau_sigma_high_s_0deg_linear_pdf = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_linear.pdf"
full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_linear_png, dpi=220)
full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_linear_pdf)

if str(full_one_layer_tx_tau_sigma_high_s_0deg_log_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_high_s_0deg_log_source_key = 'abs_err_' + str(full_one_layer_tx_tau_sigma_high_s_0deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_tx_tau_sigma_high_s_0deg_log_source_key = full_one_layer_tx_tau_sigma_high_s_0deg_log_value_key
full_one_layer_tx_tau_sigma_high_s_0deg_log_grid = full_one_layer_tx_tau_sigma_high_s_0deg_positive_grid_for_log(full_one_layer_tx_tau_sigma_high_s_0deg_aggregate_grid(full_one_layer_tx_tau_sigma_high_s_0deg_log_source_key))
full_one_layer_tx_tau_sigma_high_s_0deg_log_fig, full_one_layer_tx_tau_sigma_high_s_0deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_high_s_0deg_log_finite = full_one_layer_tx_tau_sigma_high_s_0deg_log_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_s_0deg_log_grid)]
full_one_layer_tx_tau_sigma_high_s_0deg_log_positive = full_one_layer_tx_tau_sigma_high_s_0deg_log_finite[full_one_layer_tx_tau_sigma_high_s_0deg_log_finite > 0.0]
full_one_layer_tx_tau_sigma_high_s_0deg_log_vmin = float(np.min(full_one_layer_tx_tau_sigma_high_s_0deg_log_positive))
full_one_layer_tx_tau_sigma_high_s_0deg_log_vmax = float(np.max(full_one_layer_tx_tau_sigma_high_s_0deg_log_positive))
if math.isclose(full_one_layer_tx_tau_sigma_high_s_0deg_log_vmin, full_one_layer_tx_tau_sigma_high_s_0deg_log_vmax):
    full_one_layer_tx_tau_sigma_high_s_0deg_log_vmax = full_one_layer_tx_tau_sigma_high_s_0deg_log_vmin * 1.01
full_one_layer_tx_tau_sigma_high_s_0deg_log_levels = np.geomspace(full_one_layer_tx_tau_sigma_high_s_0deg_log_vmin, full_one_layer_tx_tau_sigma_high_s_0deg_log_vmax, 256)
full_one_layer_tx_tau_sigma_high_s_0deg_log_contour = full_one_layer_tx_tau_sigma_high_s_0deg_log_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_high_s_0deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_high_s_0deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_high_s_0deg_log_grid,
    levels=full_one_layer_tx_tau_sigma_high_s_0deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_tx_tau_sigma_high_s_0deg_log_vmin, vmax=full_one_layer_tx_tau_sigma_high_s_0deg_log_vmax),
    extend='both',
)
full_one_layer_tx_tau_sigma_high_s_0deg_log_ax.set_title(full_one_layer_tx_tau_sigma_high_s_0deg_plot_title + ' [log]')
full_one_layer_tx_tau_sigma_high_s_0deg_log_ax.set_xlabel(full_one_layer_tx_tau_sigma_high_s_0deg_x_label)
full_one_layer_tx_tau_sigma_high_s_0deg_log_ax.set_ylabel(full_one_layer_tx_tau_sigma_high_s_0deg_y_label)
full_one_layer_tx_tau_sigma_high_s_0deg_log_cbar = full_one_layer_tx_tau_sigma_high_s_0deg_log_fig.colorbar(full_one_layer_tx_tau_sigma_high_s_0deg_log_contour, ax=full_one_layer_tx_tau_sigma_high_s_0deg_log_ax)
full_one_layer_tx_tau_sigma_high_s_0deg_log_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_high_s_0deg_log_source_key))
full_one_layer_tx_tau_sigma_high_s_0deg_log_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_s_0deg_log_png = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_log.png"
full_one_layer_tx_tau_sigma_high_s_0deg_log_pdf = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_log.pdf"
full_one_layer_tx_tau_sigma_high_s_0deg_log_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_log_png, dpi=220)
full_one_layer_tx_tau_sigma_high_s_0deg_log_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_log_pdf)

full_one_layer_tx_tau_sigma_high_s_0deg_corr_rows = full_one_layer_tx_tau_sigma_high_s_0deg_corr_meta['rows']
full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels = sorted({str(full_one_layer_tx_tau_sigma_high_s_0deg_row['param_i']) for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_corr_rows} | {str(full_one_layer_tx_tau_sigma_high_s_0deg_row['param_j']) for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_corr_rows})
full_one_layer_tx_tau_sigma_high_s_0deg_corr_index = {full_one_layer_tx_tau_sigma_high_s_0deg_label: full_one_layer_tx_tau_sigma_high_s_0deg_idx for full_one_layer_tx_tau_sigma_high_s_0deg_idx, full_one_layer_tx_tau_sigma_high_s_0deg_label in enumerate(full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels)}
full_one_layer_tx_tau_sigma_high_s_0deg_corr_matrix = np.full((len(full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels), len(full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_tx_tau_sigma_high_s_0deg_row in full_one_layer_tx_tau_sigma_high_s_0deg_corr_rows:
    full_one_layer_tx_tau_sigma_high_s_0deg_i = full_one_layer_tx_tau_sigma_high_s_0deg_corr_index[str(full_one_layer_tx_tau_sigma_high_s_0deg_row['param_i'])]
    full_one_layer_tx_tau_sigma_high_s_0deg_j = full_one_layer_tx_tau_sigma_high_s_0deg_corr_index[str(full_one_layer_tx_tau_sigma_high_s_0deg_row['param_j'])]
    full_one_layer_tx_tau_sigma_high_s_0deg_corr_matrix[full_one_layer_tx_tau_sigma_high_s_0deg_i, full_one_layer_tx_tau_sigma_high_s_0deg_j] = float(full_one_layer_tx_tau_sigma_high_s_0deg_row['correlation'])
    full_one_layer_tx_tau_sigma_high_s_0deg_corr_matrix[full_one_layer_tx_tau_sigma_high_s_0deg_j, full_one_layer_tx_tau_sigma_high_s_0deg_i] = float(full_one_layer_tx_tau_sigma_high_s_0deg_row['correlation'])
for full_one_layer_tx_tau_sigma_high_s_0deg_diag in range(len(full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels)):
    full_one_layer_tx_tau_sigma_high_s_0deg_corr_matrix[full_one_layer_tx_tau_sigma_high_s_0deg_diag, full_one_layer_tx_tau_sigma_high_s_0deg_diag] = 1.0

full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig, full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_tx_tau_sigma_high_s_0deg_corr_image = full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax.imshow(full_one_layer_tx_tau_sigma_high_s_0deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax.set_title(full_one_layer_tx_tau_sigma_high_s_0deg_plot_title + ' [correlation]')
full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax.set_xticks(range(len(full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels)))
full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax.set_yticks(range(len(full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels)))
full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_high_s_0deg_label, full_one_layer_tx_tau_sigma_high_s_0deg_label) for full_one_layer_tx_tau_sigma_high_s_0deg_label in full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels], rotation=45, ha='right')
full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_high_s_0deg_label, full_one_layer_tx_tau_sigma_high_s_0deg_label) for full_one_layer_tx_tau_sigma_high_s_0deg_label in full_one_layer_tx_tau_sigma_high_s_0deg_corr_labels])
full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig.colorbar(full_one_layer_tx_tau_sigma_high_s_0deg_corr_image, ax=full_one_layer_tx_tau_sigma_high_s_0deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_s_0deg_corr_png = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_corr.png"
full_one_layer_tx_tau_sigma_high_s_0deg_corr_pdf = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_corr.pdf"
full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_corr_png, dpi=220)
full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_corr_pdf)

full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig, full_one_layer_tx_tau_sigma_high_s_0deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_tx_tau_sigma_high_s_0deg_axis, full_one_layer_tx_tau_sigma_high_s_0deg_image_path, full_one_layer_tx_tau_sigma_high_s_0deg_panel_title in zip(
    full_one_layer_tx_tau_sigma_high_s_0deg_triptych_axes,
    (full_one_layer_tx_tau_sigma_high_s_0deg_linear_png, full_one_layer_tx_tau_sigma_high_s_0deg_log_png, full_one_layer_tx_tau_sigma_high_s_0deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_tx_tau_sigma_high_s_0deg_image = plt.imread(str(full_one_layer_tx_tau_sigma_high_s_0deg_image_path))
    full_one_layer_tx_tau_sigma_high_s_0deg_axis.imshow(full_one_layer_tx_tau_sigma_high_s_0deg_image)
    full_one_layer_tx_tau_sigma_high_s_0deg_axis.set_title(full_one_layer_tx_tau_sigma_high_s_0deg_panel_title)
    full_one_layer_tx_tau_sigma_high_s_0deg_axis.axis('off')
full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig.suptitle(full_one_layer_tx_tau_sigma_high_s_0deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_s_0deg_triptych_png = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_triptych.png"
full_one_layer_tx_tau_sigma_high_s_0deg_triptych_pdf = full_one_layer_tx_tau_sigma_high_s_0deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_s_0deg_study_dir.name}__section_triptych.pdf"
full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_triptych_png, dpi=220)
full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_high_s_0deg_triptych_pdf)

display(full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig)
display(full_one_layer_tx_tau_sigma_high_s_0deg_log_fig)
display(full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig)
display(full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig)
plt.close(full_one_layer_tx_tau_sigma_high_s_0deg_linear_fig)
plt.close(full_one_layer_tx_tau_sigma_high_s_0deg_log_fig)
plt.close(full_one_layer_tx_tau_sigma_high_s_0deg_corr_fig)
plt.close(full_one_layer_tx_tau_sigma_high_s_0deg_triptych_fig)

full_one_layer_tx_tau_sigma_high_s_0deg_plot_paths = {
    'linear_png': full_one_layer_tx_tau_sigma_high_s_0deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_tx_tau_sigma_high_s_0deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_tx_tau_sigma_high_s_0deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_tx_tau_sigma_high_s_0deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_tx_tau_sigma_high_s_0deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_tx_tau_sigma_high_s_0deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_tx_tau_sigma_high_s_0deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_tx_tau_sigma_high_s_0deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_tx_tau_sigma_high_s_0deg_plot_paths

### One-layer Drude transmission: $\tau$ vs $\sigma$ (45 deg, p-pol, $d=725\,\mu$m)

In [ ]:
full_one_layer_tx_tau_sigma_high_p_45deg_spec = {
  "slug": "one_layer_tx_tau_sigma_high_p_45deg",
  "title": "One-layer Drude transmission: $\\tau$ vs $\\sigma$ (45 deg, p-pol, $d=725\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 45.0,
  "polarization": "p",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    100.0,
    1000.0
  ],
  "thickness_um": 725.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ]
}
full_one_layer_tx_tau_sigma_high_p_45deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_tx_tau_sigma_high_p_45deg'
full_one_layer_tx_tau_sigma_high_p_45deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_tx_tau_sigma_high_p_45deg_result = run_lecture_map_from_spec(full_one_layer_tx_tau_sigma_high_p_45deg_spec, output_root=full_one_layer_tx_tau_sigma_high_p_45deg_output_root, profile='full')
full_one_layer_tx_tau_sigma_high_p_45deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_tx_tau_sigma_high_p_45deg_study_dir = Path(full_one_layer_tx_tau_sigma_high_p_45deg_result['study_dir'])
full_one_layer_tx_tau_sigma_high_p_45deg_plot_title = full_one_layer_tx_tau_sigma_high_p_45deg_result['title']
full_one_layer_tx_tau_sigma_high_p_45deg_x_label = "$\\tau$ (ps)"
full_one_layer_tx_tau_sigma_high_p_45deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_high_p_45deg_summary_handle:
    full_one_layer_tx_tau_sigma_high_p_45deg_summary_meta = json.load(full_one_layer_tx_tau_sigma_high_p_45deg_summary_handle)
with (full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_tx_tau_sigma_high_p_45deg_corr_handle:
    full_one_layer_tx_tau_sigma_high_p_45deg_corr_meta = json.load(full_one_layer_tx_tau_sigma_high_p_45deg_corr_handle)

full_one_layer_tx_tau_sigma_high_p_45deg_rows = []
with (full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_tx_tau_sigma_high_p_45deg_summary_csv:
    full_one_layer_tx_tau_sigma_high_p_45deg_reader = csv.DictReader(full_one_layer_tx_tau_sigma_high_p_45deg_summary_csv)
    for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_reader:
        full_one_layer_tx_tau_sigma_high_p_45deg_parsed = {}
        for full_one_layer_tx_tau_sigma_high_p_45deg_key, full_one_layer_tx_tau_sigma_high_p_45deg_value in full_one_layer_tx_tau_sigma_high_p_45deg_row.items():
            if full_one_layer_tx_tau_sigma_high_p_45deg_value is None:
                full_one_layer_tx_tau_sigma_high_p_45deg_parsed[full_one_layer_tx_tau_sigma_high_p_45deg_key] = full_one_layer_tx_tau_sigma_high_p_45deg_value
                continue
            full_one_layer_tx_tau_sigma_high_p_45deg_text = str(full_one_layer_tx_tau_sigma_high_p_45deg_value).strip()
            if full_one_layer_tx_tau_sigma_high_p_45deg_text == '':
                full_one_layer_tx_tau_sigma_high_p_45deg_parsed[full_one_layer_tx_tau_sigma_high_p_45deg_key] = full_one_layer_tx_tau_sigma_high_p_45deg_text
                continue
            try:
                full_one_layer_tx_tau_sigma_high_p_45deg_parsed[full_one_layer_tx_tau_sigma_high_p_45deg_key] = float(full_one_layer_tx_tau_sigma_high_p_45deg_text)
            except ValueError:
                full_one_layer_tx_tau_sigma_high_p_45deg_parsed[full_one_layer_tx_tau_sigma_high_p_45deg_key] = full_one_layer_tx_tau_sigma_high_p_45deg_text
        full_one_layer_tx_tau_sigma_high_p_45deg_rows.append(full_one_layer_tx_tau_sigma_high_p_45deg_parsed)

full_one_layer_tx_tau_sigma_high_p_45deg_x_key, full_one_layer_tx_tau_sigma_high_p_45deg_y_key = list(full_one_layer_tx_tau_sigma_high_p_45deg_summary_meta['recovery_keys'])
full_one_layer_tx_tau_sigma_high_p_45deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_tx_tau_sigma_high_p_45deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_high_p_45deg_x_key}',
    f'{full_one_layer_tx_tau_sigma_high_p_45deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_tx_tau_sigma_high_p_45deg_y_key}',
    f'abs_{full_one_layer_tx_tau_sigma_high_p_45deg_x_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_high_p_45deg_x_key}',
    f'abs_{full_one_layer_tx_tau_sigma_high_p_45deg_y_key}_error': f'abs_err_{full_one_layer_tx_tau_sigma_high_p_45deg_y_key}',
}
full_one_layer_tx_tau_sigma_high_p_45deg_linear_value_key = full_one_layer_tx_tau_sigma_high_p_45deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_high_p_45deg_log_value_key = full_one_layer_tx_tau_sigma_high_p_45deg_metric_options['data_fit']
full_one_layer_tx_tau_sigma_high_p_45deg_x_values = sorted({float(full_one_layer_tx_tau_sigma_high_p_45deg_row[full_one_layer_tx_tau_sigma_high_p_45deg_x_key]) for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_rows})
full_one_layer_tx_tau_sigma_high_p_45deg_y_values = sorted({float(full_one_layer_tx_tau_sigma_high_p_45deg_row[full_one_layer_tx_tau_sigma_high_p_45deg_y_key]) for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_rows})

print('Available map metrics:')
for full_one_layer_tx_tau_sigma_high_p_45deg_option_name, full_one_layer_tx_tau_sigma_high_p_45deg_option_key in full_one_layer_tx_tau_sigma_high_p_45deg_metric_options.items():
    print(f"  {full_one_layer_tx_tau_sigma_high_p_45deg_option_name}: {metric_label(full_one_layer_tx_tau_sigma_high_p_45deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_tx_tau_sigma_high_p_45deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_tx_tau_sigma_high_p_45deg_log_value_key)}")

def full_one_layer_tx_tau_sigma_high_p_45deg_aggregate_grid(value_key):
    full_one_layer_tx_tau_sigma_high_p_45deg_grid = np.full((len(full_one_layer_tx_tau_sigma_high_p_45deg_y_values), len(full_one_layer_tx_tau_sigma_high_p_45deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_tx_tau_sigma_high_p_45deg_iy, full_one_layer_tx_tau_sigma_high_p_45deg_y_value in enumerate(full_one_layer_tx_tau_sigma_high_p_45deg_y_values):
        for full_one_layer_tx_tau_sigma_high_p_45deg_ix, full_one_layer_tx_tau_sigma_high_p_45deg_x_value in enumerate(full_one_layer_tx_tau_sigma_high_p_45deg_x_values):
            full_one_layer_tx_tau_sigma_high_p_45deg_cell = [
                float(full_one_layer_tx_tau_sigma_high_p_45deg_row[value_key])
                for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_rows
                if math.isclose(float(full_one_layer_tx_tau_sigma_high_p_45deg_row[full_one_layer_tx_tau_sigma_high_p_45deg_x_key]), full_one_layer_tx_tau_sigma_high_p_45deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_tx_tau_sigma_high_p_45deg_row[full_one_layer_tx_tau_sigma_high_p_45deg_y_key]), full_one_layer_tx_tau_sigma_high_p_45deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_tx_tau_sigma_high_p_45deg_row[value_key]))
            ]
            if full_one_layer_tx_tau_sigma_high_p_45deg_cell:
                full_one_layer_tx_tau_sigma_high_p_45deg_grid[full_one_layer_tx_tau_sigma_high_p_45deg_iy, full_one_layer_tx_tau_sigma_high_p_45deg_ix] = float(np.mean(full_one_layer_tx_tau_sigma_high_p_45deg_cell))
    return full_one_layer_tx_tau_sigma_high_p_45deg_grid

def full_one_layer_tx_tau_sigma_high_p_45deg_positive_grid_for_log(grid):
    full_one_layer_tx_tau_sigma_high_p_45deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_tx_tau_sigma_high_p_45deg_finite = full_one_layer_tx_tau_sigma_high_p_45deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_p_45deg_grid)]
    full_one_layer_tx_tau_sigma_high_p_45deg_positive = full_one_layer_tx_tau_sigma_high_p_45deg_finite[full_one_layer_tx_tau_sigma_high_p_45deg_finite > 0.0]
    if full_one_layer_tx_tau_sigma_high_p_45deg_positive.size == 0:
        full_one_layer_tx_tau_sigma_high_p_45deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_p_45deg_grid)] = 1.0
        return full_one_layer_tx_tau_sigma_high_p_45deg_grid
    full_one_layer_tx_tau_sigma_high_p_45deg_floor = max(float(np.min(full_one_layer_tx_tau_sigma_high_p_45deg_positive)) * 0.5, 1e-18)
    full_one_layer_tx_tau_sigma_high_p_45deg_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_p_45deg_grid) & (full_one_layer_tx_tau_sigma_high_p_45deg_grid <= 0.0)] = full_one_layer_tx_tau_sigma_high_p_45deg_floor
    return full_one_layer_tx_tau_sigma_high_p_45deg_grid

full_one_layer_tx_tau_sigma_high_p_45deg_linear_grid = full_one_layer_tx_tau_sigma_high_p_45deg_aggregate_grid(full_one_layer_tx_tau_sigma_high_p_45deg_linear_value_key)
full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig, full_one_layer_tx_tau_sigma_high_p_45deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_high_p_45deg_linear_finite = full_one_layer_tx_tau_sigma_high_p_45deg_linear_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_p_45deg_linear_grid)]
if str(full_one_layer_tx_tau_sigma_high_p_45deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_limit = max(float(np.max(np.abs(full_one_layer_tx_tau_sigma_high_p_45deg_linear_finite))), 1e-18)
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmin = -full_one_layer_tx_tau_sigma_high_p_45deg_linear_limit
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmax = full_one_layer_tx_tau_sigma_high_p_45deg_linear_limit
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmin, full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_cmap = 'plasma'
else:
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmin = float(np.min(full_one_layer_tx_tau_sigma_high_p_45deg_linear_finite))
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmax = float(np.max(full_one_layer_tx_tau_sigma_high_p_45deg_linear_finite)) + 1e-12
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_levels = np.linspace(full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmin, full_one_layer_tx_tau_sigma_high_p_45deg_linear_vmax, 256)
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_cmap = 'plasma'
full_one_layer_tx_tau_sigma_high_p_45deg_linear_contour = full_one_layer_tx_tau_sigma_high_p_45deg_linear_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_high_p_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_high_p_45deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_high_p_45deg_linear_grid,
    levels=full_one_layer_tx_tau_sigma_high_p_45deg_linear_levels,
    cmap=full_one_layer_tx_tau_sigma_high_p_45deg_linear_cmap,
    extend='both',
)
full_one_layer_tx_tau_sigma_high_p_45deg_linear_ax.set_title(full_one_layer_tx_tau_sigma_high_p_45deg_plot_title + ' [linear]')
full_one_layer_tx_tau_sigma_high_p_45deg_linear_ax.set_xlabel(full_one_layer_tx_tau_sigma_high_p_45deg_x_label)
full_one_layer_tx_tau_sigma_high_p_45deg_linear_ax.set_ylabel(full_one_layer_tx_tau_sigma_high_p_45deg_y_label)
full_one_layer_tx_tau_sigma_high_p_45deg_linear_cbar = full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig.colorbar(full_one_layer_tx_tau_sigma_high_p_45deg_linear_contour, ax=full_one_layer_tx_tau_sigma_high_p_45deg_linear_ax)
full_one_layer_tx_tau_sigma_high_p_45deg_linear_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_high_p_45deg_linear_value_key))
full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_p_45deg_linear_png = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_linear.png"
full_one_layer_tx_tau_sigma_high_p_45deg_linear_pdf = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_linear.pdf"
full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_linear_png, dpi=220)
full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_linear_pdf)

if str(full_one_layer_tx_tau_sigma_high_p_45deg_log_value_key).startswith('signed_err_'):
    full_one_layer_tx_tau_sigma_high_p_45deg_log_source_key = 'abs_err_' + str(full_one_layer_tx_tau_sigma_high_p_45deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_tx_tau_sigma_high_p_45deg_log_source_key = full_one_layer_tx_tau_sigma_high_p_45deg_log_value_key
full_one_layer_tx_tau_sigma_high_p_45deg_log_grid = full_one_layer_tx_tau_sigma_high_p_45deg_positive_grid_for_log(full_one_layer_tx_tau_sigma_high_p_45deg_aggregate_grid(full_one_layer_tx_tau_sigma_high_p_45deg_log_source_key))
full_one_layer_tx_tau_sigma_high_p_45deg_log_fig, full_one_layer_tx_tau_sigma_high_p_45deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_tx_tau_sigma_high_p_45deg_log_finite = full_one_layer_tx_tau_sigma_high_p_45deg_log_grid[np.isfinite(full_one_layer_tx_tau_sigma_high_p_45deg_log_grid)]
full_one_layer_tx_tau_sigma_high_p_45deg_log_positive = full_one_layer_tx_tau_sigma_high_p_45deg_log_finite[full_one_layer_tx_tau_sigma_high_p_45deg_log_finite > 0.0]
full_one_layer_tx_tau_sigma_high_p_45deg_log_vmin = float(np.min(full_one_layer_tx_tau_sigma_high_p_45deg_log_positive))
full_one_layer_tx_tau_sigma_high_p_45deg_log_vmax = float(np.max(full_one_layer_tx_tau_sigma_high_p_45deg_log_positive))
if math.isclose(full_one_layer_tx_tau_sigma_high_p_45deg_log_vmin, full_one_layer_tx_tau_sigma_high_p_45deg_log_vmax):
    full_one_layer_tx_tau_sigma_high_p_45deg_log_vmax = full_one_layer_tx_tau_sigma_high_p_45deg_log_vmin * 1.01
full_one_layer_tx_tau_sigma_high_p_45deg_log_levels = np.geomspace(full_one_layer_tx_tau_sigma_high_p_45deg_log_vmin, full_one_layer_tx_tau_sigma_high_p_45deg_log_vmax, 256)
full_one_layer_tx_tau_sigma_high_p_45deg_log_contour = full_one_layer_tx_tau_sigma_high_p_45deg_log_ax.contourf(
    np.asarray(full_one_layer_tx_tau_sigma_high_p_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_tx_tau_sigma_high_p_45deg_y_values, dtype=np.float64),
    full_one_layer_tx_tau_sigma_high_p_45deg_log_grid,
    levels=full_one_layer_tx_tau_sigma_high_p_45deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_tx_tau_sigma_high_p_45deg_log_vmin, vmax=full_one_layer_tx_tau_sigma_high_p_45deg_log_vmax),
    extend='both',
)
full_one_layer_tx_tau_sigma_high_p_45deg_log_ax.set_title(full_one_layer_tx_tau_sigma_high_p_45deg_plot_title + ' [log]')
full_one_layer_tx_tau_sigma_high_p_45deg_log_ax.set_xlabel(full_one_layer_tx_tau_sigma_high_p_45deg_x_label)
full_one_layer_tx_tau_sigma_high_p_45deg_log_ax.set_ylabel(full_one_layer_tx_tau_sigma_high_p_45deg_y_label)
full_one_layer_tx_tau_sigma_high_p_45deg_log_cbar = full_one_layer_tx_tau_sigma_high_p_45deg_log_fig.colorbar(full_one_layer_tx_tau_sigma_high_p_45deg_log_contour, ax=full_one_layer_tx_tau_sigma_high_p_45deg_log_ax)
full_one_layer_tx_tau_sigma_high_p_45deg_log_cbar.set_label(metric_label(full_one_layer_tx_tau_sigma_high_p_45deg_log_source_key))
full_one_layer_tx_tau_sigma_high_p_45deg_log_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_p_45deg_log_png = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_log.png"
full_one_layer_tx_tau_sigma_high_p_45deg_log_pdf = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_log.pdf"
full_one_layer_tx_tau_sigma_high_p_45deg_log_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_log_png, dpi=220)
full_one_layer_tx_tau_sigma_high_p_45deg_log_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_log_pdf)

full_one_layer_tx_tau_sigma_high_p_45deg_corr_rows = full_one_layer_tx_tau_sigma_high_p_45deg_corr_meta['rows']
full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels = sorted({str(full_one_layer_tx_tau_sigma_high_p_45deg_row['param_i']) for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_corr_rows} | {str(full_one_layer_tx_tau_sigma_high_p_45deg_row['param_j']) for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_corr_rows})
full_one_layer_tx_tau_sigma_high_p_45deg_corr_index = {full_one_layer_tx_tau_sigma_high_p_45deg_label: full_one_layer_tx_tau_sigma_high_p_45deg_idx for full_one_layer_tx_tau_sigma_high_p_45deg_idx, full_one_layer_tx_tau_sigma_high_p_45deg_label in enumerate(full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels)}
full_one_layer_tx_tau_sigma_high_p_45deg_corr_matrix = np.full((len(full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels), len(full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_tx_tau_sigma_high_p_45deg_row in full_one_layer_tx_tau_sigma_high_p_45deg_corr_rows:
    full_one_layer_tx_tau_sigma_high_p_45deg_i = full_one_layer_tx_tau_sigma_high_p_45deg_corr_index[str(full_one_layer_tx_tau_sigma_high_p_45deg_row['param_i'])]
    full_one_layer_tx_tau_sigma_high_p_45deg_j = full_one_layer_tx_tau_sigma_high_p_45deg_corr_index[str(full_one_layer_tx_tau_sigma_high_p_45deg_row['param_j'])]
    full_one_layer_tx_tau_sigma_high_p_45deg_corr_matrix[full_one_layer_tx_tau_sigma_high_p_45deg_i, full_one_layer_tx_tau_sigma_high_p_45deg_j] = float(full_one_layer_tx_tau_sigma_high_p_45deg_row['correlation'])
    full_one_layer_tx_tau_sigma_high_p_45deg_corr_matrix[full_one_layer_tx_tau_sigma_high_p_45deg_j, full_one_layer_tx_tau_sigma_high_p_45deg_i] = float(full_one_layer_tx_tau_sigma_high_p_45deg_row['correlation'])
for full_one_layer_tx_tau_sigma_high_p_45deg_diag in range(len(full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels)):
    full_one_layer_tx_tau_sigma_high_p_45deg_corr_matrix[full_one_layer_tx_tau_sigma_high_p_45deg_diag, full_one_layer_tx_tau_sigma_high_p_45deg_diag] = 1.0

full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig, full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_tx_tau_sigma_high_p_45deg_corr_image = full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax.imshow(full_one_layer_tx_tau_sigma_high_p_45deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax.set_title(full_one_layer_tx_tau_sigma_high_p_45deg_plot_title + ' [correlation]')
full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax.set_xticks(range(len(full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels)))
full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax.set_yticks(range(len(full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels)))
full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_high_p_45deg_label, full_one_layer_tx_tau_sigma_high_p_45deg_label) for full_one_layer_tx_tau_sigma_high_p_45deg_label in full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels], rotation=45, ha='right')
full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_tx_tau_sigma_high_p_45deg_label, full_one_layer_tx_tau_sigma_high_p_45deg_label) for full_one_layer_tx_tau_sigma_high_p_45deg_label in full_one_layer_tx_tau_sigma_high_p_45deg_corr_labels])
full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig.colorbar(full_one_layer_tx_tau_sigma_high_p_45deg_corr_image, ax=full_one_layer_tx_tau_sigma_high_p_45deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_p_45deg_corr_png = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_corr.png"
full_one_layer_tx_tau_sigma_high_p_45deg_corr_pdf = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_corr.pdf"
full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_corr_png, dpi=220)
full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_corr_pdf)

full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig, full_one_layer_tx_tau_sigma_high_p_45deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_tx_tau_sigma_high_p_45deg_axis, full_one_layer_tx_tau_sigma_high_p_45deg_image_path, full_one_layer_tx_tau_sigma_high_p_45deg_panel_title in zip(
    full_one_layer_tx_tau_sigma_high_p_45deg_triptych_axes,
    (full_one_layer_tx_tau_sigma_high_p_45deg_linear_png, full_one_layer_tx_tau_sigma_high_p_45deg_log_png, full_one_layer_tx_tau_sigma_high_p_45deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_tx_tau_sigma_high_p_45deg_image = plt.imread(str(full_one_layer_tx_tau_sigma_high_p_45deg_image_path))
    full_one_layer_tx_tau_sigma_high_p_45deg_axis.imshow(full_one_layer_tx_tau_sigma_high_p_45deg_image)
    full_one_layer_tx_tau_sigma_high_p_45deg_axis.set_title(full_one_layer_tx_tau_sigma_high_p_45deg_panel_title)
    full_one_layer_tx_tau_sigma_high_p_45deg_axis.axis('off')
full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig.suptitle(full_one_layer_tx_tau_sigma_high_p_45deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig.tight_layout()
full_one_layer_tx_tau_sigma_high_p_45deg_triptych_png = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_triptych.png"
full_one_layer_tx_tau_sigma_high_p_45deg_triptych_pdf = full_one_layer_tx_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_tx_tau_sigma_high_p_45deg_study_dir.name}__section_triptych.pdf"
full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_triptych_png, dpi=220)
full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig.savefig(full_one_layer_tx_tau_sigma_high_p_45deg_triptych_pdf)

display(full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig)
display(full_one_layer_tx_tau_sigma_high_p_45deg_log_fig)
display(full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig)
display(full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig)
plt.close(full_one_layer_tx_tau_sigma_high_p_45deg_linear_fig)
plt.close(full_one_layer_tx_tau_sigma_high_p_45deg_log_fig)
plt.close(full_one_layer_tx_tau_sigma_high_p_45deg_corr_fig)
plt.close(full_one_layer_tx_tau_sigma_high_p_45deg_triptych_fig)

full_one_layer_tx_tau_sigma_high_p_45deg_plot_paths = {
    'linear_png': full_one_layer_tx_tau_sigma_high_p_45deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_tx_tau_sigma_high_p_45deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_tx_tau_sigma_high_p_45deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_tx_tau_sigma_high_p_45deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_tx_tau_sigma_high_p_45deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_tx_tau_sigma_high_p_45deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_tx_tau_sigma_high_p_45deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_tx_tau_sigma_high_p_45deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_tx_tau_sigma_high_p_45deg_plot_paths

### One-layer Drude reflection: $\tau$ vs $\sigma$ (45 deg, s-pol, $d=525\,\mu$m)

In [ ]:
full_one_layer_refl_tau_sigma_low_s_45deg_spec = {
  "slug": "one_layer_refl_tau_sigma_low_s_45deg",
  "title": "One-layer Drude reflection: $\\tau$ vs $\\sigma$ (45 deg, s-pol, $d=525\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 45.0,
  "polarization": "s",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    0.01,
    1.0
  ],
  "thickness_um": 525.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
full_one_layer_refl_tau_sigma_low_s_45deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_refl_tau_sigma_low_s_45deg'
full_one_layer_refl_tau_sigma_low_s_45deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_refl_tau_sigma_low_s_45deg_result = run_lecture_map_from_spec(full_one_layer_refl_tau_sigma_low_s_45deg_spec, output_root=full_one_layer_refl_tau_sigma_low_s_45deg_output_root, profile='full')
full_one_layer_refl_tau_sigma_low_s_45deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_refl_tau_sigma_low_s_45deg_study_dir = Path(full_one_layer_refl_tau_sigma_low_s_45deg_result['study_dir'])
full_one_layer_refl_tau_sigma_low_s_45deg_plot_title = full_one_layer_refl_tau_sigma_low_s_45deg_result['title']
full_one_layer_refl_tau_sigma_low_s_45deg_x_label = "$\\tau$ (ps)"
full_one_layer_refl_tau_sigma_low_s_45deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_low_s_45deg_summary_handle:
    full_one_layer_refl_tau_sigma_low_s_45deg_summary_meta = json.load(full_one_layer_refl_tau_sigma_low_s_45deg_summary_handle)
with (full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_low_s_45deg_corr_handle:
    full_one_layer_refl_tau_sigma_low_s_45deg_corr_meta = json.load(full_one_layer_refl_tau_sigma_low_s_45deg_corr_handle)

full_one_layer_refl_tau_sigma_low_s_45deg_rows = []
with (full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_refl_tau_sigma_low_s_45deg_summary_csv:
    full_one_layer_refl_tau_sigma_low_s_45deg_reader = csv.DictReader(full_one_layer_refl_tau_sigma_low_s_45deg_summary_csv)
    for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_reader:
        full_one_layer_refl_tau_sigma_low_s_45deg_parsed = {}
        for full_one_layer_refl_tau_sigma_low_s_45deg_key, full_one_layer_refl_tau_sigma_low_s_45deg_value in full_one_layer_refl_tau_sigma_low_s_45deg_row.items():
            if full_one_layer_refl_tau_sigma_low_s_45deg_value is None:
                full_one_layer_refl_tau_sigma_low_s_45deg_parsed[full_one_layer_refl_tau_sigma_low_s_45deg_key] = full_one_layer_refl_tau_sigma_low_s_45deg_value
                continue
            full_one_layer_refl_tau_sigma_low_s_45deg_text = str(full_one_layer_refl_tau_sigma_low_s_45deg_value).strip()
            if full_one_layer_refl_tau_sigma_low_s_45deg_text == '':
                full_one_layer_refl_tau_sigma_low_s_45deg_parsed[full_one_layer_refl_tau_sigma_low_s_45deg_key] = full_one_layer_refl_tau_sigma_low_s_45deg_text
                continue
            try:
                full_one_layer_refl_tau_sigma_low_s_45deg_parsed[full_one_layer_refl_tau_sigma_low_s_45deg_key] = float(full_one_layer_refl_tau_sigma_low_s_45deg_text)
            except ValueError:
                full_one_layer_refl_tau_sigma_low_s_45deg_parsed[full_one_layer_refl_tau_sigma_low_s_45deg_key] = full_one_layer_refl_tau_sigma_low_s_45deg_text
        full_one_layer_refl_tau_sigma_low_s_45deg_rows.append(full_one_layer_refl_tau_sigma_low_s_45deg_parsed)

full_one_layer_refl_tau_sigma_low_s_45deg_x_key, full_one_layer_refl_tau_sigma_low_s_45deg_y_key = list(full_one_layer_refl_tau_sigma_low_s_45deg_summary_meta['recovery_keys'])
full_one_layer_refl_tau_sigma_low_s_45deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_refl_tau_sigma_low_s_45deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_low_s_45deg_x_key}',
    f'{full_one_layer_refl_tau_sigma_low_s_45deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_low_s_45deg_y_key}',
    f'abs_{full_one_layer_refl_tau_sigma_low_s_45deg_x_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_low_s_45deg_x_key}',
    f'abs_{full_one_layer_refl_tau_sigma_low_s_45deg_y_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_low_s_45deg_y_key}',
}
full_one_layer_refl_tau_sigma_low_s_45deg_linear_value_key = full_one_layer_refl_tau_sigma_low_s_45deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_low_s_45deg_log_value_key = full_one_layer_refl_tau_sigma_low_s_45deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_low_s_45deg_x_values = sorted({float(full_one_layer_refl_tau_sigma_low_s_45deg_row[full_one_layer_refl_tau_sigma_low_s_45deg_x_key]) for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_rows})
full_one_layer_refl_tau_sigma_low_s_45deg_y_values = sorted({float(full_one_layer_refl_tau_sigma_low_s_45deg_row[full_one_layer_refl_tau_sigma_low_s_45deg_y_key]) for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_rows})

print('Available map metrics:')
for full_one_layer_refl_tau_sigma_low_s_45deg_option_name, full_one_layer_refl_tau_sigma_low_s_45deg_option_key in full_one_layer_refl_tau_sigma_low_s_45deg_metric_options.items():
    print(f"  {full_one_layer_refl_tau_sigma_low_s_45deg_option_name}: {metric_label(full_one_layer_refl_tau_sigma_low_s_45deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_refl_tau_sigma_low_s_45deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_refl_tau_sigma_low_s_45deg_log_value_key)}")

def full_one_layer_refl_tau_sigma_low_s_45deg_aggregate_grid(value_key):
    full_one_layer_refl_tau_sigma_low_s_45deg_grid = np.full((len(full_one_layer_refl_tau_sigma_low_s_45deg_y_values), len(full_one_layer_refl_tau_sigma_low_s_45deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_refl_tau_sigma_low_s_45deg_iy, full_one_layer_refl_tau_sigma_low_s_45deg_y_value in enumerate(full_one_layer_refl_tau_sigma_low_s_45deg_y_values):
        for full_one_layer_refl_tau_sigma_low_s_45deg_ix, full_one_layer_refl_tau_sigma_low_s_45deg_x_value in enumerate(full_one_layer_refl_tau_sigma_low_s_45deg_x_values):
            full_one_layer_refl_tau_sigma_low_s_45deg_cell = [
                float(full_one_layer_refl_tau_sigma_low_s_45deg_row[value_key])
                for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_rows
                if math.isclose(float(full_one_layer_refl_tau_sigma_low_s_45deg_row[full_one_layer_refl_tau_sigma_low_s_45deg_x_key]), full_one_layer_refl_tau_sigma_low_s_45deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_refl_tau_sigma_low_s_45deg_row[full_one_layer_refl_tau_sigma_low_s_45deg_y_key]), full_one_layer_refl_tau_sigma_low_s_45deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_refl_tau_sigma_low_s_45deg_row[value_key]))
            ]
            if full_one_layer_refl_tau_sigma_low_s_45deg_cell:
                full_one_layer_refl_tau_sigma_low_s_45deg_grid[full_one_layer_refl_tau_sigma_low_s_45deg_iy, full_one_layer_refl_tau_sigma_low_s_45deg_ix] = float(np.mean(full_one_layer_refl_tau_sigma_low_s_45deg_cell))
    return full_one_layer_refl_tau_sigma_low_s_45deg_grid

def full_one_layer_refl_tau_sigma_low_s_45deg_positive_grid_for_log(grid):
    full_one_layer_refl_tau_sigma_low_s_45deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_refl_tau_sigma_low_s_45deg_finite = full_one_layer_refl_tau_sigma_low_s_45deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_45deg_grid)]
    full_one_layer_refl_tau_sigma_low_s_45deg_positive = full_one_layer_refl_tau_sigma_low_s_45deg_finite[full_one_layer_refl_tau_sigma_low_s_45deg_finite > 0.0]
    if full_one_layer_refl_tau_sigma_low_s_45deg_positive.size == 0:
        full_one_layer_refl_tau_sigma_low_s_45deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_45deg_grid)] = 1.0
        return full_one_layer_refl_tau_sigma_low_s_45deg_grid
    full_one_layer_refl_tau_sigma_low_s_45deg_floor = max(float(np.min(full_one_layer_refl_tau_sigma_low_s_45deg_positive)) * 0.5, 1e-18)
    full_one_layer_refl_tau_sigma_low_s_45deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_45deg_grid) & (full_one_layer_refl_tau_sigma_low_s_45deg_grid <= 0.0)] = full_one_layer_refl_tau_sigma_low_s_45deg_floor
    return full_one_layer_refl_tau_sigma_low_s_45deg_grid

full_one_layer_refl_tau_sigma_low_s_45deg_linear_grid = full_one_layer_refl_tau_sigma_low_s_45deg_aggregate_grid(full_one_layer_refl_tau_sigma_low_s_45deg_linear_value_key)
full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig, full_one_layer_refl_tau_sigma_low_s_45deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_low_s_45deg_linear_finite = full_one_layer_refl_tau_sigma_low_s_45deg_linear_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_45deg_linear_grid)]
if str(full_one_layer_refl_tau_sigma_low_s_45deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_limit = max(float(np.max(np.abs(full_one_layer_refl_tau_sigma_low_s_45deg_linear_finite))), 1e-18)
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmin = -full_one_layer_refl_tau_sigma_low_s_45deg_linear_limit
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmax = full_one_layer_refl_tau_sigma_low_s_45deg_linear_limit
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmin, full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_cmap = 'plasma'
else:
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmin = float(np.min(full_one_layer_refl_tau_sigma_low_s_45deg_linear_finite))
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmax = float(np.max(full_one_layer_refl_tau_sigma_low_s_45deg_linear_finite)) + 1e-12
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmin, full_one_layer_refl_tau_sigma_low_s_45deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_cmap = 'plasma'
full_one_layer_refl_tau_sigma_low_s_45deg_linear_contour = full_one_layer_refl_tau_sigma_low_s_45deg_linear_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_low_s_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_low_s_45deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_low_s_45deg_linear_grid,
    levels=full_one_layer_refl_tau_sigma_low_s_45deg_linear_levels,
    cmap=full_one_layer_refl_tau_sigma_low_s_45deg_linear_cmap,
    extend='both',
)
full_one_layer_refl_tau_sigma_low_s_45deg_linear_ax.set_title(full_one_layer_refl_tau_sigma_low_s_45deg_plot_title + ' [linear]')
full_one_layer_refl_tau_sigma_low_s_45deg_linear_ax.set_xlabel(full_one_layer_refl_tau_sigma_low_s_45deg_x_label)
full_one_layer_refl_tau_sigma_low_s_45deg_linear_ax.set_ylabel(full_one_layer_refl_tau_sigma_low_s_45deg_y_label)
full_one_layer_refl_tau_sigma_low_s_45deg_linear_cbar = full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig.colorbar(full_one_layer_refl_tau_sigma_low_s_45deg_linear_contour, ax=full_one_layer_refl_tau_sigma_low_s_45deg_linear_ax)
full_one_layer_refl_tau_sigma_low_s_45deg_linear_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_low_s_45deg_linear_value_key))
full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_45deg_linear_png = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_linear.png"
full_one_layer_refl_tau_sigma_low_s_45deg_linear_pdf = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_linear.pdf"
full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_linear_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_linear_pdf)

if str(full_one_layer_refl_tau_sigma_low_s_45deg_log_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_low_s_45deg_log_source_key = 'abs_err_' + str(full_one_layer_refl_tau_sigma_low_s_45deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_refl_tau_sigma_low_s_45deg_log_source_key = full_one_layer_refl_tau_sigma_low_s_45deg_log_value_key
full_one_layer_refl_tau_sigma_low_s_45deg_log_grid = full_one_layer_refl_tau_sigma_low_s_45deg_positive_grid_for_log(full_one_layer_refl_tau_sigma_low_s_45deg_aggregate_grid(full_one_layer_refl_tau_sigma_low_s_45deg_log_source_key))
full_one_layer_refl_tau_sigma_low_s_45deg_log_fig, full_one_layer_refl_tau_sigma_low_s_45deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_low_s_45deg_log_finite = full_one_layer_refl_tau_sigma_low_s_45deg_log_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_45deg_log_grid)]
full_one_layer_refl_tau_sigma_low_s_45deg_log_positive = full_one_layer_refl_tau_sigma_low_s_45deg_log_finite[full_one_layer_refl_tau_sigma_low_s_45deg_log_finite > 0.0]
full_one_layer_refl_tau_sigma_low_s_45deg_log_vmin = float(np.min(full_one_layer_refl_tau_sigma_low_s_45deg_log_positive))
full_one_layer_refl_tau_sigma_low_s_45deg_log_vmax = float(np.max(full_one_layer_refl_tau_sigma_low_s_45deg_log_positive))
if math.isclose(full_one_layer_refl_tau_sigma_low_s_45deg_log_vmin, full_one_layer_refl_tau_sigma_low_s_45deg_log_vmax):
    full_one_layer_refl_tau_sigma_low_s_45deg_log_vmax = full_one_layer_refl_tau_sigma_low_s_45deg_log_vmin * 1.01
full_one_layer_refl_tau_sigma_low_s_45deg_log_levels = np.geomspace(full_one_layer_refl_tau_sigma_low_s_45deg_log_vmin, full_one_layer_refl_tau_sigma_low_s_45deg_log_vmax, 256)
full_one_layer_refl_tau_sigma_low_s_45deg_log_contour = full_one_layer_refl_tau_sigma_low_s_45deg_log_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_low_s_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_low_s_45deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_low_s_45deg_log_grid,
    levels=full_one_layer_refl_tau_sigma_low_s_45deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_refl_tau_sigma_low_s_45deg_log_vmin, vmax=full_one_layer_refl_tau_sigma_low_s_45deg_log_vmax),
    extend='both',
)
full_one_layer_refl_tau_sigma_low_s_45deg_log_ax.set_title(full_one_layer_refl_tau_sigma_low_s_45deg_plot_title + ' [log]')
full_one_layer_refl_tau_sigma_low_s_45deg_log_ax.set_xlabel(full_one_layer_refl_tau_sigma_low_s_45deg_x_label)
full_one_layer_refl_tau_sigma_low_s_45deg_log_ax.set_ylabel(full_one_layer_refl_tau_sigma_low_s_45deg_y_label)
full_one_layer_refl_tau_sigma_low_s_45deg_log_cbar = full_one_layer_refl_tau_sigma_low_s_45deg_log_fig.colorbar(full_one_layer_refl_tau_sigma_low_s_45deg_log_contour, ax=full_one_layer_refl_tau_sigma_low_s_45deg_log_ax)
full_one_layer_refl_tau_sigma_low_s_45deg_log_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_low_s_45deg_log_source_key))
full_one_layer_refl_tau_sigma_low_s_45deg_log_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_45deg_log_png = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_log.png"
full_one_layer_refl_tau_sigma_low_s_45deg_log_pdf = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_log.pdf"
full_one_layer_refl_tau_sigma_low_s_45deg_log_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_log_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_45deg_log_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_log_pdf)

full_one_layer_refl_tau_sigma_low_s_45deg_corr_rows = full_one_layer_refl_tau_sigma_low_s_45deg_corr_meta['rows']
full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels = sorted({str(full_one_layer_refl_tau_sigma_low_s_45deg_row['param_i']) for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_corr_rows} | {str(full_one_layer_refl_tau_sigma_low_s_45deg_row['param_j']) for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_corr_rows})
full_one_layer_refl_tau_sigma_low_s_45deg_corr_index = {full_one_layer_refl_tau_sigma_low_s_45deg_label: full_one_layer_refl_tau_sigma_low_s_45deg_idx for full_one_layer_refl_tau_sigma_low_s_45deg_idx, full_one_layer_refl_tau_sigma_low_s_45deg_label in enumerate(full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels)}
full_one_layer_refl_tau_sigma_low_s_45deg_corr_matrix = np.full((len(full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels), len(full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_refl_tau_sigma_low_s_45deg_row in full_one_layer_refl_tau_sigma_low_s_45deg_corr_rows:
    full_one_layer_refl_tau_sigma_low_s_45deg_i = full_one_layer_refl_tau_sigma_low_s_45deg_corr_index[str(full_one_layer_refl_tau_sigma_low_s_45deg_row['param_i'])]
    full_one_layer_refl_tau_sigma_low_s_45deg_j = full_one_layer_refl_tau_sigma_low_s_45deg_corr_index[str(full_one_layer_refl_tau_sigma_low_s_45deg_row['param_j'])]
    full_one_layer_refl_tau_sigma_low_s_45deg_corr_matrix[full_one_layer_refl_tau_sigma_low_s_45deg_i, full_one_layer_refl_tau_sigma_low_s_45deg_j] = float(full_one_layer_refl_tau_sigma_low_s_45deg_row['correlation'])
    full_one_layer_refl_tau_sigma_low_s_45deg_corr_matrix[full_one_layer_refl_tau_sigma_low_s_45deg_j, full_one_layer_refl_tau_sigma_low_s_45deg_i] = float(full_one_layer_refl_tau_sigma_low_s_45deg_row['correlation'])
for full_one_layer_refl_tau_sigma_low_s_45deg_diag in range(len(full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels)):
    full_one_layer_refl_tau_sigma_low_s_45deg_corr_matrix[full_one_layer_refl_tau_sigma_low_s_45deg_diag, full_one_layer_refl_tau_sigma_low_s_45deg_diag] = 1.0

full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig, full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_refl_tau_sigma_low_s_45deg_corr_image = full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax.imshow(full_one_layer_refl_tau_sigma_low_s_45deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax.set_title(full_one_layer_refl_tau_sigma_low_s_45deg_plot_title + ' [correlation]')
full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax.set_xticks(range(len(full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels)))
full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax.set_yticks(range(len(full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels)))
full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_low_s_45deg_label, full_one_layer_refl_tau_sigma_low_s_45deg_label) for full_one_layer_refl_tau_sigma_low_s_45deg_label in full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels], rotation=45, ha='right')
full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_low_s_45deg_label, full_one_layer_refl_tau_sigma_low_s_45deg_label) for full_one_layer_refl_tau_sigma_low_s_45deg_label in full_one_layer_refl_tau_sigma_low_s_45deg_corr_labels])
full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig.colorbar(full_one_layer_refl_tau_sigma_low_s_45deg_corr_image, ax=full_one_layer_refl_tau_sigma_low_s_45deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_45deg_corr_png = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_corr.png"
full_one_layer_refl_tau_sigma_low_s_45deg_corr_pdf = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_corr.pdf"
full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_corr_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_corr_pdf)

full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig, full_one_layer_refl_tau_sigma_low_s_45deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_refl_tau_sigma_low_s_45deg_axis, full_one_layer_refl_tau_sigma_low_s_45deg_image_path, full_one_layer_refl_tau_sigma_low_s_45deg_panel_title in zip(
    full_one_layer_refl_tau_sigma_low_s_45deg_triptych_axes,
    (full_one_layer_refl_tau_sigma_low_s_45deg_linear_png, full_one_layer_refl_tau_sigma_low_s_45deg_log_png, full_one_layer_refl_tau_sigma_low_s_45deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_refl_tau_sigma_low_s_45deg_image = plt.imread(str(full_one_layer_refl_tau_sigma_low_s_45deg_image_path))
    full_one_layer_refl_tau_sigma_low_s_45deg_axis.imshow(full_one_layer_refl_tau_sigma_low_s_45deg_image)
    full_one_layer_refl_tau_sigma_low_s_45deg_axis.set_title(full_one_layer_refl_tau_sigma_low_s_45deg_panel_title)
    full_one_layer_refl_tau_sigma_low_s_45deg_axis.axis('off')
full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig.suptitle(full_one_layer_refl_tau_sigma_low_s_45deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_45deg_triptych_png = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_triptych.png"
full_one_layer_refl_tau_sigma_low_s_45deg_triptych_pdf = full_one_layer_refl_tau_sigma_low_s_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_45deg_study_dir.name}__section_triptych.pdf"
full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_triptych_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_low_s_45deg_triptych_pdf)

display(full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig)
display(full_one_layer_refl_tau_sigma_low_s_45deg_log_fig)
display(full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig)
display(full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_45deg_linear_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_45deg_log_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_45deg_corr_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_45deg_triptych_fig)

full_one_layer_refl_tau_sigma_low_s_45deg_plot_paths = {
    'linear_png': full_one_layer_refl_tau_sigma_low_s_45deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_refl_tau_sigma_low_s_45deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_refl_tau_sigma_low_s_45deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_refl_tau_sigma_low_s_45deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_refl_tau_sigma_low_s_45deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_refl_tau_sigma_low_s_45deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_refl_tau_sigma_low_s_45deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_refl_tau_sigma_low_s_45deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_refl_tau_sigma_low_s_45deg_plot_paths

### One-layer Drude reflection: $\tau$ vs $\sigma$ (60 deg, s-pol, $d=525\,\mu$m)

In [ ]:
full_one_layer_refl_tau_sigma_low_s_60deg_spec = {
  "slug": "one_layer_refl_tau_sigma_low_s_60deg",
  "title": "One-layer Drude reflection: $\\tau$ vs $\\sigma$ (60 deg, s-pol, $d=525\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 60.0,
  "polarization": "s",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    0.01,
    1.0
  ],
  "thickness_um": 525.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
full_one_layer_refl_tau_sigma_low_s_60deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_refl_tau_sigma_low_s_60deg'
full_one_layer_refl_tau_sigma_low_s_60deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_refl_tau_sigma_low_s_60deg_result = run_lecture_map_from_spec(full_one_layer_refl_tau_sigma_low_s_60deg_spec, output_root=full_one_layer_refl_tau_sigma_low_s_60deg_output_root, profile='full')
full_one_layer_refl_tau_sigma_low_s_60deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_refl_tau_sigma_low_s_60deg_study_dir = Path(full_one_layer_refl_tau_sigma_low_s_60deg_result['study_dir'])
full_one_layer_refl_tau_sigma_low_s_60deg_plot_title = full_one_layer_refl_tau_sigma_low_s_60deg_result['title']
full_one_layer_refl_tau_sigma_low_s_60deg_x_label = "$\\tau$ (ps)"
full_one_layer_refl_tau_sigma_low_s_60deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_low_s_60deg_summary_handle:
    full_one_layer_refl_tau_sigma_low_s_60deg_summary_meta = json.load(full_one_layer_refl_tau_sigma_low_s_60deg_summary_handle)
with (full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_low_s_60deg_corr_handle:
    full_one_layer_refl_tau_sigma_low_s_60deg_corr_meta = json.load(full_one_layer_refl_tau_sigma_low_s_60deg_corr_handle)

full_one_layer_refl_tau_sigma_low_s_60deg_rows = []
with (full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_refl_tau_sigma_low_s_60deg_summary_csv:
    full_one_layer_refl_tau_sigma_low_s_60deg_reader = csv.DictReader(full_one_layer_refl_tau_sigma_low_s_60deg_summary_csv)
    for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_reader:
        full_one_layer_refl_tau_sigma_low_s_60deg_parsed = {}
        for full_one_layer_refl_tau_sigma_low_s_60deg_key, full_one_layer_refl_tau_sigma_low_s_60deg_value in full_one_layer_refl_tau_sigma_low_s_60deg_row.items():
            if full_one_layer_refl_tau_sigma_low_s_60deg_value is None:
                full_one_layer_refl_tau_sigma_low_s_60deg_parsed[full_one_layer_refl_tau_sigma_low_s_60deg_key] = full_one_layer_refl_tau_sigma_low_s_60deg_value
                continue
            full_one_layer_refl_tau_sigma_low_s_60deg_text = str(full_one_layer_refl_tau_sigma_low_s_60deg_value).strip()
            if full_one_layer_refl_tau_sigma_low_s_60deg_text == '':
                full_one_layer_refl_tau_sigma_low_s_60deg_parsed[full_one_layer_refl_tau_sigma_low_s_60deg_key] = full_one_layer_refl_tau_sigma_low_s_60deg_text
                continue
            try:
                full_one_layer_refl_tau_sigma_low_s_60deg_parsed[full_one_layer_refl_tau_sigma_low_s_60deg_key] = float(full_one_layer_refl_tau_sigma_low_s_60deg_text)
            except ValueError:
                full_one_layer_refl_tau_sigma_low_s_60deg_parsed[full_one_layer_refl_tau_sigma_low_s_60deg_key] = full_one_layer_refl_tau_sigma_low_s_60deg_text
        full_one_layer_refl_tau_sigma_low_s_60deg_rows.append(full_one_layer_refl_tau_sigma_low_s_60deg_parsed)

full_one_layer_refl_tau_sigma_low_s_60deg_x_key, full_one_layer_refl_tau_sigma_low_s_60deg_y_key = list(full_one_layer_refl_tau_sigma_low_s_60deg_summary_meta['recovery_keys'])
full_one_layer_refl_tau_sigma_low_s_60deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_refl_tau_sigma_low_s_60deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_low_s_60deg_x_key}',
    f'{full_one_layer_refl_tau_sigma_low_s_60deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_low_s_60deg_y_key}',
    f'abs_{full_one_layer_refl_tau_sigma_low_s_60deg_x_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_low_s_60deg_x_key}',
    f'abs_{full_one_layer_refl_tau_sigma_low_s_60deg_y_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_low_s_60deg_y_key}',
}
full_one_layer_refl_tau_sigma_low_s_60deg_linear_value_key = full_one_layer_refl_tau_sigma_low_s_60deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_low_s_60deg_log_value_key = full_one_layer_refl_tau_sigma_low_s_60deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_low_s_60deg_x_values = sorted({float(full_one_layer_refl_tau_sigma_low_s_60deg_row[full_one_layer_refl_tau_sigma_low_s_60deg_x_key]) for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_rows})
full_one_layer_refl_tau_sigma_low_s_60deg_y_values = sorted({float(full_one_layer_refl_tau_sigma_low_s_60deg_row[full_one_layer_refl_tau_sigma_low_s_60deg_y_key]) for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_rows})

print('Available map metrics:')
for full_one_layer_refl_tau_sigma_low_s_60deg_option_name, full_one_layer_refl_tau_sigma_low_s_60deg_option_key in full_one_layer_refl_tau_sigma_low_s_60deg_metric_options.items():
    print(f"  {full_one_layer_refl_tau_sigma_low_s_60deg_option_name}: {metric_label(full_one_layer_refl_tau_sigma_low_s_60deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_refl_tau_sigma_low_s_60deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_refl_tau_sigma_low_s_60deg_log_value_key)}")

def full_one_layer_refl_tau_sigma_low_s_60deg_aggregate_grid(value_key):
    full_one_layer_refl_tau_sigma_low_s_60deg_grid = np.full((len(full_one_layer_refl_tau_sigma_low_s_60deg_y_values), len(full_one_layer_refl_tau_sigma_low_s_60deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_refl_tau_sigma_low_s_60deg_iy, full_one_layer_refl_tau_sigma_low_s_60deg_y_value in enumerate(full_one_layer_refl_tau_sigma_low_s_60deg_y_values):
        for full_one_layer_refl_tau_sigma_low_s_60deg_ix, full_one_layer_refl_tau_sigma_low_s_60deg_x_value in enumerate(full_one_layer_refl_tau_sigma_low_s_60deg_x_values):
            full_one_layer_refl_tau_sigma_low_s_60deg_cell = [
                float(full_one_layer_refl_tau_sigma_low_s_60deg_row[value_key])
                for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_rows
                if math.isclose(float(full_one_layer_refl_tau_sigma_low_s_60deg_row[full_one_layer_refl_tau_sigma_low_s_60deg_x_key]), full_one_layer_refl_tau_sigma_low_s_60deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_refl_tau_sigma_low_s_60deg_row[full_one_layer_refl_tau_sigma_low_s_60deg_y_key]), full_one_layer_refl_tau_sigma_low_s_60deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_refl_tau_sigma_low_s_60deg_row[value_key]))
            ]
            if full_one_layer_refl_tau_sigma_low_s_60deg_cell:
                full_one_layer_refl_tau_sigma_low_s_60deg_grid[full_one_layer_refl_tau_sigma_low_s_60deg_iy, full_one_layer_refl_tau_sigma_low_s_60deg_ix] = float(np.mean(full_one_layer_refl_tau_sigma_low_s_60deg_cell))
    return full_one_layer_refl_tau_sigma_low_s_60deg_grid

def full_one_layer_refl_tau_sigma_low_s_60deg_positive_grid_for_log(grid):
    full_one_layer_refl_tau_sigma_low_s_60deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_refl_tau_sigma_low_s_60deg_finite = full_one_layer_refl_tau_sigma_low_s_60deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_60deg_grid)]
    full_one_layer_refl_tau_sigma_low_s_60deg_positive = full_one_layer_refl_tau_sigma_low_s_60deg_finite[full_one_layer_refl_tau_sigma_low_s_60deg_finite > 0.0]
    if full_one_layer_refl_tau_sigma_low_s_60deg_positive.size == 0:
        full_one_layer_refl_tau_sigma_low_s_60deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_60deg_grid)] = 1.0
        return full_one_layer_refl_tau_sigma_low_s_60deg_grid
    full_one_layer_refl_tau_sigma_low_s_60deg_floor = max(float(np.min(full_one_layer_refl_tau_sigma_low_s_60deg_positive)) * 0.5, 1e-18)
    full_one_layer_refl_tau_sigma_low_s_60deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_60deg_grid) & (full_one_layer_refl_tau_sigma_low_s_60deg_grid <= 0.0)] = full_one_layer_refl_tau_sigma_low_s_60deg_floor
    return full_one_layer_refl_tau_sigma_low_s_60deg_grid

full_one_layer_refl_tau_sigma_low_s_60deg_linear_grid = full_one_layer_refl_tau_sigma_low_s_60deg_aggregate_grid(full_one_layer_refl_tau_sigma_low_s_60deg_linear_value_key)
full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig, full_one_layer_refl_tau_sigma_low_s_60deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_low_s_60deg_linear_finite = full_one_layer_refl_tau_sigma_low_s_60deg_linear_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_60deg_linear_grid)]
if str(full_one_layer_refl_tau_sigma_low_s_60deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_limit = max(float(np.max(np.abs(full_one_layer_refl_tau_sigma_low_s_60deg_linear_finite))), 1e-18)
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmin = -full_one_layer_refl_tau_sigma_low_s_60deg_linear_limit
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmax = full_one_layer_refl_tau_sigma_low_s_60deg_linear_limit
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmin, full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_cmap = 'plasma'
else:
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmin = float(np.min(full_one_layer_refl_tau_sigma_low_s_60deg_linear_finite))
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmax = float(np.max(full_one_layer_refl_tau_sigma_low_s_60deg_linear_finite)) + 1e-12
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmin, full_one_layer_refl_tau_sigma_low_s_60deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_cmap = 'plasma'
full_one_layer_refl_tau_sigma_low_s_60deg_linear_contour = full_one_layer_refl_tau_sigma_low_s_60deg_linear_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_low_s_60deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_low_s_60deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_low_s_60deg_linear_grid,
    levels=full_one_layer_refl_tau_sigma_low_s_60deg_linear_levels,
    cmap=full_one_layer_refl_tau_sigma_low_s_60deg_linear_cmap,
    extend='both',
)
full_one_layer_refl_tau_sigma_low_s_60deg_linear_ax.set_title(full_one_layer_refl_tau_sigma_low_s_60deg_plot_title + ' [linear]')
full_one_layer_refl_tau_sigma_low_s_60deg_linear_ax.set_xlabel(full_one_layer_refl_tau_sigma_low_s_60deg_x_label)
full_one_layer_refl_tau_sigma_low_s_60deg_linear_ax.set_ylabel(full_one_layer_refl_tau_sigma_low_s_60deg_y_label)
full_one_layer_refl_tau_sigma_low_s_60deg_linear_cbar = full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig.colorbar(full_one_layer_refl_tau_sigma_low_s_60deg_linear_contour, ax=full_one_layer_refl_tau_sigma_low_s_60deg_linear_ax)
full_one_layer_refl_tau_sigma_low_s_60deg_linear_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_low_s_60deg_linear_value_key))
full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_60deg_linear_png = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_linear.png"
full_one_layer_refl_tau_sigma_low_s_60deg_linear_pdf = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_linear.pdf"
full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_linear_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_linear_pdf)

if str(full_one_layer_refl_tau_sigma_low_s_60deg_log_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_low_s_60deg_log_source_key = 'abs_err_' + str(full_one_layer_refl_tau_sigma_low_s_60deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_refl_tau_sigma_low_s_60deg_log_source_key = full_one_layer_refl_tau_sigma_low_s_60deg_log_value_key
full_one_layer_refl_tau_sigma_low_s_60deg_log_grid = full_one_layer_refl_tau_sigma_low_s_60deg_positive_grid_for_log(full_one_layer_refl_tau_sigma_low_s_60deg_aggregate_grid(full_one_layer_refl_tau_sigma_low_s_60deg_log_source_key))
full_one_layer_refl_tau_sigma_low_s_60deg_log_fig, full_one_layer_refl_tau_sigma_low_s_60deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_low_s_60deg_log_finite = full_one_layer_refl_tau_sigma_low_s_60deg_log_grid[np.isfinite(full_one_layer_refl_tau_sigma_low_s_60deg_log_grid)]
full_one_layer_refl_tau_sigma_low_s_60deg_log_positive = full_one_layer_refl_tau_sigma_low_s_60deg_log_finite[full_one_layer_refl_tau_sigma_low_s_60deg_log_finite > 0.0]
full_one_layer_refl_tau_sigma_low_s_60deg_log_vmin = float(np.min(full_one_layer_refl_tau_sigma_low_s_60deg_log_positive))
full_one_layer_refl_tau_sigma_low_s_60deg_log_vmax = float(np.max(full_one_layer_refl_tau_sigma_low_s_60deg_log_positive))
if math.isclose(full_one_layer_refl_tau_sigma_low_s_60deg_log_vmin, full_one_layer_refl_tau_sigma_low_s_60deg_log_vmax):
    full_one_layer_refl_tau_sigma_low_s_60deg_log_vmax = full_one_layer_refl_tau_sigma_low_s_60deg_log_vmin * 1.01
full_one_layer_refl_tau_sigma_low_s_60deg_log_levels = np.geomspace(full_one_layer_refl_tau_sigma_low_s_60deg_log_vmin, full_one_layer_refl_tau_sigma_low_s_60deg_log_vmax, 256)
full_one_layer_refl_tau_sigma_low_s_60deg_log_contour = full_one_layer_refl_tau_sigma_low_s_60deg_log_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_low_s_60deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_low_s_60deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_low_s_60deg_log_grid,
    levels=full_one_layer_refl_tau_sigma_low_s_60deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_refl_tau_sigma_low_s_60deg_log_vmin, vmax=full_one_layer_refl_tau_sigma_low_s_60deg_log_vmax),
    extend='both',
)
full_one_layer_refl_tau_sigma_low_s_60deg_log_ax.set_title(full_one_layer_refl_tau_sigma_low_s_60deg_plot_title + ' [log]')
full_one_layer_refl_tau_sigma_low_s_60deg_log_ax.set_xlabel(full_one_layer_refl_tau_sigma_low_s_60deg_x_label)
full_one_layer_refl_tau_sigma_low_s_60deg_log_ax.set_ylabel(full_one_layer_refl_tau_sigma_low_s_60deg_y_label)
full_one_layer_refl_tau_sigma_low_s_60deg_log_cbar = full_one_layer_refl_tau_sigma_low_s_60deg_log_fig.colorbar(full_one_layer_refl_tau_sigma_low_s_60deg_log_contour, ax=full_one_layer_refl_tau_sigma_low_s_60deg_log_ax)
full_one_layer_refl_tau_sigma_low_s_60deg_log_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_low_s_60deg_log_source_key))
full_one_layer_refl_tau_sigma_low_s_60deg_log_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_60deg_log_png = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_log.png"
full_one_layer_refl_tau_sigma_low_s_60deg_log_pdf = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_log.pdf"
full_one_layer_refl_tau_sigma_low_s_60deg_log_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_log_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_60deg_log_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_log_pdf)

full_one_layer_refl_tau_sigma_low_s_60deg_corr_rows = full_one_layer_refl_tau_sigma_low_s_60deg_corr_meta['rows']
full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels = sorted({str(full_one_layer_refl_tau_sigma_low_s_60deg_row['param_i']) for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_corr_rows} | {str(full_one_layer_refl_tau_sigma_low_s_60deg_row['param_j']) for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_corr_rows})
full_one_layer_refl_tau_sigma_low_s_60deg_corr_index = {full_one_layer_refl_tau_sigma_low_s_60deg_label: full_one_layer_refl_tau_sigma_low_s_60deg_idx for full_one_layer_refl_tau_sigma_low_s_60deg_idx, full_one_layer_refl_tau_sigma_low_s_60deg_label in enumerate(full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels)}
full_one_layer_refl_tau_sigma_low_s_60deg_corr_matrix = np.full((len(full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels), len(full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_refl_tau_sigma_low_s_60deg_row in full_one_layer_refl_tau_sigma_low_s_60deg_corr_rows:
    full_one_layer_refl_tau_sigma_low_s_60deg_i = full_one_layer_refl_tau_sigma_low_s_60deg_corr_index[str(full_one_layer_refl_tau_sigma_low_s_60deg_row['param_i'])]
    full_one_layer_refl_tau_sigma_low_s_60deg_j = full_one_layer_refl_tau_sigma_low_s_60deg_corr_index[str(full_one_layer_refl_tau_sigma_low_s_60deg_row['param_j'])]
    full_one_layer_refl_tau_sigma_low_s_60deg_corr_matrix[full_one_layer_refl_tau_sigma_low_s_60deg_i, full_one_layer_refl_tau_sigma_low_s_60deg_j] = float(full_one_layer_refl_tau_sigma_low_s_60deg_row['correlation'])
    full_one_layer_refl_tau_sigma_low_s_60deg_corr_matrix[full_one_layer_refl_tau_sigma_low_s_60deg_j, full_one_layer_refl_tau_sigma_low_s_60deg_i] = float(full_one_layer_refl_tau_sigma_low_s_60deg_row['correlation'])
for full_one_layer_refl_tau_sigma_low_s_60deg_diag in range(len(full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels)):
    full_one_layer_refl_tau_sigma_low_s_60deg_corr_matrix[full_one_layer_refl_tau_sigma_low_s_60deg_diag, full_one_layer_refl_tau_sigma_low_s_60deg_diag] = 1.0

full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig, full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_refl_tau_sigma_low_s_60deg_corr_image = full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax.imshow(full_one_layer_refl_tau_sigma_low_s_60deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax.set_title(full_one_layer_refl_tau_sigma_low_s_60deg_plot_title + ' [correlation]')
full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax.set_xticks(range(len(full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels)))
full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax.set_yticks(range(len(full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels)))
full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_low_s_60deg_label, full_one_layer_refl_tau_sigma_low_s_60deg_label) for full_one_layer_refl_tau_sigma_low_s_60deg_label in full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels], rotation=45, ha='right')
full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_low_s_60deg_label, full_one_layer_refl_tau_sigma_low_s_60deg_label) for full_one_layer_refl_tau_sigma_low_s_60deg_label in full_one_layer_refl_tau_sigma_low_s_60deg_corr_labels])
full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig.colorbar(full_one_layer_refl_tau_sigma_low_s_60deg_corr_image, ax=full_one_layer_refl_tau_sigma_low_s_60deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_60deg_corr_png = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_corr.png"
full_one_layer_refl_tau_sigma_low_s_60deg_corr_pdf = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_corr.pdf"
full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_corr_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_corr_pdf)

full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig, full_one_layer_refl_tau_sigma_low_s_60deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_refl_tau_sigma_low_s_60deg_axis, full_one_layer_refl_tau_sigma_low_s_60deg_image_path, full_one_layer_refl_tau_sigma_low_s_60deg_panel_title in zip(
    full_one_layer_refl_tau_sigma_low_s_60deg_triptych_axes,
    (full_one_layer_refl_tau_sigma_low_s_60deg_linear_png, full_one_layer_refl_tau_sigma_low_s_60deg_log_png, full_one_layer_refl_tau_sigma_low_s_60deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_refl_tau_sigma_low_s_60deg_image = plt.imread(str(full_one_layer_refl_tau_sigma_low_s_60deg_image_path))
    full_one_layer_refl_tau_sigma_low_s_60deg_axis.imshow(full_one_layer_refl_tau_sigma_low_s_60deg_image)
    full_one_layer_refl_tau_sigma_low_s_60deg_axis.set_title(full_one_layer_refl_tau_sigma_low_s_60deg_panel_title)
    full_one_layer_refl_tau_sigma_low_s_60deg_axis.axis('off')
full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig.suptitle(full_one_layer_refl_tau_sigma_low_s_60deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig.tight_layout()
full_one_layer_refl_tau_sigma_low_s_60deg_triptych_png = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_triptych.png"
full_one_layer_refl_tau_sigma_low_s_60deg_triptych_pdf = full_one_layer_refl_tau_sigma_low_s_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_low_s_60deg_study_dir.name}__section_triptych.pdf"
full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_triptych_png, dpi=220)
full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_low_s_60deg_triptych_pdf)

display(full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig)
display(full_one_layer_refl_tau_sigma_low_s_60deg_log_fig)
display(full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig)
display(full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_60deg_linear_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_60deg_log_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_60deg_corr_fig)
plt.close(full_one_layer_refl_tau_sigma_low_s_60deg_triptych_fig)

full_one_layer_refl_tau_sigma_low_s_60deg_plot_paths = {
    'linear_png': full_one_layer_refl_tau_sigma_low_s_60deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_refl_tau_sigma_low_s_60deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_refl_tau_sigma_low_s_60deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_refl_tau_sigma_low_s_60deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_refl_tau_sigma_low_s_60deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_refl_tau_sigma_low_s_60deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_refl_tau_sigma_low_s_60deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_refl_tau_sigma_low_s_60deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_refl_tau_sigma_low_s_60deg_plot_paths

### One-layer Drude reflection: $\tau$ vs $\sigma$ (45 deg, p-pol, $d=725\,\mu$m)

In [ ]:
full_one_layer_refl_tau_sigma_high_p_45deg_spec = {
  "slug": "one_layer_refl_tau_sigma_high_p_45deg",
  "title": "One-layer Drude reflection: $\\tau$ vs $\\sigma$ (45 deg, p-pol, $d=725\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 45.0,
  "polarization": "p",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    100.0,
    1000.0
  ],
  "thickness_um": 725.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ]
}
full_one_layer_refl_tau_sigma_high_p_45deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_refl_tau_sigma_high_p_45deg'
full_one_layer_refl_tau_sigma_high_p_45deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_refl_tau_sigma_high_p_45deg_result = run_lecture_map_from_spec(full_one_layer_refl_tau_sigma_high_p_45deg_spec, output_root=full_one_layer_refl_tau_sigma_high_p_45deg_output_root, profile='full')
full_one_layer_refl_tau_sigma_high_p_45deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_refl_tau_sigma_high_p_45deg_study_dir = Path(full_one_layer_refl_tau_sigma_high_p_45deg_result['study_dir'])
full_one_layer_refl_tau_sigma_high_p_45deg_plot_title = full_one_layer_refl_tau_sigma_high_p_45deg_result['title']
full_one_layer_refl_tau_sigma_high_p_45deg_x_label = "$\\tau$ (ps)"
full_one_layer_refl_tau_sigma_high_p_45deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_high_p_45deg_summary_handle:
    full_one_layer_refl_tau_sigma_high_p_45deg_summary_meta = json.load(full_one_layer_refl_tau_sigma_high_p_45deg_summary_handle)
with (full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_high_p_45deg_corr_handle:
    full_one_layer_refl_tau_sigma_high_p_45deg_corr_meta = json.load(full_one_layer_refl_tau_sigma_high_p_45deg_corr_handle)

full_one_layer_refl_tau_sigma_high_p_45deg_rows = []
with (full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_refl_tau_sigma_high_p_45deg_summary_csv:
    full_one_layer_refl_tau_sigma_high_p_45deg_reader = csv.DictReader(full_one_layer_refl_tau_sigma_high_p_45deg_summary_csv)
    for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_reader:
        full_one_layer_refl_tau_sigma_high_p_45deg_parsed = {}
        for full_one_layer_refl_tau_sigma_high_p_45deg_key, full_one_layer_refl_tau_sigma_high_p_45deg_value in full_one_layer_refl_tau_sigma_high_p_45deg_row.items():
            if full_one_layer_refl_tau_sigma_high_p_45deg_value is None:
                full_one_layer_refl_tau_sigma_high_p_45deg_parsed[full_one_layer_refl_tau_sigma_high_p_45deg_key] = full_one_layer_refl_tau_sigma_high_p_45deg_value
                continue
            full_one_layer_refl_tau_sigma_high_p_45deg_text = str(full_one_layer_refl_tau_sigma_high_p_45deg_value).strip()
            if full_one_layer_refl_tau_sigma_high_p_45deg_text == '':
                full_one_layer_refl_tau_sigma_high_p_45deg_parsed[full_one_layer_refl_tau_sigma_high_p_45deg_key] = full_one_layer_refl_tau_sigma_high_p_45deg_text
                continue
            try:
                full_one_layer_refl_tau_sigma_high_p_45deg_parsed[full_one_layer_refl_tau_sigma_high_p_45deg_key] = float(full_one_layer_refl_tau_sigma_high_p_45deg_text)
            except ValueError:
                full_one_layer_refl_tau_sigma_high_p_45deg_parsed[full_one_layer_refl_tau_sigma_high_p_45deg_key] = full_one_layer_refl_tau_sigma_high_p_45deg_text
        full_one_layer_refl_tau_sigma_high_p_45deg_rows.append(full_one_layer_refl_tau_sigma_high_p_45deg_parsed)

full_one_layer_refl_tau_sigma_high_p_45deg_x_key, full_one_layer_refl_tau_sigma_high_p_45deg_y_key = list(full_one_layer_refl_tau_sigma_high_p_45deg_summary_meta['recovery_keys'])
full_one_layer_refl_tau_sigma_high_p_45deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_refl_tau_sigma_high_p_45deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_high_p_45deg_x_key}',
    f'{full_one_layer_refl_tau_sigma_high_p_45deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_high_p_45deg_y_key}',
    f'abs_{full_one_layer_refl_tau_sigma_high_p_45deg_x_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_high_p_45deg_x_key}',
    f'abs_{full_one_layer_refl_tau_sigma_high_p_45deg_y_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_high_p_45deg_y_key}',
}
full_one_layer_refl_tau_sigma_high_p_45deg_linear_value_key = full_one_layer_refl_tau_sigma_high_p_45deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_high_p_45deg_log_value_key = full_one_layer_refl_tau_sigma_high_p_45deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_high_p_45deg_x_values = sorted({float(full_one_layer_refl_tau_sigma_high_p_45deg_row[full_one_layer_refl_tau_sigma_high_p_45deg_x_key]) for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_rows})
full_one_layer_refl_tau_sigma_high_p_45deg_y_values = sorted({float(full_one_layer_refl_tau_sigma_high_p_45deg_row[full_one_layer_refl_tau_sigma_high_p_45deg_y_key]) for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_rows})

print('Available map metrics:')
for full_one_layer_refl_tau_sigma_high_p_45deg_option_name, full_one_layer_refl_tau_sigma_high_p_45deg_option_key in full_one_layer_refl_tau_sigma_high_p_45deg_metric_options.items():
    print(f"  {full_one_layer_refl_tau_sigma_high_p_45deg_option_name}: {metric_label(full_one_layer_refl_tau_sigma_high_p_45deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_refl_tau_sigma_high_p_45deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_refl_tau_sigma_high_p_45deg_log_value_key)}")

def full_one_layer_refl_tau_sigma_high_p_45deg_aggregate_grid(value_key):
    full_one_layer_refl_tau_sigma_high_p_45deg_grid = np.full((len(full_one_layer_refl_tau_sigma_high_p_45deg_y_values), len(full_one_layer_refl_tau_sigma_high_p_45deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_refl_tau_sigma_high_p_45deg_iy, full_one_layer_refl_tau_sigma_high_p_45deg_y_value in enumerate(full_one_layer_refl_tau_sigma_high_p_45deg_y_values):
        for full_one_layer_refl_tau_sigma_high_p_45deg_ix, full_one_layer_refl_tau_sigma_high_p_45deg_x_value in enumerate(full_one_layer_refl_tau_sigma_high_p_45deg_x_values):
            full_one_layer_refl_tau_sigma_high_p_45deg_cell = [
                float(full_one_layer_refl_tau_sigma_high_p_45deg_row[value_key])
                for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_rows
                if math.isclose(float(full_one_layer_refl_tau_sigma_high_p_45deg_row[full_one_layer_refl_tau_sigma_high_p_45deg_x_key]), full_one_layer_refl_tau_sigma_high_p_45deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_refl_tau_sigma_high_p_45deg_row[full_one_layer_refl_tau_sigma_high_p_45deg_y_key]), full_one_layer_refl_tau_sigma_high_p_45deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_refl_tau_sigma_high_p_45deg_row[value_key]))
            ]
            if full_one_layer_refl_tau_sigma_high_p_45deg_cell:
                full_one_layer_refl_tau_sigma_high_p_45deg_grid[full_one_layer_refl_tau_sigma_high_p_45deg_iy, full_one_layer_refl_tau_sigma_high_p_45deg_ix] = float(np.mean(full_one_layer_refl_tau_sigma_high_p_45deg_cell))
    return full_one_layer_refl_tau_sigma_high_p_45deg_grid

def full_one_layer_refl_tau_sigma_high_p_45deg_positive_grid_for_log(grid):
    full_one_layer_refl_tau_sigma_high_p_45deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_refl_tau_sigma_high_p_45deg_finite = full_one_layer_refl_tau_sigma_high_p_45deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_45deg_grid)]
    full_one_layer_refl_tau_sigma_high_p_45deg_positive = full_one_layer_refl_tau_sigma_high_p_45deg_finite[full_one_layer_refl_tau_sigma_high_p_45deg_finite > 0.0]
    if full_one_layer_refl_tau_sigma_high_p_45deg_positive.size == 0:
        full_one_layer_refl_tau_sigma_high_p_45deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_45deg_grid)] = 1.0
        return full_one_layer_refl_tau_sigma_high_p_45deg_grid
    full_one_layer_refl_tau_sigma_high_p_45deg_floor = max(float(np.min(full_one_layer_refl_tau_sigma_high_p_45deg_positive)) * 0.5, 1e-18)
    full_one_layer_refl_tau_sigma_high_p_45deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_45deg_grid) & (full_one_layer_refl_tau_sigma_high_p_45deg_grid <= 0.0)] = full_one_layer_refl_tau_sigma_high_p_45deg_floor
    return full_one_layer_refl_tau_sigma_high_p_45deg_grid

full_one_layer_refl_tau_sigma_high_p_45deg_linear_grid = full_one_layer_refl_tau_sigma_high_p_45deg_aggregate_grid(full_one_layer_refl_tau_sigma_high_p_45deg_linear_value_key)
full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig, full_one_layer_refl_tau_sigma_high_p_45deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_high_p_45deg_linear_finite = full_one_layer_refl_tau_sigma_high_p_45deg_linear_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_45deg_linear_grid)]
if str(full_one_layer_refl_tau_sigma_high_p_45deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_limit = max(float(np.max(np.abs(full_one_layer_refl_tau_sigma_high_p_45deg_linear_finite))), 1e-18)
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmin = -full_one_layer_refl_tau_sigma_high_p_45deg_linear_limit
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmax = full_one_layer_refl_tau_sigma_high_p_45deg_linear_limit
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmin, full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_cmap = 'plasma'
else:
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmin = float(np.min(full_one_layer_refl_tau_sigma_high_p_45deg_linear_finite))
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmax = float(np.max(full_one_layer_refl_tau_sigma_high_p_45deg_linear_finite)) + 1e-12
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmin, full_one_layer_refl_tau_sigma_high_p_45deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_cmap = 'plasma'
full_one_layer_refl_tau_sigma_high_p_45deg_linear_contour = full_one_layer_refl_tau_sigma_high_p_45deg_linear_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_high_p_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_high_p_45deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_high_p_45deg_linear_grid,
    levels=full_one_layer_refl_tau_sigma_high_p_45deg_linear_levels,
    cmap=full_one_layer_refl_tau_sigma_high_p_45deg_linear_cmap,
    extend='both',
)
full_one_layer_refl_tau_sigma_high_p_45deg_linear_ax.set_title(full_one_layer_refl_tau_sigma_high_p_45deg_plot_title + ' [linear]')
full_one_layer_refl_tau_sigma_high_p_45deg_linear_ax.set_xlabel(full_one_layer_refl_tau_sigma_high_p_45deg_x_label)
full_one_layer_refl_tau_sigma_high_p_45deg_linear_ax.set_ylabel(full_one_layer_refl_tau_sigma_high_p_45deg_y_label)
full_one_layer_refl_tau_sigma_high_p_45deg_linear_cbar = full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig.colorbar(full_one_layer_refl_tau_sigma_high_p_45deg_linear_contour, ax=full_one_layer_refl_tau_sigma_high_p_45deg_linear_ax)
full_one_layer_refl_tau_sigma_high_p_45deg_linear_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_high_p_45deg_linear_value_key))
full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_45deg_linear_png = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_linear.png"
full_one_layer_refl_tau_sigma_high_p_45deg_linear_pdf = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_linear.pdf"
full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_linear_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_linear_pdf)

if str(full_one_layer_refl_tau_sigma_high_p_45deg_log_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_high_p_45deg_log_source_key = 'abs_err_' + str(full_one_layer_refl_tau_sigma_high_p_45deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_refl_tau_sigma_high_p_45deg_log_source_key = full_one_layer_refl_tau_sigma_high_p_45deg_log_value_key
full_one_layer_refl_tau_sigma_high_p_45deg_log_grid = full_one_layer_refl_tau_sigma_high_p_45deg_positive_grid_for_log(full_one_layer_refl_tau_sigma_high_p_45deg_aggregate_grid(full_one_layer_refl_tau_sigma_high_p_45deg_log_source_key))
full_one_layer_refl_tau_sigma_high_p_45deg_log_fig, full_one_layer_refl_tau_sigma_high_p_45deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_high_p_45deg_log_finite = full_one_layer_refl_tau_sigma_high_p_45deg_log_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_45deg_log_grid)]
full_one_layer_refl_tau_sigma_high_p_45deg_log_positive = full_one_layer_refl_tau_sigma_high_p_45deg_log_finite[full_one_layer_refl_tau_sigma_high_p_45deg_log_finite > 0.0]
full_one_layer_refl_tau_sigma_high_p_45deg_log_vmin = float(np.min(full_one_layer_refl_tau_sigma_high_p_45deg_log_positive))
full_one_layer_refl_tau_sigma_high_p_45deg_log_vmax = float(np.max(full_one_layer_refl_tau_sigma_high_p_45deg_log_positive))
if math.isclose(full_one_layer_refl_tau_sigma_high_p_45deg_log_vmin, full_one_layer_refl_tau_sigma_high_p_45deg_log_vmax):
    full_one_layer_refl_tau_sigma_high_p_45deg_log_vmax = full_one_layer_refl_tau_sigma_high_p_45deg_log_vmin * 1.01
full_one_layer_refl_tau_sigma_high_p_45deg_log_levels = np.geomspace(full_one_layer_refl_tau_sigma_high_p_45deg_log_vmin, full_one_layer_refl_tau_sigma_high_p_45deg_log_vmax, 256)
full_one_layer_refl_tau_sigma_high_p_45deg_log_contour = full_one_layer_refl_tau_sigma_high_p_45deg_log_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_high_p_45deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_high_p_45deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_high_p_45deg_log_grid,
    levels=full_one_layer_refl_tau_sigma_high_p_45deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_refl_tau_sigma_high_p_45deg_log_vmin, vmax=full_one_layer_refl_tau_sigma_high_p_45deg_log_vmax),
    extend='both',
)
full_one_layer_refl_tau_sigma_high_p_45deg_log_ax.set_title(full_one_layer_refl_tau_sigma_high_p_45deg_plot_title + ' [log]')
full_one_layer_refl_tau_sigma_high_p_45deg_log_ax.set_xlabel(full_one_layer_refl_tau_sigma_high_p_45deg_x_label)
full_one_layer_refl_tau_sigma_high_p_45deg_log_ax.set_ylabel(full_one_layer_refl_tau_sigma_high_p_45deg_y_label)
full_one_layer_refl_tau_sigma_high_p_45deg_log_cbar = full_one_layer_refl_tau_sigma_high_p_45deg_log_fig.colorbar(full_one_layer_refl_tau_sigma_high_p_45deg_log_contour, ax=full_one_layer_refl_tau_sigma_high_p_45deg_log_ax)
full_one_layer_refl_tau_sigma_high_p_45deg_log_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_high_p_45deg_log_source_key))
full_one_layer_refl_tau_sigma_high_p_45deg_log_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_45deg_log_png = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_log.png"
full_one_layer_refl_tau_sigma_high_p_45deg_log_pdf = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_log.pdf"
full_one_layer_refl_tau_sigma_high_p_45deg_log_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_log_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_45deg_log_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_log_pdf)

full_one_layer_refl_tau_sigma_high_p_45deg_corr_rows = full_one_layer_refl_tau_sigma_high_p_45deg_corr_meta['rows']
full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels = sorted({str(full_one_layer_refl_tau_sigma_high_p_45deg_row['param_i']) for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_corr_rows} | {str(full_one_layer_refl_tau_sigma_high_p_45deg_row['param_j']) for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_corr_rows})
full_one_layer_refl_tau_sigma_high_p_45deg_corr_index = {full_one_layer_refl_tau_sigma_high_p_45deg_label: full_one_layer_refl_tau_sigma_high_p_45deg_idx for full_one_layer_refl_tau_sigma_high_p_45deg_idx, full_one_layer_refl_tau_sigma_high_p_45deg_label in enumerate(full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels)}
full_one_layer_refl_tau_sigma_high_p_45deg_corr_matrix = np.full((len(full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels), len(full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_refl_tau_sigma_high_p_45deg_row in full_one_layer_refl_tau_sigma_high_p_45deg_corr_rows:
    full_one_layer_refl_tau_sigma_high_p_45deg_i = full_one_layer_refl_tau_sigma_high_p_45deg_corr_index[str(full_one_layer_refl_tau_sigma_high_p_45deg_row['param_i'])]
    full_one_layer_refl_tau_sigma_high_p_45deg_j = full_one_layer_refl_tau_sigma_high_p_45deg_corr_index[str(full_one_layer_refl_tau_sigma_high_p_45deg_row['param_j'])]
    full_one_layer_refl_tau_sigma_high_p_45deg_corr_matrix[full_one_layer_refl_tau_sigma_high_p_45deg_i, full_one_layer_refl_tau_sigma_high_p_45deg_j] = float(full_one_layer_refl_tau_sigma_high_p_45deg_row['correlation'])
    full_one_layer_refl_tau_sigma_high_p_45deg_corr_matrix[full_one_layer_refl_tau_sigma_high_p_45deg_j, full_one_layer_refl_tau_sigma_high_p_45deg_i] = float(full_one_layer_refl_tau_sigma_high_p_45deg_row['correlation'])
for full_one_layer_refl_tau_sigma_high_p_45deg_diag in range(len(full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels)):
    full_one_layer_refl_tau_sigma_high_p_45deg_corr_matrix[full_one_layer_refl_tau_sigma_high_p_45deg_diag, full_one_layer_refl_tau_sigma_high_p_45deg_diag] = 1.0

full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig, full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_refl_tau_sigma_high_p_45deg_corr_image = full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax.imshow(full_one_layer_refl_tau_sigma_high_p_45deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax.set_title(full_one_layer_refl_tau_sigma_high_p_45deg_plot_title + ' [correlation]')
full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax.set_xticks(range(len(full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels)))
full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax.set_yticks(range(len(full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels)))
full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_high_p_45deg_label, full_one_layer_refl_tau_sigma_high_p_45deg_label) for full_one_layer_refl_tau_sigma_high_p_45deg_label in full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels], rotation=45, ha='right')
full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_high_p_45deg_label, full_one_layer_refl_tau_sigma_high_p_45deg_label) for full_one_layer_refl_tau_sigma_high_p_45deg_label in full_one_layer_refl_tau_sigma_high_p_45deg_corr_labels])
full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig.colorbar(full_one_layer_refl_tau_sigma_high_p_45deg_corr_image, ax=full_one_layer_refl_tau_sigma_high_p_45deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_45deg_corr_png = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_corr.png"
full_one_layer_refl_tau_sigma_high_p_45deg_corr_pdf = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_corr.pdf"
full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_corr_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_corr_pdf)

full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig, full_one_layer_refl_tau_sigma_high_p_45deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_refl_tau_sigma_high_p_45deg_axis, full_one_layer_refl_tau_sigma_high_p_45deg_image_path, full_one_layer_refl_tau_sigma_high_p_45deg_panel_title in zip(
    full_one_layer_refl_tau_sigma_high_p_45deg_triptych_axes,
    (full_one_layer_refl_tau_sigma_high_p_45deg_linear_png, full_one_layer_refl_tau_sigma_high_p_45deg_log_png, full_one_layer_refl_tau_sigma_high_p_45deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_refl_tau_sigma_high_p_45deg_image = plt.imread(str(full_one_layer_refl_tau_sigma_high_p_45deg_image_path))
    full_one_layer_refl_tau_sigma_high_p_45deg_axis.imshow(full_one_layer_refl_tau_sigma_high_p_45deg_image)
    full_one_layer_refl_tau_sigma_high_p_45deg_axis.set_title(full_one_layer_refl_tau_sigma_high_p_45deg_panel_title)
    full_one_layer_refl_tau_sigma_high_p_45deg_axis.axis('off')
full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig.suptitle(full_one_layer_refl_tau_sigma_high_p_45deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_45deg_triptych_png = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_triptych.png"
full_one_layer_refl_tau_sigma_high_p_45deg_triptych_pdf = full_one_layer_refl_tau_sigma_high_p_45deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_45deg_study_dir.name}__section_triptych.pdf"
full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_triptych_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_high_p_45deg_triptych_pdf)

display(full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig)
display(full_one_layer_refl_tau_sigma_high_p_45deg_log_fig)
display(full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig)
display(full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_45deg_linear_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_45deg_log_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_45deg_corr_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_45deg_triptych_fig)

full_one_layer_refl_tau_sigma_high_p_45deg_plot_paths = {
    'linear_png': full_one_layer_refl_tau_sigma_high_p_45deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_refl_tau_sigma_high_p_45deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_refl_tau_sigma_high_p_45deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_refl_tau_sigma_high_p_45deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_refl_tau_sigma_high_p_45deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_refl_tau_sigma_high_p_45deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_refl_tau_sigma_high_p_45deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_refl_tau_sigma_high_p_45deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_refl_tau_sigma_high_p_45deg_plot_paths

### One-layer Drude reflection: $\tau$ vs $\sigma$ (60 deg, p-pol, $d=725\,\mu$m)

In [ ]:
full_one_layer_refl_tau_sigma_high_p_60deg_spec = {
  "slug": "one_layer_refl_tau_sigma_high_p_60deg",
  "title": "One-layer Drude reflection: $\\tau$ vs $\\sigma$ (60 deg, p-pol, $d=725\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 60.0,
  "polarization": "p",
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_range": [
    100.0,
    1000.0
  ],
  "thickness_um": 725.0,
  "tau_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "sigma_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ]
}
full_one_layer_refl_tau_sigma_high_p_60deg_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_one_layer_refl_tau_sigma_high_p_60deg'
full_one_layer_refl_tau_sigma_high_p_60deg_output_root.mkdir(parents=True, exist_ok=True)
full_one_layer_refl_tau_sigma_high_p_60deg_result = run_lecture_map_from_spec(full_one_layer_refl_tau_sigma_high_p_60deg_spec, output_root=full_one_layer_refl_tau_sigma_high_p_60deg_output_root, profile='full')
full_one_layer_refl_tau_sigma_high_p_60deg_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other one-layer study plots.

In [ ]:
full_one_layer_refl_tau_sigma_high_p_60deg_study_dir = Path(full_one_layer_refl_tau_sigma_high_p_60deg_result['study_dir'])
full_one_layer_refl_tau_sigma_high_p_60deg_plot_title = full_one_layer_refl_tau_sigma_high_p_60deg_result['title']
full_one_layer_refl_tau_sigma_high_p_60deg_x_label = "$\\tau$ (ps)"
full_one_layer_refl_tau_sigma_high_p_60deg_y_label = "$\\sigma$ (S/m)"

with (full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_high_p_60deg_summary_handle:
    full_one_layer_refl_tau_sigma_high_p_60deg_summary_meta = json.load(full_one_layer_refl_tau_sigma_high_p_60deg_summary_handle)
with (full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_one_layer_refl_tau_sigma_high_p_60deg_corr_handle:
    full_one_layer_refl_tau_sigma_high_p_60deg_corr_meta = json.load(full_one_layer_refl_tau_sigma_high_p_60deg_corr_handle)

full_one_layer_refl_tau_sigma_high_p_60deg_rows = []
with (full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_one_layer_refl_tau_sigma_high_p_60deg_summary_csv:
    full_one_layer_refl_tau_sigma_high_p_60deg_reader = csv.DictReader(full_one_layer_refl_tau_sigma_high_p_60deg_summary_csv)
    for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_reader:
        full_one_layer_refl_tau_sigma_high_p_60deg_parsed = {}
        for full_one_layer_refl_tau_sigma_high_p_60deg_key, full_one_layer_refl_tau_sigma_high_p_60deg_value in full_one_layer_refl_tau_sigma_high_p_60deg_row.items():
            if full_one_layer_refl_tau_sigma_high_p_60deg_value is None:
                full_one_layer_refl_tau_sigma_high_p_60deg_parsed[full_one_layer_refl_tau_sigma_high_p_60deg_key] = full_one_layer_refl_tau_sigma_high_p_60deg_value
                continue
            full_one_layer_refl_tau_sigma_high_p_60deg_text = str(full_one_layer_refl_tau_sigma_high_p_60deg_value).strip()
            if full_one_layer_refl_tau_sigma_high_p_60deg_text == '':
                full_one_layer_refl_tau_sigma_high_p_60deg_parsed[full_one_layer_refl_tau_sigma_high_p_60deg_key] = full_one_layer_refl_tau_sigma_high_p_60deg_text
                continue
            try:
                full_one_layer_refl_tau_sigma_high_p_60deg_parsed[full_one_layer_refl_tau_sigma_high_p_60deg_key] = float(full_one_layer_refl_tau_sigma_high_p_60deg_text)
            except ValueError:
                full_one_layer_refl_tau_sigma_high_p_60deg_parsed[full_one_layer_refl_tau_sigma_high_p_60deg_key] = full_one_layer_refl_tau_sigma_high_p_60deg_text
        full_one_layer_refl_tau_sigma_high_p_60deg_rows.append(full_one_layer_refl_tau_sigma_high_p_60deg_parsed)

full_one_layer_refl_tau_sigma_high_p_60deg_x_key, full_one_layer_refl_tau_sigma_high_p_60deg_y_key = list(full_one_layer_refl_tau_sigma_high_p_60deg_summary_meta['recovery_keys'])
full_one_layer_refl_tau_sigma_high_p_60deg_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_one_layer_refl_tau_sigma_high_p_60deg_x_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_high_p_60deg_x_key}',
    f'{full_one_layer_refl_tau_sigma_high_p_60deg_y_key}_true_minus_fit': f'signed_err_{full_one_layer_refl_tau_sigma_high_p_60deg_y_key}',
    f'abs_{full_one_layer_refl_tau_sigma_high_p_60deg_x_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_high_p_60deg_x_key}',
    f'abs_{full_one_layer_refl_tau_sigma_high_p_60deg_y_key}_error': f'abs_err_{full_one_layer_refl_tau_sigma_high_p_60deg_y_key}',
}
full_one_layer_refl_tau_sigma_high_p_60deg_linear_value_key = full_one_layer_refl_tau_sigma_high_p_60deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_high_p_60deg_log_value_key = full_one_layer_refl_tau_sigma_high_p_60deg_metric_options['data_fit']
full_one_layer_refl_tau_sigma_high_p_60deg_x_values = sorted({float(full_one_layer_refl_tau_sigma_high_p_60deg_row[full_one_layer_refl_tau_sigma_high_p_60deg_x_key]) for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_rows})
full_one_layer_refl_tau_sigma_high_p_60deg_y_values = sorted({float(full_one_layer_refl_tau_sigma_high_p_60deg_row[full_one_layer_refl_tau_sigma_high_p_60deg_y_key]) for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_rows})

print('Available map metrics:')
for full_one_layer_refl_tau_sigma_high_p_60deg_option_name, full_one_layer_refl_tau_sigma_high_p_60deg_option_key in full_one_layer_refl_tau_sigma_high_p_60deg_metric_options.items():
    print(f"  {full_one_layer_refl_tau_sigma_high_p_60deg_option_name}: {metric_label(full_one_layer_refl_tau_sigma_high_p_60deg_option_key)}")
print(f"Current linear map: {metric_label(full_one_layer_refl_tau_sigma_high_p_60deg_linear_value_key)}")
print(f"Current log map: {metric_label(full_one_layer_refl_tau_sigma_high_p_60deg_log_value_key)}")

def full_one_layer_refl_tau_sigma_high_p_60deg_aggregate_grid(value_key):
    full_one_layer_refl_tau_sigma_high_p_60deg_grid = np.full((len(full_one_layer_refl_tau_sigma_high_p_60deg_y_values), len(full_one_layer_refl_tau_sigma_high_p_60deg_x_values)), np.nan, dtype=np.float64)
    for full_one_layer_refl_tau_sigma_high_p_60deg_iy, full_one_layer_refl_tau_sigma_high_p_60deg_y_value in enumerate(full_one_layer_refl_tau_sigma_high_p_60deg_y_values):
        for full_one_layer_refl_tau_sigma_high_p_60deg_ix, full_one_layer_refl_tau_sigma_high_p_60deg_x_value in enumerate(full_one_layer_refl_tau_sigma_high_p_60deg_x_values):
            full_one_layer_refl_tau_sigma_high_p_60deg_cell = [
                float(full_one_layer_refl_tau_sigma_high_p_60deg_row[value_key])
                for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_rows
                if math.isclose(float(full_one_layer_refl_tau_sigma_high_p_60deg_row[full_one_layer_refl_tau_sigma_high_p_60deg_x_key]), full_one_layer_refl_tau_sigma_high_p_60deg_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_one_layer_refl_tau_sigma_high_p_60deg_row[full_one_layer_refl_tau_sigma_high_p_60deg_y_key]), full_one_layer_refl_tau_sigma_high_p_60deg_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_one_layer_refl_tau_sigma_high_p_60deg_row[value_key]))
            ]
            if full_one_layer_refl_tau_sigma_high_p_60deg_cell:
                full_one_layer_refl_tau_sigma_high_p_60deg_grid[full_one_layer_refl_tau_sigma_high_p_60deg_iy, full_one_layer_refl_tau_sigma_high_p_60deg_ix] = float(np.mean(full_one_layer_refl_tau_sigma_high_p_60deg_cell))
    return full_one_layer_refl_tau_sigma_high_p_60deg_grid

def full_one_layer_refl_tau_sigma_high_p_60deg_positive_grid_for_log(grid):
    full_one_layer_refl_tau_sigma_high_p_60deg_grid = np.asarray(grid, dtype=np.float64).copy()
    full_one_layer_refl_tau_sigma_high_p_60deg_finite = full_one_layer_refl_tau_sigma_high_p_60deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_60deg_grid)]
    full_one_layer_refl_tau_sigma_high_p_60deg_positive = full_one_layer_refl_tau_sigma_high_p_60deg_finite[full_one_layer_refl_tau_sigma_high_p_60deg_finite > 0.0]
    if full_one_layer_refl_tau_sigma_high_p_60deg_positive.size == 0:
        full_one_layer_refl_tau_sigma_high_p_60deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_60deg_grid)] = 1.0
        return full_one_layer_refl_tau_sigma_high_p_60deg_grid
    full_one_layer_refl_tau_sigma_high_p_60deg_floor = max(float(np.min(full_one_layer_refl_tau_sigma_high_p_60deg_positive)) * 0.5, 1e-18)
    full_one_layer_refl_tau_sigma_high_p_60deg_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_60deg_grid) & (full_one_layer_refl_tau_sigma_high_p_60deg_grid <= 0.0)] = full_one_layer_refl_tau_sigma_high_p_60deg_floor
    return full_one_layer_refl_tau_sigma_high_p_60deg_grid

full_one_layer_refl_tau_sigma_high_p_60deg_linear_grid = full_one_layer_refl_tau_sigma_high_p_60deg_aggregate_grid(full_one_layer_refl_tau_sigma_high_p_60deg_linear_value_key)
full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig, full_one_layer_refl_tau_sigma_high_p_60deg_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_high_p_60deg_linear_finite = full_one_layer_refl_tau_sigma_high_p_60deg_linear_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_60deg_linear_grid)]
if str(full_one_layer_refl_tau_sigma_high_p_60deg_linear_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_limit = max(float(np.max(np.abs(full_one_layer_refl_tau_sigma_high_p_60deg_linear_finite))), 1e-18)
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmin = -full_one_layer_refl_tau_sigma_high_p_60deg_linear_limit
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmax = full_one_layer_refl_tau_sigma_high_p_60deg_linear_limit
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmin, full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_cmap = 'plasma'
else:
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmin = float(np.min(full_one_layer_refl_tau_sigma_high_p_60deg_linear_finite))
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmax = float(np.max(full_one_layer_refl_tau_sigma_high_p_60deg_linear_finite)) + 1e-12
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_levels = np.linspace(full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmin, full_one_layer_refl_tau_sigma_high_p_60deg_linear_vmax, 256)
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_cmap = 'plasma'
full_one_layer_refl_tau_sigma_high_p_60deg_linear_contour = full_one_layer_refl_tau_sigma_high_p_60deg_linear_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_high_p_60deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_high_p_60deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_high_p_60deg_linear_grid,
    levels=full_one_layer_refl_tau_sigma_high_p_60deg_linear_levels,
    cmap=full_one_layer_refl_tau_sigma_high_p_60deg_linear_cmap,
    extend='both',
)
full_one_layer_refl_tau_sigma_high_p_60deg_linear_ax.set_title(full_one_layer_refl_tau_sigma_high_p_60deg_plot_title + ' [linear]')
full_one_layer_refl_tau_sigma_high_p_60deg_linear_ax.set_xlabel(full_one_layer_refl_tau_sigma_high_p_60deg_x_label)
full_one_layer_refl_tau_sigma_high_p_60deg_linear_ax.set_ylabel(full_one_layer_refl_tau_sigma_high_p_60deg_y_label)
full_one_layer_refl_tau_sigma_high_p_60deg_linear_cbar = full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig.colorbar(full_one_layer_refl_tau_sigma_high_p_60deg_linear_contour, ax=full_one_layer_refl_tau_sigma_high_p_60deg_linear_ax)
full_one_layer_refl_tau_sigma_high_p_60deg_linear_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_high_p_60deg_linear_value_key))
full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_60deg_linear_png = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_linear.png"
full_one_layer_refl_tau_sigma_high_p_60deg_linear_pdf = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_linear.pdf"
full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_linear_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_linear_pdf)

if str(full_one_layer_refl_tau_sigma_high_p_60deg_log_value_key).startswith('signed_err_'):
    full_one_layer_refl_tau_sigma_high_p_60deg_log_source_key = 'abs_err_' + str(full_one_layer_refl_tau_sigma_high_p_60deg_log_value_key)[len('signed_err_'):]
else:
    full_one_layer_refl_tau_sigma_high_p_60deg_log_source_key = full_one_layer_refl_tau_sigma_high_p_60deg_log_value_key
full_one_layer_refl_tau_sigma_high_p_60deg_log_grid = full_one_layer_refl_tau_sigma_high_p_60deg_positive_grid_for_log(full_one_layer_refl_tau_sigma_high_p_60deg_aggregate_grid(full_one_layer_refl_tau_sigma_high_p_60deg_log_source_key))
full_one_layer_refl_tau_sigma_high_p_60deg_log_fig, full_one_layer_refl_tau_sigma_high_p_60deg_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_one_layer_refl_tau_sigma_high_p_60deg_log_finite = full_one_layer_refl_tau_sigma_high_p_60deg_log_grid[np.isfinite(full_one_layer_refl_tau_sigma_high_p_60deg_log_grid)]
full_one_layer_refl_tau_sigma_high_p_60deg_log_positive = full_one_layer_refl_tau_sigma_high_p_60deg_log_finite[full_one_layer_refl_tau_sigma_high_p_60deg_log_finite > 0.0]
full_one_layer_refl_tau_sigma_high_p_60deg_log_vmin = float(np.min(full_one_layer_refl_tau_sigma_high_p_60deg_log_positive))
full_one_layer_refl_tau_sigma_high_p_60deg_log_vmax = float(np.max(full_one_layer_refl_tau_sigma_high_p_60deg_log_positive))
if math.isclose(full_one_layer_refl_tau_sigma_high_p_60deg_log_vmin, full_one_layer_refl_tau_sigma_high_p_60deg_log_vmax):
    full_one_layer_refl_tau_sigma_high_p_60deg_log_vmax = full_one_layer_refl_tau_sigma_high_p_60deg_log_vmin * 1.01
full_one_layer_refl_tau_sigma_high_p_60deg_log_levels = np.geomspace(full_one_layer_refl_tau_sigma_high_p_60deg_log_vmin, full_one_layer_refl_tau_sigma_high_p_60deg_log_vmax, 256)
full_one_layer_refl_tau_sigma_high_p_60deg_log_contour = full_one_layer_refl_tau_sigma_high_p_60deg_log_ax.contourf(
    np.asarray(full_one_layer_refl_tau_sigma_high_p_60deg_x_values, dtype=np.float64),
    np.asarray(full_one_layer_refl_tau_sigma_high_p_60deg_y_values, dtype=np.float64),
    full_one_layer_refl_tau_sigma_high_p_60deg_log_grid,
    levels=full_one_layer_refl_tau_sigma_high_p_60deg_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_one_layer_refl_tau_sigma_high_p_60deg_log_vmin, vmax=full_one_layer_refl_tau_sigma_high_p_60deg_log_vmax),
    extend='both',
)
full_one_layer_refl_tau_sigma_high_p_60deg_log_ax.set_title(full_one_layer_refl_tau_sigma_high_p_60deg_plot_title + ' [log]')
full_one_layer_refl_tau_sigma_high_p_60deg_log_ax.set_xlabel(full_one_layer_refl_tau_sigma_high_p_60deg_x_label)
full_one_layer_refl_tau_sigma_high_p_60deg_log_ax.set_ylabel(full_one_layer_refl_tau_sigma_high_p_60deg_y_label)
full_one_layer_refl_tau_sigma_high_p_60deg_log_cbar = full_one_layer_refl_tau_sigma_high_p_60deg_log_fig.colorbar(full_one_layer_refl_tau_sigma_high_p_60deg_log_contour, ax=full_one_layer_refl_tau_sigma_high_p_60deg_log_ax)
full_one_layer_refl_tau_sigma_high_p_60deg_log_cbar.set_label(metric_label(full_one_layer_refl_tau_sigma_high_p_60deg_log_source_key))
full_one_layer_refl_tau_sigma_high_p_60deg_log_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_60deg_log_png = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_log.png"
full_one_layer_refl_tau_sigma_high_p_60deg_log_pdf = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_log.pdf"
full_one_layer_refl_tau_sigma_high_p_60deg_log_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_log_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_60deg_log_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_log_pdf)

full_one_layer_refl_tau_sigma_high_p_60deg_corr_rows = full_one_layer_refl_tau_sigma_high_p_60deg_corr_meta['rows']
full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels = sorted({str(full_one_layer_refl_tau_sigma_high_p_60deg_row['param_i']) for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_corr_rows} | {str(full_one_layer_refl_tau_sigma_high_p_60deg_row['param_j']) for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_corr_rows})
full_one_layer_refl_tau_sigma_high_p_60deg_corr_index = {full_one_layer_refl_tau_sigma_high_p_60deg_label: full_one_layer_refl_tau_sigma_high_p_60deg_idx for full_one_layer_refl_tau_sigma_high_p_60deg_idx, full_one_layer_refl_tau_sigma_high_p_60deg_label in enumerate(full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels)}
full_one_layer_refl_tau_sigma_high_p_60deg_corr_matrix = np.full((len(full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels), len(full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels)), np.nan, dtype=np.float64)
for full_one_layer_refl_tau_sigma_high_p_60deg_row in full_one_layer_refl_tau_sigma_high_p_60deg_corr_rows:
    full_one_layer_refl_tau_sigma_high_p_60deg_i = full_one_layer_refl_tau_sigma_high_p_60deg_corr_index[str(full_one_layer_refl_tau_sigma_high_p_60deg_row['param_i'])]
    full_one_layer_refl_tau_sigma_high_p_60deg_j = full_one_layer_refl_tau_sigma_high_p_60deg_corr_index[str(full_one_layer_refl_tau_sigma_high_p_60deg_row['param_j'])]
    full_one_layer_refl_tau_sigma_high_p_60deg_corr_matrix[full_one_layer_refl_tau_sigma_high_p_60deg_i, full_one_layer_refl_tau_sigma_high_p_60deg_j] = float(full_one_layer_refl_tau_sigma_high_p_60deg_row['correlation'])
    full_one_layer_refl_tau_sigma_high_p_60deg_corr_matrix[full_one_layer_refl_tau_sigma_high_p_60deg_j, full_one_layer_refl_tau_sigma_high_p_60deg_i] = float(full_one_layer_refl_tau_sigma_high_p_60deg_row['correlation'])
for full_one_layer_refl_tau_sigma_high_p_60deg_diag in range(len(full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels)):
    full_one_layer_refl_tau_sigma_high_p_60deg_corr_matrix[full_one_layer_refl_tau_sigma_high_p_60deg_diag, full_one_layer_refl_tau_sigma_high_p_60deg_diag] = 1.0

full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig, full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_one_layer_refl_tau_sigma_high_p_60deg_corr_image = full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax.imshow(full_one_layer_refl_tau_sigma_high_p_60deg_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax.set_title(full_one_layer_refl_tau_sigma_high_p_60deg_plot_title + ' [correlation]')
full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax.set_xticks(range(len(full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels)))
full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax.set_yticks(range(len(full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels)))
full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_high_p_60deg_label, full_one_layer_refl_tau_sigma_high_p_60deg_label) for full_one_layer_refl_tau_sigma_high_p_60deg_label in full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels], rotation=45, ha='right')
full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_one_layer_refl_tau_sigma_high_p_60deg_label, full_one_layer_refl_tau_sigma_high_p_60deg_label) for full_one_layer_refl_tau_sigma_high_p_60deg_label in full_one_layer_refl_tau_sigma_high_p_60deg_corr_labels])
full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig.colorbar(full_one_layer_refl_tau_sigma_high_p_60deg_corr_image, ax=full_one_layer_refl_tau_sigma_high_p_60deg_corr_ax, label=r'$\rho_{ij}$')
full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_60deg_corr_png = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_corr.png"
full_one_layer_refl_tau_sigma_high_p_60deg_corr_pdf = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_corr.pdf"
full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_corr_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_corr_pdf)

full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig, full_one_layer_refl_tau_sigma_high_p_60deg_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_one_layer_refl_tau_sigma_high_p_60deg_axis, full_one_layer_refl_tau_sigma_high_p_60deg_image_path, full_one_layer_refl_tau_sigma_high_p_60deg_panel_title in zip(
    full_one_layer_refl_tau_sigma_high_p_60deg_triptych_axes,
    (full_one_layer_refl_tau_sigma_high_p_60deg_linear_png, full_one_layer_refl_tau_sigma_high_p_60deg_log_png, full_one_layer_refl_tau_sigma_high_p_60deg_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_one_layer_refl_tau_sigma_high_p_60deg_image = plt.imread(str(full_one_layer_refl_tau_sigma_high_p_60deg_image_path))
    full_one_layer_refl_tau_sigma_high_p_60deg_axis.imshow(full_one_layer_refl_tau_sigma_high_p_60deg_image)
    full_one_layer_refl_tau_sigma_high_p_60deg_axis.set_title(full_one_layer_refl_tau_sigma_high_p_60deg_panel_title)
    full_one_layer_refl_tau_sigma_high_p_60deg_axis.axis('off')
full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig.suptitle(full_one_layer_refl_tau_sigma_high_p_60deg_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig.tight_layout()
full_one_layer_refl_tau_sigma_high_p_60deg_triptych_png = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_triptych.png"
full_one_layer_refl_tau_sigma_high_p_60deg_triptych_pdf = full_one_layer_refl_tau_sigma_high_p_60deg_study_dir / f"{full_one_layer_refl_tau_sigma_high_p_60deg_study_dir.name}__section_triptych.pdf"
full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_triptych_png, dpi=220)
full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig.savefig(full_one_layer_refl_tau_sigma_high_p_60deg_triptych_pdf)

display(full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig)
display(full_one_layer_refl_tau_sigma_high_p_60deg_log_fig)
display(full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig)
display(full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_60deg_linear_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_60deg_log_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_60deg_corr_fig)
plt.close(full_one_layer_refl_tau_sigma_high_p_60deg_triptych_fig)

full_one_layer_refl_tau_sigma_high_p_60deg_plot_paths = {
    'linear_png': full_one_layer_refl_tau_sigma_high_p_60deg_linear_png.resolve().as_posix(),
    'linear_pdf': full_one_layer_refl_tau_sigma_high_p_60deg_linear_pdf.resolve().as_posix(),
    'log_png': full_one_layer_refl_tau_sigma_high_p_60deg_log_png.resolve().as_posix(),
    'log_pdf': full_one_layer_refl_tau_sigma_high_p_60deg_log_pdf.resolve().as_posix(),
    'corr_png': full_one_layer_refl_tau_sigma_high_p_60deg_corr_png.resolve().as_posix(),
    'corr_pdf': full_one_layer_refl_tau_sigma_high_p_60deg_corr_pdf.resolve().as_posix(),
    'triptych_png': full_one_layer_refl_tau_sigma_high_p_60deg_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_one_layer_refl_tau_sigma_high_p_60deg_triptych_pdf.resolve().as_posix(),
}
full_one_layer_refl_tau_sigma_high_p_60deg_plot_paths

## Full-Resolution Advanced Epi Study Blocks

Every block below contains its own calculation cell and its own complete plotting cell.

### Two-Drude transmission: $\tau_1$ vs $\tau_2$ (0 deg, s-pol, $d_{\mathrm{sub}}=525\,\mu$m, $d_{\mathrm{epi}}=10\,\mu$m)

In [ ]:
full_advanced_tx_tau1_tau2_0deg_s_525um_spec = {
  "slug": "advanced_tx_tau1_tau2_0deg_s_525um",
  "title": "Two-Drude transmission: $\\tau_1$ vs $\\tau_2$ (0 deg, s-pol, $d_{\\mathrm{sub}}=525\\,\\mu$m, $d_{\\mathrm{epi}}=10\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 0.0,
  "polarization": "s",
  "map_kind": "tau",
  "substrate_thickness_um": 525.0,
  "epi_thickness_um": 10.0,
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_fixed": 0.1,
  "x_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "y_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ]
}
full_advanced_tx_tau1_tau2_0deg_s_525um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_tx_tau1_tau2_0deg_s_525um'
full_advanced_tx_tau1_tau2_0deg_s_525um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_tx_tau1_tau2_0deg_s_525um_result = run_lecture_map_from_spec(full_advanced_tx_tau1_tau2_0deg_s_525um_spec, output_root=full_advanced_tx_tau1_tau2_0deg_s_525um_output_root, profile='full')
full_advanced_tx_tau1_tau2_0deg_s_525um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir = Path(full_advanced_tx_tau1_tau2_0deg_s_525um_result['study_dir'])
full_advanced_tx_tau1_tau2_0deg_s_525um_plot_title = full_advanced_tx_tau1_tau2_0deg_s_525um_result['title']
full_advanced_tx_tau1_tau2_0deg_s_525um_x_label = "$\\tau_1$ (ps)"
full_advanced_tx_tau1_tau2_0deg_s_525um_y_label = "$\\tau_2$ (ps)"

with (full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_tx_tau1_tau2_0deg_s_525um_summary_handle:
    full_advanced_tx_tau1_tau2_0deg_s_525um_summary_meta = json.load(full_advanced_tx_tau1_tau2_0deg_s_525um_summary_handle)
with (full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_tx_tau1_tau2_0deg_s_525um_corr_handle:
    full_advanced_tx_tau1_tau2_0deg_s_525um_corr_meta = json.load(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_handle)

full_advanced_tx_tau1_tau2_0deg_s_525um_rows = []
with (full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_tx_tau1_tau2_0deg_s_525um_summary_csv:
    full_advanced_tx_tau1_tau2_0deg_s_525um_reader = csv.DictReader(full_advanced_tx_tau1_tau2_0deg_s_525um_summary_csv)
    for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_reader:
        full_advanced_tx_tau1_tau2_0deg_s_525um_parsed = {}
        for full_advanced_tx_tau1_tau2_0deg_s_525um_key, full_advanced_tx_tau1_tau2_0deg_s_525um_value in full_advanced_tx_tau1_tau2_0deg_s_525um_row.items():
            if full_advanced_tx_tau1_tau2_0deg_s_525um_value is None:
                full_advanced_tx_tau1_tau2_0deg_s_525um_parsed[full_advanced_tx_tau1_tau2_0deg_s_525um_key] = full_advanced_tx_tau1_tau2_0deg_s_525um_value
                continue
            full_advanced_tx_tau1_tau2_0deg_s_525um_text = str(full_advanced_tx_tau1_tau2_0deg_s_525um_value).strip()
            if full_advanced_tx_tau1_tau2_0deg_s_525um_text == '':
                full_advanced_tx_tau1_tau2_0deg_s_525um_parsed[full_advanced_tx_tau1_tau2_0deg_s_525um_key] = full_advanced_tx_tau1_tau2_0deg_s_525um_text
                continue
            try:
                full_advanced_tx_tau1_tau2_0deg_s_525um_parsed[full_advanced_tx_tau1_tau2_0deg_s_525um_key] = float(full_advanced_tx_tau1_tau2_0deg_s_525um_text)
            except ValueError:
                full_advanced_tx_tau1_tau2_0deg_s_525um_parsed[full_advanced_tx_tau1_tau2_0deg_s_525um_key] = full_advanced_tx_tau1_tau2_0deg_s_525um_text
        full_advanced_tx_tau1_tau2_0deg_s_525um_rows.append(full_advanced_tx_tau1_tau2_0deg_s_525um_parsed)

full_advanced_tx_tau1_tau2_0deg_s_525um_x_key, full_advanced_tx_tau1_tau2_0deg_s_525um_y_key = list(full_advanced_tx_tau1_tau2_0deg_s_525um_summary_meta['recovery_keys'])
full_advanced_tx_tau1_tau2_0deg_s_525um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_tx_tau1_tau2_0deg_s_525um_x_key}_true_minus_fit': f'signed_err_{full_advanced_tx_tau1_tau2_0deg_s_525um_x_key}',
    f'{full_advanced_tx_tau1_tau2_0deg_s_525um_y_key}_true_minus_fit': f'signed_err_{full_advanced_tx_tau1_tau2_0deg_s_525um_y_key}',
    f'abs_{full_advanced_tx_tau1_tau2_0deg_s_525um_x_key}_error': f'abs_err_{full_advanced_tx_tau1_tau2_0deg_s_525um_x_key}',
    f'abs_{full_advanced_tx_tau1_tau2_0deg_s_525um_y_key}_error': f'abs_err_{full_advanced_tx_tau1_tau2_0deg_s_525um_y_key}',
}
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key = full_advanced_tx_tau1_tau2_0deg_s_525um_metric_options['data_fit']
full_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key = full_advanced_tx_tau1_tau2_0deg_s_525um_metric_options['data_fit']
full_advanced_tx_tau1_tau2_0deg_s_525um_x_values = sorted({float(full_advanced_tx_tau1_tau2_0deg_s_525um_row[full_advanced_tx_tau1_tau2_0deg_s_525um_x_key]) for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_rows})
full_advanced_tx_tau1_tau2_0deg_s_525um_y_values = sorted({float(full_advanced_tx_tau1_tau2_0deg_s_525um_row[full_advanced_tx_tau1_tau2_0deg_s_525um_y_key]) for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_rows})

print('Available map metrics:')
for full_advanced_tx_tau1_tau2_0deg_s_525um_option_name, full_advanced_tx_tau1_tau2_0deg_s_525um_option_key in full_advanced_tx_tau1_tau2_0deg_s_525um_metric_options.items():
    print(f"  {full_advanced_tx_tau1_tau2_0deg_s_525um_option_name}: {metric_label(full_advanced_tx_tau1_tau2_0deg_s_525um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key)}")

def full_advanced_tx_tau1_tau2_0deg_s_525um_aggregate_grid(value_key):
    full_advanced_tx_tau1_tau2_0deg_s_525um_grid = np.full((len(full_advanced_tx_tau1_tau2_0deg_s_525um_y_values), len(full_advanced_tx_tau1_tau2_0deg_s_525um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_tx_tau1_tau2_0deg_s_525um_iy, full_advanced_tx_tau1_tau2_0deg_s_525um_y_value in enumerate(full_advanced_tx_tau1_tau2_0deg_s_525um_y_values):
        for full_advanced_tx_tau1_tau2_0deg_s_525um_ix, full_advanced_tx_tau1_tau2_0deg_s_525um_x_value in enumerate(full_advanced_tx_tau1_tau2_0deg_s_525um_x_values):
            full_advanced_tx_tau1_tau2_0deg_s_525um_cell = [
                float(full_advanced_tx_tau1_tau2_0deg_s_525um_row[value_key])
                for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_rows
                if math.isclose(float(full_advanced_tx_tau1_tau2_0deg_s_525um_row[full_advanced_tx_tau1_tau2_0deg_s_525um_x_key]), full_advanced_tx_tau1_tau2_0deg_s_525um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_tx_tau1_tau2_0deg_s_525um_row[full_advanced_tx_tau1_tau2_0deg_s_525um_y_key]), full_advanced_tx_tau1_tau2_0deg_s_525um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_tx_tau1_tau2_0deg_s_525um_row[value_key]))
            ]
            if full_advanced_tx_tau1_tau2_0deg_s_525um_cell:
                full_advanced_tx_tau1_tau2_0deg_s_525um_grid[full_advanced_tx_tau1_tau2_0deg_s_525um_iy, full_advanced_tx_tau1_tau2_0deg_s_525um_ix] = float(np.mean(full_advanced_tx_tau1_tau2_0deg_s_525um_cell))
    return full_advanced_tx_tau1_tau2_0deg_s_525um_grid

def full_advanced_tx_tau1_tau2_0deg_s_525um_positive_grid_for_log(grid):
    full_advanced_tx_tau1_tau2_0deg_s_525um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_tx_tau1_tau2_0deg_s_525um_finite = full_advanced_tx_tau1_tau2_0deg_s_525um_grid[np.isfinite(full_advanced_tx_tau1_tau2_0deg_s_525um_grid)]
    full_advanced_tx_tau1_tau2_0deg_s_525um_positive = full_advanced_tx_tau1_tau2_0deg_s_525um_finite[full_advanced_tx_tau1_tau2_0deg_s_525um_finite > 0.0]
    if full_advanced_tx_tau1_tau2_0deg_s_525um_positive.size == 0:
        full_advanced_tx_tau1_tau2_0deg_s_525um_grid[np.isfinite(full_advanced_tx_tau1_tau2_0deg_s_525um_grid)] = 1.0
        return full_advanced_tx_tau1_tau2_0deg_s_525um_grid
    full_advanced_tx_tau1_tau2_0deg_s_525um_floor = max(float(np.min(full_advanced_tx_tau1_tau2_0deg_s_525um_positive)) * 0.5, 1e-18)
    full_advanced_tx_tau1_tau2_0deg_s_525um_grid[np.isfinite(full_advanced_tx_tau1_tau2_0deg_s_525um_grid) & (full_advanced_tx_tau1_tau2_0deg_s_525um_grid <= 0.0)] = full_advanced_tx_tau1_tau2_0deg_s_525um_floor
    return full_advanced_tx_tau1_tau2_0deg_s_525um_grid

full_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid = full_advanced_tx_tau1_tau2_0deg_s_525um_aggregate_grid(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key)
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig, full_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite = full_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid[np.isfinite(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid)]
if str(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key).startswith('signed_err_'):
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_limit = max(float(np.max(np.abs(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite))), 1e-18)
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin = -full_advanced_tx_tau1_tau2_0deg_s_525um_linear_limit
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax = full_advanced_tx_tau1_tau2_0deg_s_525um_linear_limit
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_levels = np.linspace(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin, full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax, 256)
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_cmap = 'plasma'
else:
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin = float(np.min(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite))
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax = float(np.max(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_finite)) + 1e-12
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_levels = np.linspace(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmin, full_advanced_tx_tau1_tau2_0deg_s_525um_linear_vmax, 256)
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_cmap = 'plasma'
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_contour = full_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.contourf(
    np.asarray(full_advanced_tx_tau1_tau2_0deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_tau1_tau2_0deg_s_525um_y_values, dtype=np.float64),
    full_advanced_tx_tau1_tau2_0deg_s_525um_linear_grid,
    levels=full_advanced_tx_tau1_tau2_0deg_s_525um_linear_levels,
    cmap=full_advanced_tx_tau1_tau2_0deg_s_525um_linear_cmap,
    extend='both',
)
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.set_title(full_advanced_tx_tau1_tau2_0deg_s_525um_plot_title + ' [linear]')
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.set_xlabel(full_advanced_tx_tau1_tau2_0deg_s_525um_x_label)
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax.set_ylabel(full_advanced_tx_tau1_tau2_0deg_s_525um_y_label)
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_cbar = full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.colorbar(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_contour, ax=full_advanced_tx_tau1_tau2_0deg_s_525um_linear_ax)
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_cbar.set_label(metric_label(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_value_key))
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.tight_layout()
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_png = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_linear.png"
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_pdf = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_linear.pdf"
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_png, dpi=220)
full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_pdf)

if str(full_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key).startswith('signed_err_'):
    full_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key = 'abs_err_' + str(full_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key = full_advanced_tx_tau1_tau2_0deg_s_525um_log_value_key
full_advanced_tx_tau1_tau2_0deg_s_525um_log_grid = full_advanced_tx_tau1_tau2_0deg_s_525um_positive_grid_for_log(full_advanced_tx_tau1_tau2_0deg_s_525um_aggregate_grid(full_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key))
full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig, full_advanced_tx_tau1_tau2_0deg_s_525um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_tau1_tau2_0deg_s_525um_log_finite = full_advanced_tx_tau1_tau2_0deg_s_525um_log_grid[np.isfinite(full_advanced_tx_tau1_tau2_0deg_s_525um_log_grid)]
full_advanced_tx_tau1_tau2_0deg_s_525um_log_positive = full_advanced_tx_tau1_tau2_0deg_s_525um_log_finite[full_advanced_tx_tau1_tau2_0deg_s_525um_log_finite > 0.0]
full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin = float(np.min(full_advanced_tx_tau1_tau2_0deg_s_525um_log_positive))
full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax = float(np.max(full_advanced_tx_tau1_tau2_0deg_s_525um_log_positive))
if math.isclose(full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin, full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax):
    full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax = full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin * 1.01
full_advanced_tx_tau1_tau2_0deg_s_525um_log_levels = np.geomspace(full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin, full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax, 256)
full_advanced_tx_tau1_tau2_0deg_s_525um_log_contour = full_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.contourf(
    np.asarray(full_advanced_tx_tau1_tau2_0deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_tau1_tau2_0deg_s_525um_y_values, dtype=np.float64),
    full_advanced_tx_tau1_tau2_0deg_s_525um_log_grid,
    levels=full_advanced_tx_tau1_tau2_0deg_s_525um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmin, vmax=full_advanced_tx_tau1_tau2_0deg_s_525um_log_vmax),
    extend='both',
)
full_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.set_title(full_advanced_tx_tau1_tau2_0deg_s_525um_plot_title + ' [log]')
full_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.set_xlabel(full_advanced_tx_tau1_tau2_0deg_s_525um_x_label)
full_advanced_tx_tau1_tau2_0deg_s_525um_log_ax.set_ylabel(full_advanced_tx_tau1_tau2_0deg_s_525um_y_label)
full_advanced_tx_tau1_tau2_0deg_s_525um_log_cbar = full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.colorbar(full_advanced_tx_tau1_tau2_0deg_s_525um_log_contour, ax=full_advanced_tx_tau1_tau2_0deg_s_525um_log_ax)
full_advanced_tx_tau1_tau2_0deg_s_525um_log_cbar.set_label(metric_label(full_advanced_tx_tau1_tau2_0deg_s_525um_log_source_key))
full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.tight_layout()
full_advanced_tx_tau1_tau2_0deg_s_525um_log_png = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_log.png"
full_advanced_tx_tau1_tau2_0deg_s_525um_log_pdf = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_log.pdf"
full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_log_png, dpi=220)
full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_log_pdf)

full_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows = full_advanced_tx_tau1_tau2_0deg_s_525um_corr_meta['rows']
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels = sorted({str(full_advanced_tx_tau1_tau2_0deg_s_525um_row['param_i']) for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows} | {str(full_advanced_tx_tau1_tau2_0deg_s_525um_row['param_j']) for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows})
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_index = {full_advanced_tx_tau1_tau2_0deg_s_525um_label: full_advanced_tx_tau1_tau2_0deg_s_525um_idx for full_advanced_tx_tau1_tau2_0deg_s_525um_idx, full_advanced_tx_tau1_tau2_0deg_s_525um_label in enumerate(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)}
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix = np.full((len(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels), len(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_tx_tau1_tau2_0deg_s_525um_row in full_advanced_tx_tau1_tau2_0deg_s_525um_corr_rows:
    full_advanced_tx_tau1_tau2_0deg_s_525um_i = full_advanced_tx_tau1_tau2_0deg_s_525um_corr_index[str(full_advanced_tx_tau1_tau2_0deg_s_525um_row['param_i'])]
    full_advanced_tx_tau1_tau2_0deg_s_525um_j = full_advanced_tx_tau1_tau2_0deg_s_525um_corr_index[str(full_advanced_tx_tau1_tau2_0deg_s_525um_row['param_j'])]
    full_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix[full_advanced_tx_tau1_tau2_0deg_s_525um_i, full_advanced_tx_tau1_tau2_0deg_s_525um_j] = float(full_advanced_tx_tau1_tau2_0deg_s_525um_row['correlation'])
    full_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix[full_advanced_tx_tau1_tau2_0deg_s_525um_j, full_advanced_tx_tau1_tau2_0deg_s_525um_i] = float(full_advanced_tx_tau1_tau2_0deg_s_525um_row['correlation'])
for full_advanced_tx_tau1_tau2_0deg_s_525um_diag in range(len(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)):
    full_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix[full_advanced_tx_tau1_tau2_0deg_s_525um_diag, full_advanced_tx_tau1_tau2_0deg_s_525um_diag] = 1.0

full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig, full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_image = full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.imshow(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_title(full_advanced_tx_tau1_tau2_0deg_s_525um_plot_title + ' [correlation]')
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_xticks(range(len(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)))
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_yticks(range(len(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels)))
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_tx_tau1_tau2_0deg_s_525um_label, full_advanced_tx_tau1_tau2_0deg_s_525um_label) for full_advanced_tx_tau1_tau2_0deg_s_525um_label in full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels], rotation=45, ha='right')
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_tx_tau1_tau2_0deg_s_525um_label, full_advanced_tx_tau1_tau2_0deg_s_525um_label) for full_advanced_tx_tau1_tau2_0deg_s_525um_label in full_advanced_tx_tau1_tau2_0deg_s_525um_corr_labels])
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.colorbar(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_image, ax=full_advanced_tx_tau1_tau2_0deg_s_525um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.tight_layout()
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_png = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_corr.png"
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_pdf = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_corr.pdf"
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_png, dpi=220)
full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_pdf)

full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig, full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_tx_tau1_tau2_0deg_s_525um_axis, full_advanced_tx_tau1_tau2_0deg_s_525um_image_path, full_advanced_tx_tau1_tau2_0deg_s_525um_panel_title in zip(
    full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_axes,
    (full_advanced_tx_tau1_tau2_0deg_s_525um_linear_png, full_advanced_tx_tau1_tau2_0deg_s_525um_log_png, full_advanced_tx_tau1_tau2_0deg_s_525um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_tx_tau1_tau2_0deg_s_525um_image = plt.imread(str(full_advanced_tx_tau1_tau2_0deg_s_525um_image_path))
    full_advanced_tx_tau1_tau2_0deg_s_525um_axis.imshow(full_advanced_tx_tau1_tau2_0deg_s_525um_image)
    full_advanced_tx_tau1_tau2_0deg_s_525um_axis.set_title(full_advanced_tx_tau1_tau2_0deg_s_525um_panel_title)
    full_advanced_tx_tau1_tau2_0deg_s_525um_axis.axis('off')
full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.suptitle(full_advanced_tx_tau1_tau2_0deg_s_525um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.tight_layout()
full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_png = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_triptych.png"
full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_pdf = full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir / f"{full_advanced_tx_tau1_tau2_0deg_s_525um_study_dir.name}__section_triptych.pdf"
full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_png, dpi=220)
full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig.savefig(full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_pdf)

display(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig)
display(full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig)
display(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig)
display(full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig)
plt.close(full_advanced_tx_tau1_tau2_0deg_s_525um_linear_fig)
plt.close(full_advanced_tx_tau1_tau2_0deg_s_525um_log_fig)
plt.close(full_advanced_tx_tau1_tau2_0deg_s_525um_corr_fig)
plt.close(full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_fig)

full_advanced_tx_tau1_tau2_0deg_s_525um_plot_paths = {
    'linear_png': full_advanced_tx_tau1_tau2_0deg_s_525um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_tx_tau1_tau2_0deg_s_525um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_tx_tau1_tau2_0deg_s_525um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_tx_tau1_tau2_0deg_s_525um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_tx_tau1_tau2_0deg_s_525um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_tx_tau1_tau2_0deg_s_525um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_tx_tau1_tau2_0deg_s_525um_triptych_pdf.resolve().as_posix(),
}
full_advanced_tx_tau1_tau2_0deg_s_525um_plot_paths

### Two-Drude transmission: $\sigma_1$ vs $\sigma_2$ (0 deg, s-pol, $d_{\mathrm{sub}}=525\,\mu$m, $d_{\mathrm{epi}}=10\,\mu$m)

In [ ]:
full_advanced_tx_sigma1_sigma2_0deg_s_525um_spec = {
  "slug": "advanced_tx_sigma1_sigma2_0deg_s_525um",
  "title": "Two-Drude transmission: $\\sigma_1$ vs $\\sigma_2$ (0 deg, s-pol, $d_{\\mathrm{sub}}=525\\,\\mu$m, $d_{\\mathrm{epi}}=10\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 0.0,
  "polarization": "s",
  "map_kind": "sigma",
  "substrate_thickness_um": 525.0,
  "epi_thickness_um": 10.0,
  "sigma_range": [
    0.01,
    1.0
  ],
  "tau_fixed": 0.316,
  "x_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ],
  "y_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
full_advanced_tx_sigma1_sigma2_0deg_s_525um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_tx_sigma1_sigma2_0deg_s_525um'
full_advanced_tx_sigma1_sigma2_0deg_s_525um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_result = run_lecture_map_from_spec(full_advanced_tx_sigma1_sigma2_0deg_s_525um_spec, output_root=full_advanced_tx_sigma1_sigma2_0deg_s_525um_output_root, profile='full')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir = Path(full_advanced_tx_sigma1_sigma2_0deg_s_525um_result['study_dir'])
full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_title = full_advanced_tx_sigma1_sigma2_0deg_s_525um_result['title']
full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_label = "$\\sigma_1$ (S/m)"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_label = "$\\sigma_2$ (S/m)"

with (full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_tx_sigma1_sigma2_0deg_s_525um_summary_handle:
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_summary_meta = json.load(full_advanced_tx_sigma1_sigma2_0deg_s_525um_summary_handle)
with (full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_handle:
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_meta = json.load(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_handle)

full_advanced_tx_sigma1_sigma2_0deg_s_525um_rows = []
with (full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_tx_sigma1_sigma2_0deg_s_525um_summary_csv:
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_reader = csv.DictReader(full_advanced_tx_sigma1_sigma2_0deg_s_525um_summary_csv)
    for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_reader:
        full_advanced_tx_sigma1_sigma2_0deg_s_525um_parsed = {}
        for full_advanced_tx_sigma1_sigma2_0deg_s_525um_key, full_advanced_tx_sigma1_sigma2_0deg_s_525um_value in full_advanced_tx_sigma1_sigma2_0deg_s_525um_row.items():
            if full_advanced_tx_sigma1_sigma2_0deg_s_525um_value is None:
                full_advanced_tx_sigma1_sigma2_0deg_s_525um_parsed[full_advanced_tx_sigma1_sigma2_0deg_s_525um_key] = full_advanced_tx_sigma1_sigma2_0deg_s_525um_value
                continue
            full_advanced_tx_sigma1_sigma2_0deg_s_525um_text = str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_value).strip()
            if full_advanced_tx_sigma1_sigma2_0deg_s_525um_text == '':
                full_advanced_tx_sigma1_sigma2_0deg_s_525um_parsed[full_advanced_tx_sigma1_sigma2_0deg_s_525um_key] = full_advanced_tx_sigma1_sigma2_0deg_s_525um_text
                continue
            try:
                full_advanced_tx_sigma1_sigma2_0deg_s_525um_parsed[full_advanced_tx_sigma1_sigma2_0deg_s_525um_key] = float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_text)
            except ValueError:
                full_advanced_tx_sigma1_sigma2_0deg_s_525um_parsed[full_advanced_tx_sigma1_sigma2_0deg_s_525um_key] = full_advanced_tx_sigma1_sigma2_0deg_s_525um_text
        full_advanced_tx_sigma1_sigma2_0deg_s_525um_rows.append(full_advanced_tx_sigma1_sigma2_0deg_s_525um_parsed)

full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key, full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key = list(full_advanced_tx_sigma1_sigma2_0deg_s_525um_summary_meta['recovery_keys'])
full_advanced_tx_sigma1_sigma2_0deg_s_525um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key}_true_minus_fit': f'signed_err_{full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key}',
    f'{full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key}_true_minus_fit': f'signed_err_{full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key}',
    f'abs_{full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key}_error': f'abs_err_{full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key}',
    f'abs_{full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key}_error': f'abs_err_{full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key}',
}
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_value_key = full_advanced_tx_sigma1_sigma2_0deg_s_525um_metric_options['data_fit']
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_value_key = full_advanced_tx_sigma1_sigma2_0deg_s_525um_metric_options['data_fit']
full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_values = sorted({float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row[full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key]) for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_rows})
full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_values = sorted({float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row[full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key]) for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_rows})

print('Available map metrics:')
for full_advanced_tx_sigma1_sigma2_0deg_s_525um_option_name, full_advanced_tx_sigma1_sigma2_0deg_s_525um_option_key in full_advanced_tx_sigma1_sigma2_0deg_s_525um_metric_options.items():
    print(f"  {full_advanced_tx_sigma1_sigma2_0deg_s_525um_option_name}: {metric_label(full_advanced_tx_sigma1_sigma2_0deg_s_525um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_value_key)}")

def full_advanced_tx_sigma1_sigma2_0deg_s_525um_aggregate_grid(value_key):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid = np.full((len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_values), len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_tx_sigma1_sigma2_0deg_s_525um_iy, full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_value in enumerate(full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_values):
        for full_advanced_tx_sigma1_sigma2_0deg_s_525um_ix, full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_value in enumerate(full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_values):
            full_advanced_tx_sigma1_sigma2_0deg_s_525um_cell = [
                float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row[value_key])
                for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_rows
                if math.isclose(float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row[full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_key]), full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row[full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_key]), full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row[value_key]))
            ]
            if full_advanced_tx_sigma1_sigma2_0deg_s_525um_cell:
                full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid[full_advanced_tx_sigma1_sigma2_0deg_s_525um_iy, full_advanced_tx_sigma1_sigma2_0deg_s_525um_ix] = float(np.mean(full_advanced_tx_sigma1_sigma2_0deg_s_525um_cell))
    return full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid

def full_advanced_tx_sigma1_sigma2_0deg_s_525um_positive_grid_for_log(grid):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_finite = full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid)]
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_positive = full_advanced_tx_sigma1_sigma2_0deg_s_525um_finite[full_advanced_tx_sigma1_sigma2_0deg_s_525um_finite > 0.0]
    if full_advanced_tx_sigma1_sigma2_0deg_s_525um_positive.size == 0:
        full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid)] = 1.0
        return full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_floor = max(float(np.min(full_advanced_tx_sigma1_sigma2_0deg_s_525um_positive)) * 0.5, 1e-18)
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid) & (full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid <= 0.0)] = full_advanced_tx_sigma1_sigma2_0deg_s_525um_floor
    return full_advanced_tx_sigma1_sigma2_0deg_s_525um_grid

full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_grid = full_advanced_tx_sigma1_sigma2_0deg_s_525um_aggregate_grid(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_value_key)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig, full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_finite = full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_grid)]
if str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_value_key).startswith('signed_err_'):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_limit = max(float(np.max(np.abs(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_finite))), 1e-18)
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmin = -full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_limit
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmax = full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_limit
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_levels = np.linspace(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmin, full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmax, 256)
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_cmap = 'plasma'
else:
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmin = float(np.min(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_finite))
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmax = float(np.max(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_finite)) + 1e-12
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_levels = np.linspace(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmin, full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_vmax, 256)
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_cmap = 'plasma'
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_contour = full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_ax.contourf(
    np.asarray(full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_values, dtype=np.float64),
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_grid,
    levels=full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_levels,
    cmap=full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_cmap,
    extend='both',
)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_ax.set_title(full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_title + ' [linear]')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_ax.set_xlabel(full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_label)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_ax.set_ylabel(full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_label)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_cbar = full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig.colorbar(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_contour, ax=full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_ax)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_cbar.set_label(metric_label(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_value_key))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_png = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_linear.png"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_pdf = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_linear.pdf"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_png, dpi=220)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_pdf)

if str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_value_key).startswith('signed_err_'):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_source_key = 'abs_err_' + str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_source_key = full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_value_key
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_grid = full_advanced_tx_sigma1_sigma2_0deg_s_525um_positive_grid_for_log(full_advanced_tx_sigma1_sigma2_0deg_s_525um_aggregate_grid(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_source_key))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig, full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_finite = full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_grid)]
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_positive = full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_finite[full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_finite > 0.0]
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmin = float(np.min(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_positive))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmax = float(np.max(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_positive))
if math.isclose(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmin, full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmax):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmax = full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmin * 1.01
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_levels = np.geomspace(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmin, full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmax, 256)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_contour = full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_ax.contourf(
    np.asarray(full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_values, dtype=np.float64),
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_grid,
    levels=full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmin, vmax=full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_vmax),
    extend='both',
)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_ax.set_title(full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_title + ' [log]')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_ax.set_xlabel(full_advanced_tx_sigma1_sigma2_0deg_s_525um_x_label)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_ax.set_ylabel(full_advanced_tx_sigma1_sigma2_0deg_s_525um_y_label)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_cbar = full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig.colorbar(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_contour, ax=full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_ax)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_cbar.set_label(metric_label(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_source_key))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_png = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_log.png"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_pdf = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_log.pdf"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_png, dpi=220)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_pdf)

full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_rows = full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_meta['rows']
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels = sorted({str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row['param_i']) for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_rows} | {str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row['param_j']) for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_rows})
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_index = {full_advanced_tx_sigma1_sigma2_0deg_s_525um_label: full_advanced_tx_sigma1_sigma2_0deg_s_525um_idx for full_advanced_tx_sigma1_sigma2_0deg_s_525um_idx, full_advanced_tx_sigma1_sigma2_0deg_s_525um_label in enumerate(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels)}
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_matrix = np.full((len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels), len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_tx_sigma1_sigma2_0deg_s_525um_row in full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_rows:
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_i = full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_index[str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row['param_i'])]
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_j = full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_index[str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row['param_j'])]
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_matrix[full_advanced_tx_sigma1_sigma2_0deg_s_525um_i, full_advanced_tx_sigma1_sigma2_0deg_s_525um_j] = float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row['correlation'])
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_matrix[full_advanced_tx_sigma1_sigma2_0deg_s_525um_j, full_advanced_tx_sigma1_sigma2_0deg_s_525um_i] = float(full_advanced_tx_sigma1_sigma2_0deg_s_525um_row['correlation'])
for full_advanced_tx_sigma1_sigma2_0deg_s_525um_diag in range(len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels)):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_matrix[full_advanced_tx_sigma1_sigma2_0deg_s_525um_diag, full_advanced_tx_sigma1_sigma2_0deg_s_525um_diag] = 1.0

full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig, full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_image = full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax.imshow(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax.set_title(full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_title + ' [correlation]')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax.set_xticks(range(len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels)))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax.set_yticks(range(len(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels)))
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_tx_sigma1_sigma2_0deg_s_525um_label, full_advanced_tx_sigma1_sigma2_0deg_s_525um_label) for full_advanced_tx_sigma1_sigma2_0deg_s_525um_label in full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels], rotation=45, ha='right')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_tx_sigma1_sigma2_0deg_s_525um_label, full_advanced_tx_sigma1_sigma2_0deg_s_525um_label) for full_advanced_tx_sigma1_sigma2_0deg_s_525um_label in full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_labels])
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig.colorbar(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_image, ax=full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_png = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_corr.png"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_pdf = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_corr.pdf"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_png, dpi=220)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_pdf)

full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig, full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_tx_sigma1_sigma2_0deg_s_525um_axis, full_advanced_tx_sigma1_sigma2_0deg_s_525um_image_path, full_advanced_tx_sigma1_sigma2_0deg_s_525um_panel_title in zip(
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_axes,
    (full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_png, full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_png, full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_image = plt.imread(str(full_advanced_tx_sigma1_sigma2_0deg_s_525um_image_path))
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_axis.imshow(full_advanced_tx_sigma1_sigma2_0deg_s_525um_image)
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_axis.set_title(full_advanced_tx_sigma1_sigma2_0deg_s_525um_panel_title)
    full_advanced_tx_sigma1_sigma2_0deg_s_525um_axis.axis('off')
full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig.suptitle(full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_png = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_triptych.png"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_pdf = full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir / f"{full_advanced_tx_sigma1_sigma2_0deg_s_525um_study_dir.name}__section_triptych.pdf"
full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_png, dpi=220)
full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig.savefig(full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_pdf)

display(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig)
display(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig)
display(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig)
display(full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig)
plt.close(full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_fig)
plt.close(full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_fig)
plt.close(full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_fig)
plt.close(full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_fig)

full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_paths = {
    'linear_png': full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_tx_sigma1_sigma2_0deg_s_525um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_tx_sigma1_sigma2_0deg_s_525um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_tx_sigma1_sigma2_0deg_s_525um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_tx_sigma1_sigma2_0deg_s_525um_triptych_pdf.resolve().as_posix(),
}
full_advanced_tx_sigma1_sigma2_0deg_s_525um_plot_paths

### Two-Drude transmission: $\tau_1$ vs $\tau_2$ (45 deg, s-pol, $d_{\mathrm{sub}}=725\,\mu$m, $d_{\mathrm{epi}}=50\,\mu$m)

In [ ]:
full_advanced_tx_tau1_tau2_45deg_s_725um_spec = {
  "slug": "advanced_tx_tau1_tau2_45deg_s_725um",
  "title": "Two-Drude transmission: $\\tau_1$ vs $\\tau_2$ (45 deg, s-pol, $d_{\\mathrm{sub}}=725\\,\\mu$m, $d_{\\mathrm{epi}}=50\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 45.0,
  "polarization": "s",
  "map_kind": "tau",
  "substrate_thickness_um": 725.0,
  "epi_thickness_um": 50.0,
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_fixed": 316.2,
  "x_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "y_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ]
}
full_advanced_tx_tau1_tau2_45deg_s_725um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_tx_tau1_tau2_45deg_s_725um'
full_advanced_tx_tau1_tau2_45deg_s_725um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_tx_tau1_tau2_45deg_s_725um_result = run_lecture_map_from_spec(full_advanced_tx_tau1_tau2_45deg_s_725um_spec, output_root=full_advanced_tx_tau1_tau2_45deg_s_725um_output_root, profile='full')
full_advanced_tx_tau1_tau2_45deg_s_725um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir = Path(full_advanced_tx_tau1_tau2_45deg_s_725um_result['study_dir'])
full_advanced_tx_tau1_tau2_45deg_s_725um_plot_title = full_advanced_tx_tau1_tau2_45deg_s_725um_result['title']
full_advanced_tx_tau1_tau2_45deg_s_725um_x_label = "$\\tau_1$ (ps)"
full_advanced_tx_tau1_tau2_45deg_s_725um_y_label = "$\\tau_2$ (ps)"

with (full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_tx_tau1_tau2_45deg_s_725um_summary_handle:
    full_advanced_tx_tau1_tau2_45deg_s_725um_summary_meta = json.load(full_advanced_tx_tau1_tau2_45deg_s_725um_summary_handle)
with (full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_tx_tau1_tau2_45deg_s_725um_corr_handle:
    full_advanced_tx_tau1_tau2_45deg_s_725um_corr_meta = json.load(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_handle)

full_advanced_tx_tau1_tau2_45deg_s_725um_rows = []
with (full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_tx_tau1_tau2_45deg_s_725um_summary_csv:
    full_advanced_tx_tau1_tau2_45deg_s_725um_reader = csv.DictReader(full_advanced_tx_tau1_tau2_45deg_s_725um_summary_csv)
    for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_reader:
        full_advanced_tx_tau1_tau2_45deg_s_725um_parsed = {}
        for full_advanced_tx_tau1_tau2_45deg_s_725um_key, full_advanced_tx_tau1_tau2_45deg_s_725um_value in full_advanced_tx_tau1_tau2_45deg_s_725um_row.items():
            if full_advanced_tx_tau1_tau2_45deg_s_725um_value is None:
                full_advanced_tx_tau1_tau2_45deg_s_725um_parsed[full_advanced_tx_tau1_tau2_45deg_s_725um_key] = full_advanced_tx_tau1_tau2_45deg_s_725um_value
                continue
            full_advanced_tx_tau1_tau2_45deg_s_725um_text = str(full_advanced_tx_tau1_tau2_45deg_s_725um_value).strip()
            if full_advanced_tx_tau1_tau2_45deg_s_725um_text == '':
                full_advanced_tx_tau1_tau2_45deg_s_725um_parsed[full_advanced_tx_tau1_tau2_45deg_s_725um_key] = full_advanced_tx_tau1_tau2_45deg_s_725um_text
                continue
            try:
                full_advanced_tx_tau1_tau2_45deg_s_725um_parsed[full_advanced_tx_tau1_tau2_45deg_s_725um_key] = float(full_advanced_tx_tau1_tau2_45deg_s_725um_text)
            except ValueError:
                full_advanced_tx_tau1_tau2_45deg_s_725um_parsed[full_advanced_tx_tau1_tau2_45deg_s_725um_key] = full_advanced_tx_tau1_tau2_45deg_s_725um_text
        full_advanced_tx_tau1_tau2_45deg_s_725um_rows.append(full_advanced_tx_tau1_tau2_45deg_s_725um_parsed)

full_advanced_tx_tau1_tau2_45deg_s_725um_x_key, full_advanced_tx_tau1_tau2_45deg_s_725um_y_key = list(full_advanced_tx_tau1_tau2_45deg_s_725um_summary_meta['recovery_keys'])
full_advanced_tx_tau1_tau2_45deg_s_725um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_tx_tau1_tau2_45deg_s_725um_x_key}_true_minus_fit': f'signed_err_{full_advanced_tx_tau1_tau2_45deg_s_725um_x_key}',
    f'{full_advanced_tx_tau1_tau2_45deg_s_725um_y_key}_true_minus_fit': f'signed_err_{full_advanced_tx_tau1_tau2_45deg_s_725um_y_key}',
    f'abs_{full_advanced_tx_tau1_tau2_45deg_s_725um_x_key}_error': f'abs_err_{full_advanced_tx_tau1_tau2_45deg_s_725um_x_key}',
    f'abs_{full_advanced_tx_tau1_tau2_45deg_s_725um_y_key}_error': f'abs_err_{full_advanced_tx_tau1_tau2_45deg_s_725um_y_key}',
}
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_value_key = full_advanced_tx_tau1_tau2_45deg_s_725um_metric_options['data_fit']
full_advanced_tx_tau1_tau2_45deg_s_725um_log_value_key = full_advanced_tx_tau1_tau2_45deg_s_725um_metric_options['data_fit']
full_advanced_tx_tau1_tau2_45deg_s_725um_x_values = sorted({float(full_advanced_tx_tau1_tau2_45deg_s_725um_row[full_advanced_tx_tau1_tau2_45deg_s_725um_x_key]) for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_rows})
full_advanced_tx_tau1_tau2_45deg_s_725um_y_values = sorted({float(full_advanced_tx_tau1_tau2_45deg_s_725um_row[full_advanced_tx_tau1_tau2_45deg_s_725um_y_key]) for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_rows})

print('Available map metrics:')
for full_advanced_tx_tau1_tau2_45deg_s_725um_option_name, full_advanced_tx_tau1_tau2_45deg_s_725um_option_key in full_advanced_tx_tau1_tau2_45deg_s_725um_metric_options.items():
    print(f"  {full_advanced_tx_tau1_tau2_45deg_s_725um_option_name}: {metric_label(full_advanced_tx_tau1_tau2_45deg_s_725um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_tx_tau1_tau2_45deg_s_725um_log_value_key)}")

def full_advanced_tx_tau1_tau2_45deg_s_725um_aggregate_grid(value_key):
    full_advanced_tx_tau1_tau2_45deg_s_725um_grid = np.full((len(full_advanced_tx_tau1_tau2_45deg_s_725um_y_values), len(full_advanced_tx_tau1_tau2_45deg_s_725um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_tx_tau1_tau2_45deg_s_725um_iy, full_advanced_tx_tau1_tau2_45deg_s_725um_y_value in enumerate(full_advanced_tx_tau1_tau2_45deg_s_725um_y_values):
        for full_advanced_tx_tau1_tau2_45deg_s_725um_ix, full_advanced_tx_tau1_tau2_45deg_s_725um_x_value in enumerate(full_advanced_tx_tau1_tau2_45deg_s_725um_x_values):
            full_advanced_tx_tau1_tau2_45deg_s_725um_cell = [
                float(full_advanced_tx_tau1_tau2_45deg_s_725um_row[value_key])
                for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_rows
                if math.isclose(float(full_advanced_tx_tau1_tau2_45deg_s_725um_row[full_advanced_tx_tau1_tau2_45deg_s_725um_x_key]), full_advanced_tx_tau1_tau2_45deg_s_725um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_tx_tau1_tau2_45deg_s_725um_row[full_advanced_tx_tau1_tau2_45deg_s_725um_y_key]), full_advanced_tx_tau1_tau2_45deg_s_725um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_tx_tau1_tau2_45deg_s_725um_row[value_key]))
            ]
            if full_advanced_tx_tau1_tau2_45deg_s_725um_cell:
                full_advanced_tx_tau1_tau2_45deg_s_725um_grid[full_advanced_tx_tau1_tau2_45deg_s_725um_iy, full_advanced_tx_tau1_tau2_45deg_s_725um_ix] = float(np.mean(full_advanced_tx_tau1_tau2_45deg_s_725um_cell))
    return full_advanced_tx_tau1_tau2_45deg_s_725um_grid

def full_advanced_tx_tau1_tau2_45deg_s_725um_positive_grid_for_log(grid):
    full_advanced_tx_tau1_tau2_45deg_s_725um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_tx_tau1_tau2_45deg_s_725um_finite = full_advanced_tx_tau1_tau2_45deg_s_725um_grid[np.isfinite(full_advanced_tx_tau1_tau2_45deg_s_725um_grid)]
    full_advanced_tx_tau1_tau2_45deg_s_725um_positive = full_advanced_tx_tau1_tau2_45deg_s_725um_finite[full_advanced_tx_tau1_tau2_45deg_s_725um_finite > 0.0]
    if full_advanced_tx_tau1_tau2_45deg_s_725um_positive.size == 0:
        full_advanced_tx_tau1_tau2_45deg_s_725um_grid[np.isfinite(full_advanced_tx_tau1_tau2_45deg_s_725um_grid)] = 1.0
        return full_advanced_tx_tau1_tau2_45deg_s_725um_grid
    full_advanced_tx_tau1_tau2_45deg_s_725um_floor = max(float(np.min(full_advanced_tx_tau1_tau2_45deg_s_725um_positive)) * 0.5, 1e-18)
    full_advanced_tx_tau1_tau2_45deg_s_725um_grid[np.isfinite(full_advanced_tx_tau1_tau2_45deg_s_725um_grid) & (full_advanced_tx_tau1_tau2_45deg_s_725um_grid <= 0.0)] = full_advanced_tx_tau1_tau2_45deg_s_725um_floor
    return full_advanced_tx_tau1_tau2_45deg_s_725um_grid

full_advanced_tx_tau1_tau2_45deg_s_725um_linear_grid = full_advanced_tx_tau1_tau2_45deg_s_725um_aggregate_grid(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_value_key)
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig, full_advanced_tx_tau1_tau2_45deg_s_725um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_finite = full_advanced_tx_tau1_tau2_45deg_s_725um_linear_grid[np.isfinite(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_grid)]
if str(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_value_key).startswith('signed_err_'):
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_limit = max(float(np.max(np.abs(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_finite))), 1e-18)
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmin = -full_advanced_tx_tau1_tau2_45deg_s_725um_linear_limit
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmax = full_advanced_tx_tau1_tau2_45deg_s_725um_linear_limit
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_levels = np.linspace(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmin, full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmax, 256)
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_cmap = 'plasma'
else:
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmin = float(np.min(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_finite))
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmax = float(np.max(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_finite)) + 1e-12
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_levels = np.linspace(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmin, full_advanced_tx_tau1_tau2_45deg_s_725um_linear_vmax, 256)
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_cmap = 'plasma'
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_contour = full_advanced_tx_tau1_tau2_45deg_s_725um_linear_ax.contourf(
    np.asarray(full_advanced_tx_tau1_tau2_45deg_s_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_tau1_tau2_45deg_s_725um_y_values, dtype=np.float64),
    full_advanced_tx_tau1_tau2_45deg_s_725um_linear_grid,
    levels=full_advanced_tx_tau1_tau2_45deg_s_725um_linear_levels,
    cmap=full_advanced_tx_tau1_tau2_45deg_s_725um_linear_cmap,
    extend='both',
)
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_ax.set_title(full_advanced_tx_tau1_tau2_45deg_s_725um_plot_title + ' [linear]')
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_ax.set_xlabel(full_advanced_tx_tau1_tau2_45deg_s_725um_x_label)
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_ax.set_ylabel(full_advanced_tx_tau1_tau2_45deg_s_725um_y_label)
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_cbar = full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig.colorbar(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_contour, ax=full_advanced_tx_tau1_tau2_45deg_s_725um_linear_ax)
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_cbar.set_label(metric_label(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_value_key))
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig.tight_layout()
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_png = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_linear.png"
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_pdf = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_linear.pdf"
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_png, dpi=220)
full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_pdf)

if str(full_advanced_tx_tau1_tau2_45deg_s_725um_log_value_key).startswith('signed_err_'):
    full_advanced_tx_tau1_tau2_45deg_s_725um_log_source_key = 'abs_err_' + str(full_advanced_tx_tau1_tau2_45deg_s_725um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_tx_tau1_tau2_45deg_s_725um_log_source_key = full_advanced_tx_tau1_tau2_45deg_s_725um_log_value_key
full_advanced_tx_tau1_tau2_45deg_s_725um_log_grid = full_advanced_tx_tau1_tau2_45deg_s_725um_positive_grid_for_log(full_advanced_tx_tau1_tau2_45deg_s_725um_aggregate_grid(full_advanced_tx_tau1_tau2_45deg_s_725um_log_source_key))
full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig, full_advanced_tx_tau1_tau2_45deg_s_725um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_tau1_tau2_45deg_s_725um_log_finite = full_advanced_tx_tau1_tau2_45deg_s_725um_log_grid[np.isfinite(full_advanced_tx_tau1_tau2_45deg_s_725um_log_grid)]
full_advanced_tx_tau1_tau2_45deg_s_725um_log_positive = full_advanced_tx_tau1_tau2_45deg_s_725um_log_finite[full_advanced_tx_tau1_tau2_45deg_s_725um_log_finite > 0.0]
full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmin = float(np.min(full_advanced_tx_tau1_tau2_45deg_s_725um_log_positive))
full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmax = float(np.max(full_advanced_tx_tau1_tau2_45deg_s_725um_log_positive))
if math.isclose(full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmin, full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmax):
    full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmax = full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmin * 1.01
full_advanced_tx_tau1_tau2_45deg_s_725um_log_levels = np.geomspace(full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmin, full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmax, 256)
full_advanced_tx_tau1_tau2_45deg_s_725um_log_contour = full_advanced_tx_tau1_tau2_45deg_s_725um_log_ax.contourf(
    np.asarray(full_advanced_tx_tau1_tau2_45deg_s_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_tau1_tau2_45deg_s_725um_y_values, dtype=np.float64),
    full_advanced_tx_tau1_tau2_45deg_s_725um_log_grid,
    levels=full_advanced_tx_tau1_tau2_45deg_s_725um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmin, vmax=full_advanced_tx_tau1_tau2_45deg_s_725um_log_vmax),
    extend='both',
)
full_advanced_tx_tau1_tau2_45deg_s_725um_log_ax.set_title(full_advanced_tx_tau1_tau2_45deg_s_725um_plot_title + ' [log]')
full_advanced_tx_tau1_tau2_45deg_s_725um_log_ax.set_xlabel(full_advanced_tx_tau1_tau2_45deg_s_725um_x_label)
full_advanced_tx_tau1_tau2_45deg_s_725um_log_ax.set_ylabel(full_advanced_tx_tau1_tau2_45deg_s_725um_y_label)
full_advanced_tx_tau1_tau2_45deg_s_725um_log_cbar = full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig.colorbar(full_advanced_tx_tau1_tau2_45deg_s_725um_log_contour, ax=full_advanced_tx_tau1_tau2_45deg_s_725um_log_ax)
full_advanced_tx_tau1_tau2_45deg_s_725um_log_cbar.set_label(metric_label(full_advanced_tx_tau1_tau2_45deg_s_725um_log_source_key))
full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig.tight_layout()
full_advanced_tx_tau1_tau2_45deg_s_725um_log_png = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_log.png"
full_advanced_tx_tau1_tau2_45deg_s_725um_log_pdf = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_log.pdf"
full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_log_png, dpi=220)
full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_log_pdf)

full_advanced_tx_tau1_tau2_45deg_s_725um_corr_rows = full_advanced_tx_tau1_tau2_45deg_s_725um_corr_meta['rows']
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels = sorted({str(full_advanced_tx_tau1_tau2_45deg_s_725um_row['param_i']) for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_corr_rows} | {str(full_advanced_tx_tau1_tau2_45deg_s_725um_row['param_j']) for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_corr_rows})
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_index = {full_advanced_tx_tau1_tau2_45deg_s_725um_label: full_advanced_tx_tau1_tau2_45deg_s_725um_idx for full_advanced_tx_tau1_tau2_45deg_s_725um_idx, full_advanced_tx_tau1_tau2_45deg_s_725um_label in enumerate(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels)}
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_matrix = np.full((len(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels), len(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_tx_tau1_tau2_45deg_s_725um_row in full_advanced_tx_tau1_tau2_45deg_s_725um_corr_rows:
    full_advanced_tx_tau1_tau2_45deg_s_725um_i = full_advanced_tx_tau1_tau2_45deg_s_725um_corr_index[str(full_advanced_tx_tau1_tau2_45deg_s_725um_row['param_i'])]
    full_advanced_tx_tau1_tau2_45deg_s_725um_j = full_advanced_tx_tau1_tau2_45deg_s_725um_corr_index[str(full_advanced_tx_tau1_tau2_45deg_s_725um_row['param_j'])]
    full_advanced_tx_tau1_tau2_45deg_s_725um_corr_matrix[full_advanced_tx_tau1_tau2_45deg_s_725um_i, full_advanced_tx_tau1_tau2_45deg_s_725um_j] = float(full_advanced_tx_tau1_tau2_45deg_s_725um_row['correlation'])
    full_advanced_tx_tau1_tau2_45deg_s_725um_corr_matrix[full_advanced_tx_tau1_tau2_45deg_s_725um_j, full_advanced_tx_tau1_tau2_45deg_s_725um_i] = float(full_advanced_tx_tau1_tau2_45deg_s_725um_row['correlation'])
for full_advanced_tx_tau1_tau2_45deg_s_725um_diag in range(len(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels)):
    full_advanced_tx_tau1_tau2_45deg_s_725um_corr_matrix[full_advanced_tx_tau1_tau2_45deg_s_725um_diag, full_advanced_tx_tau1_tau2_45deg_s_725um_diag] = 1.0

full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig, full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_image = full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax.imshow(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax.set_title(full_advanced_tx_tau1_tau2_45deg_s_725um_plot_title + ' [correlation]')
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax.set_xticks(range(len(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels)))
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax.set_yticks(range(len(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels)))
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_tx_tau1_tau2_45deg_s_725um_label, full_advanced_tx_tau1_tau2_45deg_s_725um_label) for full_advanced_tx_tau1_tau2_45deg_s_725um_label in full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels], rotation=45, ha='right')
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_tx_tau1_tau2_45deg_s_725um_label, full_advanced_tx_tau1_tau2_45deg_s_725um_label) for full_advanced_tx_tau1_tau2_45deg_s_725um_label in full_advanced_tx_tau1_tau2_45deg_s_725um_corr_labels])
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig.colorbar(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_image, ax=full_advanced_tx_tau1_tau2_45deg_s_725um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig.tight_layout()
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_png = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_corr.png"
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_pdf = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_corr.pdf"
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_png, dpi=220)
full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_pdf)

full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig, full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_tx_tau1_tau2_45deg_s_725um_axis, full_advanced_tx_tau1_tau2_45deg_s_725um_image_path, full_advanced_tx_tau1_tau2_45deg_s_725um_panel_title in zip(
    full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_axes,
    (full_advanced_tx_tau1_tau2_45deg_s_725um_linear_png, full_advanced_tx_tau1_tau2_45deg_s_725um_log_png, full_advanced_tx_tau1_tau2_45deg_s_725um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_tx_tau1_tau2_45deg_s_725um_image = plt.imread(str(full_advanced_tx_tau1_tau2_45deg_s_725um_image_path))
    full_advanced_tx_tau1_tau2_45deg_s_725um_axis.imshow(full_advanced_tx_tau1_tau2_45deg_s_725um_image)
    full_advanced_tx_tau1_tau2_45deg_s_725um_axis.set_title(full_advanced_tx_tau1_tau2_45deg_s_725um_panel_title)
    full_advanced_tx_tau1_tau2_45deg_s_725um_axis.axis('off')
full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig.suptitle(full_advanced_tx_tau1_tau2_45deg_s_725um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig.tight_layout()
full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_png = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_triptych.png"
full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_pdf = full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir / f"{full_advanced_tx_tau1_tau2_45deg_s_725um_study_dir.name}__section_triptych.pdf"
full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_png, dpi=220)
full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig.savefig(full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_pdf)

display(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig)
display(full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig)
display(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig)
display(full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig)
plt.close(full_advanced_tx_tau1_tau2_45deg_s_725um_linear_fig)
plt.close(full_advanced_tx_tau1_tau2_45deg_s_725um_log_fig)
plt.close(full_advanced_tx_tau1_tau2_45deg_s_725um_corr_fig)
plt.close(full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_fig)

full_advanced_tx_tau1_tau2_45deg_s_725um_plot_paths = {
    'linear_png': full_advanced_tx_tau1_tau2_45deg_s_725um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_tx_tau1_tau2_45deg_s_725um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_tx_tau1_tau2_45deg_s_725um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_tx_tau1_tau2_45deg_s_725um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_tx_tau1_tau2_45deg_s_725um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_tx_tau1_tau2_45deg_s_725um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_tx_tau1_tau2_45deg_s_725um_triptych_pdf.resolve().as_posix(),
}
full_advanced_tx_tau1_tau2_45deg_s_725um_plot_paths

### Two-Drude transmission: $\sigma_1$ vs $\sigma_2$ (45 deg, s-pol, $d_{\mathrm{sub}}=725\,\mu$m, $d_{\mathrm{epi}}=50\,\mu$m)

In [ ]:
full_advanced_tx_sigma1_sigma2_45deg_s_725um_spec = {
  "slug": "advanced_tx_sigma1_sigma2_45deg_s_725um",
  "title": "Two-Drude transmission: $\\sigma_1$ vs $\\sigma_2$ (45 deg, s-pol, $d_{\\mathrm{sub}}=725\\,\\mu$m, $d_{\\mathrm{epi}}=50\\,\\mu$m)",
  "mode": "transmission",
  "angle_deg": 45.0,
  "polarization": "s",
  "map_kind": "sigma",
  "substrate_thickness_um": 725.0,
  "epi_thickness_um": 50.0,
  "sigma_range": [
    100.0,
    1000.0
  ],
  "tau_fixed": 0.316,
  "x_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ],
  "y_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ]
}
full_advanced_tx_sigma1_sigma2_45deg_s_725um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_tx_sigma1_sigma2_45deg_s_725um'
full_advanced_tx_sigma1_sigma2_45deg_s_725um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_result = run_lecture_map_from_spec(full_advanced_tx_sigma1_sigma2_45deg_s_725um_spec, output_root=full_advanced_tx_sigma1_sigma2_45deg_s_725um_output_root, profile='full')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir = Path(full_advanced_tx_sigma1_sigma2_45deg_s_725um_result['study_dir'])
full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_title = full_advanced_tx_sigma1_sigma2_45deg_s_725um_result['title']
full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_label = "$\\sigma_1$ (S/m)"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_label = "$\\sigma_2$ (S/m)"

with (full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_tx_sigma1_sigma2_45deg_s_725um_summary_handle:
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_summary_meta = json.load(full_advanced_tx_sigma1_sigma2_45deg_s_725um_summary_handle)
with (full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_handle:
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_meta = json.load(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_handle)

full_advanced_tx_sigma1_sigma2_45deg_s_725um_rows = []
with (full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_tx_sigma1_sigma2_45deg_s_725um_summary_csv:
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_reader = csv.DictReader(full_advanced_tx_sigma1_sigma2_45deg_s_725um_summary_csv)
    for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_reader:
        full_advanced_tx_sigma1_sigma2_45deg_s_725um_parsed = {}
        for full_advanced_tx_sigma1_sigma2_45deg_s_725um_key, full_advanced_tx_sigma1_sigma2_45deg_s_725um_value in full_advanced_tx_sigma1_sigma2_45deg_s_725um_row.items():
            if full_advanced_tx_sigma1_sigma2_45deg_s_725um_value is None:
                full_advanced_tx_sigma1_sigma2_45deg_s_725um_parsed[full_advanced_tx_sigma1_sigma2_45deg_s_725um_key] = full_advanced_tx_sigma1_sigma2_45deg_s_725um_value
                continue
            full_advanced_tx_sigma1_sigma2_45deg_s_725um_text = str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_value).strip()
            if full_advanced_tx_sigma1_sigma2_45deg_s_725um_text == '':
                full_advanced_tx_sigma1_sigma2_45deg_s_725um_parsed[full_advanced_tx_sigma1_sigma2_45deg_s_725um_key] = full_advanced_tx_sigma1_sigma2_45deg_s_725um_text
                continue
            try:
                full_advanced_tx_sigma1_sigma2_45deg_s_725um_parsed[full_advanced_tx_sigma1_sigma2_45deg_s_725um_key] = float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_text)
            except ValueError:
                full_advanced_tx_sigma1_sigma2_45deg_s_725um_parsed[full_advanced_tx_sigma1_sigma2_45deg_s_725um_key] = full_advanced_tx_sigma1_sigma2_45deg_s_725um_text
        full_advanced_tx_sigma1_sigma2_45deg_s_725um_rows.append(full_advanced_tx_sigma1_sigma2_45deg_s_725um_parsed)

full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key, full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key = list(full_advanced_tx_sigma1_sigma2_45deg_s_725um_summary_meta['recovery_keys'])
full_advanced_tx_sigma1_sigma2_45deg_s_725um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key}_true_minus_fit': f'signed_err_{full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key}',
    f'{full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key}_true_minus_fit': f'signed_err_{full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key}',
    f'abs_{full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key}_error': f'abs_err_{full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key}',
    f'abs_{full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key}_error': f'abs_err_{full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key}',
}
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_value_key = full_advanced_tx_sigma1_sigma2_45deg_s_725um_metric_options['data_fit']
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_value_key = full_advanced_tx_sigma1_sigma2_45deg_s_725um_metric_options['data_fit']
full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_values = sorted({float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row[full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key]) for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_rows})
full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_values = sorted({float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row[full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key]) for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_rows})

print('Available map metrics:')
for full_advanced_tx_sigma1_sigma2_45deg_s_725um_option_name, full_advanced_tx_sigma1_sigma2_45deg_s_725um_option_key in full_advanced_tx_sigma1_sigma2_45deg_s_725um_metric_options.items():
    print(f"  {full_advanced_tx_sigma1_sigma2_45deg_s_725um_option_name}: {metric_label(full_advanced_tx_sigma1_sigma2_45deg_s_725um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_value_key)}")

def full_advanced_tx_sigma1_sigma2_45deg_s_725um_aggregate_grid(value_key):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid = np.full((len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_values), len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_tx_sigma1_sigma2_45deg_s_725um_iy, full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_value in enumerate(full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_values):
        for full_advanced_tx_sigma1_sigma2_45deg_s_725um_ix, full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_value in enumerate(full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_values):
            full_advanced_tx_sigma1_sigma2_45deg_s_725um_cell = [
                float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row[value_key])
                for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_rows
                if math.isclose(float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row[full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_key]), full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row[full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_key]), full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row[value_key]))
            ]
            if full_advanced_tx_sigma1_sigma2_45deg_s_725um_cell:
                full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid[full_advanced_tx_sigma1_sigma2_45deg_s_725um_iy, full_advanced_tx_sigma1_sigma2_45deg_s_725um_ix] = float(np.mean(full_advanced_tx_sigma1_sigma2_45deg_s_725um_cell))
    return full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid

def full_advanced_tx_sigma1_sigma2_45deg_s_725um_positive_grid_for_log(grid):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_finite = full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid)]
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_positive = full_advanced_tx_sigma1_sigma2_45deg_s_725um_finite[full_advanced_tx_sigma1_sigma2_45deg_s_725um_finite > 0.0]
    if full_advanced_tx_sigma1_sigma2_45deg_s_725um_positive.size == 0:
        full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid)] = 1.0
        return full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_floor = max(float(np.min(full_advanced_tx_sigma1_sigma2_45deg_s_725um_positive)) * 0.5, 1e-18)
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid) & (full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid <= 0.0)] = full_advanced_tx_sigma1_sigma2_45deg_s_725um_floor
    return full_advanced_tx_sigma1_sigma2_45deg_s_725um_grid

full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_grid = full_advanced_tx_sigma1_sigma2_45deg_s_725um_aggregate_grid(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_value_key)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig, full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_finite = full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_grid)]
if str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_value_key).startswith('signed_err_'):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_limit = max(float(np.max(np.abs(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_finite))), 1e-18)
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmin = -full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_limit
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmax = full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_limit
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_levels = np.linspace(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmin, full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmax, 256)
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_cmap = 'plasma'
else:
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmin = float(np.min(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_finite))
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmax = float(np.max(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_finite)) + 1e-12
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_levels = np.linspace(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmin, full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_vmax, 256)
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_cmap = 'plasma'
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_contour = full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_ax.contourf(
    np.asarray(full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_values, dtype=np.float64),
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_grid,
    levels=full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_levels,
    cmap=full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_cmap,
    extend='both',
)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_ax.set_title(full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_title + ' [linear]')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_ax.set_xlabel(full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_label)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_ax.set_ylabel(full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_label)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_cbar = full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig.colorbar(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_contour, ax=full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_ax)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_cbar.set_label(metric_label(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_value_key))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_png = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_linear.png"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_pdf = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_linear.pdf"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_png, dpi=220)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_pdf)

if str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_value_key).startswith('signed_err_'):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_source_key = 'abs_err_' + str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_source_key = full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_value_key
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_grid = full_advanced_tx_sigma1_sigma2_45deg_s_725um_positive_grid_for_log(full_advanced_tx_sigma1_sigma2_45deg_s_725um_aggregate_grid(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_source_key))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig, full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_finite = full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_grid[np.isfinite(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_grid)]
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_positive = full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_finite[full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_finite > 0.0]
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmin = float(np.min(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_positive))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmax = float(np.max(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_positive))
if math.isclose(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmin, full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmax):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmax = full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmin * 1.01
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_levels = np.geomspace(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmin, full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmax, 256)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_contour = full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_ax.contourf(
    np.asarray(full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_values, dtype=np.float64),
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_grid,
    levels=full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmin, vmax=full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_vmax),
    extend='both',
)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_ax.set_title(full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_title + ' [log]')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_ax.set_xlabel(full_advanced_tx_sigma1_sigma2_45deg_s_725um_x_label)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_ax.set_ylabel(full_advanced_tx_sigma1_sigma2_45deg_s_725um_y_label)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_cbar = full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig.colorbar(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_contour, ax=full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_ax)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_cbar.set_label(metric_label(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_source_key))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_png = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_log.png"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_pdf = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_log.pdf"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_png, dpi=220)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_pdf)

full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_rows = full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_meta['rows']
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels = sorted({str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row['param_i']) for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_rows} | {str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row['param_j']) for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_rows})
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_index = {full_advanced_tx_sigma1_sigma2_45deg_s_725um_label: full_advanced_tx_sigma1_sigma2_45deg_s_725um_idx for full_advanced_tx_sigma1_sigma2_45deg_s_725um_idx, full_advanced_tx_sigma1_sigma2_45deg_s_725um_label in enumerate(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels)}
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_matrix = np.full((len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels), len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_tx_sigma1_sigma2_45deg_s_725um_row in full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_rows:
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_i = full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_index[str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row['param_i'])]
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_j = full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_index[str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row['param_j'])]
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_matrix[full_advanced_tx_sigma1_sigma2_45deg_s_725um_i, full_advanced_tx_sigma1_sigma2_45deg_s_725um_j] = float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row['correlation'])
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_matrix[full_advanced_tx_sigma1_sigma2_45deg_s_725um_j, full_advanced_tx_sigma1_sigma2_45deg_s_725um_i] = float(full_advanced_tx_sigma1_sigma2_45deg_s_725um_row['correlation'])
for full_advanced_tx_sigma1_sigma2_45deg_s_725um_diag in range(len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels)):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_matrix[full_advanced_tx_sigma1_sigma2_45deg_s_725um_diag, full_advanced_tx_sigma1_sigma2_45deg_s_725um_diag] = 1.0

full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig, full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_image = full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax.imshow(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax.set_title(full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_title + ' [correlation]')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax.set_xticks(range(len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels)))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax.set_yticks(range(len(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels)))
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_tx_sigma1_sigma2_45deg_s_725um_label, full_advanced_tx_sigma1_sigma2_45deg_s_725um_label) for full_advanced_tx_sigma1_sigma2_45deg_s_725um_label in full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels], rotation=45, ha='right')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_tx_sigma1_sigma2_45deg_s_725um_label, full_advanced_tx_sigma1_sigma2_45deg_s_725um_label) for full_advanced_tx_sigma1_sigma2_45deg_s_725um_label in full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_labels])
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig.colorbar(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_image, ax=full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_png = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_corr.png"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_pdf = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_corr.pdf"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_png, dpi=220)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_pdf)

full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig, full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_tx_sigma1_sigma2_45deg_s_725um_axis, full_advanced_tx_sigma1_sigma2_45deg_s_725um_image_path, full_advanced_tx_sigma1_sigma2_45deg_s_725um_panel_title in zip(
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_axes,
    (full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_png, full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_png, full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_image = plt.imread(str(full_advanced_tx_sigma1_sigma2_45deg_s_725um_image_path))
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_axis.imshow(full_advanced_tx_sigma1_sigma2_45deg_s_725um_image)
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_axis.set_title(full_advanced_tx_sigma1_sigma2_45deg_s_725um_panel_title)
    full_advanced_tx_sigma1_sigma2_45deg_s_725um_axis.axis('off')
full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig.suptitle(full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig.tight_layout()
full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_png = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_triptych.png"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_pdf = full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir / f"{full_advanced_tx_sigma1_sigma2_45deg_s_725um_study_dir.name}__section_triptych.pdf"
full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_png, dpi=220)
full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig.savefig(full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_pdf)

display(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig)
display(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig)
display(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig)
display(full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig)
plt.close(full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_fig)
plt.close(full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_fig)
plt.close(full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_fig)
plt.close(full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_fig)

full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_paths = {
    'linear_png': full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_tx_sigma1_sigma2_45deg_s_725um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_tx_sigma1_sigma2_45deg_s_725um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_tx_sigma1_sigma2_45deg_s_725um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_tx_sigma1_sigma2_45deg_s_725um_triptych_pdf.resolve().as_posix(),
}
full_advanced_tx_sigma1_sigma2_45deg_s_725um_plot_paths

### Two-Drude reflection: $\tau_1$ vs $\tau_2$ (45 deg, s-pol, $d_{\mathrm{sub}}=525\,\mu$m, $d_{\mathrm{epi}}=10\,\mu$m)

In [ ]:
full_advanced_refl_tau1_tau2_45deg_s_525um_spec = {
  "slug": "advanced_refl_tau1_tau2_45deg_s_525um",
  "title": "Two-Drude reflection: $\\tau_1$ vs $\\tau_2$ (45 deg, s-pol, $d_{\\mathrm{sub}}=525\\,\\mu$m, $d_{\\mathrm{epi}}=10\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 45.0,
  "polarization": "s",
  "map_kind": "tau",
  "substrate_thickness_um": 525.0,
  "epi_thickness_um": 10.0,
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_fixed": 0.1,
  "x_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "y_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ]
}
full_advanced_refl_tau1_tau2_45deg_s_525um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_refl_tau1_tau2_45deg_s_525um'
full_advanced_refl_tau1_tau2_45deg_s_525um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_refl_tau1_tau2_45deg_s_525um_result = run_lecture_map_from_spec(full_advanced_refl_tau1_tau2_45deg_s_525um_spec, output_root=full_advanced_refl_tau1_tau2_45deg_s_525um_output_root, profile='full')
full_advanced_refl_tau1_tau2_45deg_s_525um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir = Path(full_advanced_refl_tau1_tau2_45deg_s_525um_result['study_dir'])
full_advanced_refl_tau1_tau2_45deg_s_525um_plot_title = full_advanced_refl_tau1_tau2_45deg_s_525um_result['title']
full_advanced_refl_tau1_tau2_45deg_s_525um_x_label = "$\\tau_1$ (ps)"
full_advanced_refl_tau1_tau2_45deg_s_525um_y_label = "$\\tau_2$ (ps)"

with (full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_refl_tau1_tau2_45deg_s_525um_summary_handle:
    full_advanced_refl_tau1_tau2_45deg_s_525um_summary_meta = json.load(full_advanced_refl_tau1_tau2_45deg_s_525um_summary_handle)
with (full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_refl_tau1_tau2_45deg_s_525um_corr_handle:
    full_advanced_refl_tau1_tau2_45deg_s_525um_corr_meta = json.load(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_handle)

full_advanced_refl_tau1_tau2_45deg_s_525um_rows = []
with (full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_refl_tau1_tau2_45deg_s_525um_summary_csv:
    full_advanced_refl_tau1_tau2_45deg_s_525um_reader = csv.DictReader(full_advanced_refl_tau1_tau2_45deg_s_525um_summary_csv)
    for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_reader:
        full_advanced_refl_tau1_tau2_45deg_s_525um_parsed = {}
        for full_advanced_refl_tau1_tau2_45deg_s_525um_key, full_advanced_refl_tau1_tau2_45deg_s_525um_value in full_advanced_refl_tau1_tau2_45deg_s_525um_row.items():
            if full_advanced_refl_tau1_tau2_45deg_s_525um_value is None:
                full_advanced_refl_tau1_tau2_45deg_s_525um_parsed[full_advanced_refl_tau1_tau2_45deg_s_525um_key] = full_advanced_refl_tau1_tau2_45deg_s_525um_value
                continue
            full_advanced_refl_tau1_tau2_45deg_s_525um_text = str(full_advanced_refl_tau1_tau2_45deg_s_525um_value).strip()
            if full_advanced_refl_tau1_tau2_45deg_s_525um_text == '':
                full_advanced_refl_tau1_tau2_45deg_s_525um_parsed[full_advanced_refl_tau1_tau2_45deg_s_525um_key] = full_advanced_refl_tau1_tau2_45deg_s_525um_text
                continue
            try:
                full_advanced_refl_tau1_tau2_45deg_s_525um_parsed[full_advanced_refl_tau1_tau2_45deg_s_525um_key] = float(full_advanced_refl_tau1_tau2_45deg_s_525um_text)
            except ValueError:
                full_advanced_refl_tau1_tau2_45deg_s_525um_parsed[full_advanced_refl_tau1_tau2_45deg_s_525um_key] = full_advanced_refl_tau1_tau2_45deg_s_525um_text
        full_advanced_refl_tau1_tau2_45deg_s_525um_rows.append(full_advanced_refl_tau1_tau2_45deg_s_525um_parsed)

full_advanced_refl_tau1_tau2_45deg_s_525um_x_key, full_advanced_refl_tau1_tau2_45deg_s_525um_y_key = list(full_advanced_refl_tau1_tau2_45deg_s_525um_summary_meta['recovery_keys'])
full_advanced_refl_tau1_tau2_45deg_s_525um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_refl_tau1_tau2_45deg_s_525um_x_key}_true_minus_fit': f'signed_err_{full_advanced_refl_tau1_tau2_45deg_s_525um_x_key}',
    f'{full_advanced_refl_tau1_tau2_45deg_s_525um_y_key}_true_minus_fit': f'signed_err_{full_advanced_refl_tau1_tau2_45deg_s_525um_y_key}',
    f'abs_{full_advanced_refl_tau1_tau2_45deg_s_525um_x_key}_error': f'abs_err_{full_advanced_refl_tau1_tau2_45deg_s_525um_x_key}',
    f'abs_{full_advanced_refl_tau1_tau2_45deg_s_525um_y_key}_error': f'abs_err_{full_advanced_refl_tau1_tau2_45deg_s_525um_y_key}',
}
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_value_key = full_advanced_refl_tau1_tau2_45deg_s_525um_metric_options['data_fit']
full_advanced_refl_tau1_tau2_45deg_s_525um_log_value_key = full_advanced_refl_tau1_tau2_45deg_s_525um_metric_options['data_fit']
full_advanced_refl_tau1_tau2_45deg_s_525um_x_values = sorted({float(full_advanced_refl_tau1_tau2_45deg_s_525um_row[full_advanced_refl_tau1_tau2_45deg_s_525um_x_key]) for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_rows})
full_advanced_refl_tau1_tau2_45deg_s_525um_y_values = sorted({float(full_advanced_refl_tau1_tau2_45deg_s_525um_row[full_advanced_refl_tau1_tau2_45deg_s_525um_y_key]) for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_rows})

print('Available map metrics:')
for full_advanced_refl_tau1_tau2_45deg_s_525um_option_name, full_advanced_refl_tau1_tau2_45deg_s_525um_option_key in full_advanced_refl_tau1_tau2_45deg_s_525um_metric_options.items():
    print(f"  {full_advanced_refl_tau1_tau2_45deg_s_525um_option_name}: {metric_label(full_advanced_refl_tau1_tau2_45deg_s_525um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_refl_tau1_tau2_45deg_s_525um_log_value_key)}")

def full_advanced_refl_tau1_tau2_45deg_s_525um_aggregate_grid(value_key):
    full_advanced_refl_tau1_tau2_45deg_s_525um_grid = np.full((len(full_advanced_refl_tau1_tau2_45deg_s_525um_y_values), len(full_advanced_refl_tau1_tau2_45deg_s_525um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_refl_tau1_tau2_45deg_s_525um_iy, full_advanced_refl_tau1_tau2_45deg_s_525um_y_value in enumerate(full_advanced_refl_tau1_tau2_45deg_s_525um_y_values):
        for full_advanced_refl_tau1_tau2_45deg_s_525um_ix, full_advanced_refl_tau1_tau2_45deg_s_525um_x_value in enumerate(full_advanced_refl_tau1_tau2_45deg_s_525um_x_values):
            full_advanced_refl_tau1_tau2_45deg_s_525um_cell = [
                float(full_advanced_refl_tau1_tau2_45deg_s_525um_row[value_key])
                for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_rows
                if math.isclose(float(full_advanced_refl_tau1_tau2_45deg_s_525um_row[full_advanced_refl_tau1_tau2_45deg_s_525um_x_key]), full_advanced_refl_tau1_tau2_45deg_s_525um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_refl_tau1_tau2_45deg_s_525um_row[full_advanced_refl_tau1_tau2_45deg_s_525um_y_key]), full_advanced_refl_tau1_tau2_45deg_s_525um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_refl_tau1_tau2_45deg_s_525um_row[value_key]))
            ]
            if full_advanced_refl_tau1_tau2_45deg_s_525um_cell:
                full_advanced_refl_tau1_tau2_45deg_s_525um_grid[full_advanced_refl_tau1_tau2_45deg_s_525um_iy, full_advanced_refl_tau1_tau2_45deg_s_525um_ix] = float(np.mean(full_advanced_refl_tau1_tau2_45deg_s_525um_cell))
    return full_advanced_refl_tau1_tau2_45deg_s_525um_grid

def full_advanced_refl_tau1_tau2_45deg_s_525um_positive_grid_for_log(grid):
    full_advanced_refl_tau1_tau2_45deg_s_525um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_refl_tau1_tau2_45deg_s_525um_finite = full_advanced_refl_tau1_tau2_45deg_s_525um_grid[np.isfinite(full_advanced_refl_tau1_tau2_45deg_s_525um_grid)]
    full_advanced_refl_tau1_tau2_45deg_s_525um_positive = full_advanced_refl_tau1_tau2_45deg_s_525um_finite[full_advanced_refl_tau1_tau2_45deg_s_525um_finite > 0.0]
    if full_advanced_refl_tau1_tau2_45deg_s_525um_positive.size == 0:
        full_advanced_refl_tau1_tau2_45deg_s_525um_grid[np.isfinite(full_advanced_refl_tau1_tau2_45deg_s_525um_grid)] = 1.0
        return full_advanced_refl_tau1_tau2_45deg_s_525um_grid
    full_advanced_refl_tau1_tau2_45deg_s_525um_floor = max(float(np.min(full_advanced_refl_tau1_tau2_45deg_s_525um_positive)) * 0.5, 1e-18)
    full_advanced_refl_tau1_tau2_45deg_s_525um_grid[np.isfinite(full_advanced_refl_tau1_tau2_45deg_s_525um_grid) & (full_advanced_refl_tau1_tau2_45deg_s_525um_grid <= 0.0)] = full_advanced_refl_tau1_tau2_45deg_s_525um_floor
    return full_advanced_refl_tau1_tau2_45deg_s_525um_grid

full_advanced_refl_tau1_tau2_45deg_s_525um_linear_grid = full_advanced_refl_tau1_tau2_45deg_s_525um_aggregate_grid(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_value_key)
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig, full_advanced_refl_tau1_tau2_45deg_s_525um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_finite = full_advanced_refl_tau1_tau2_45deg_s_525um_linear_grid[np.isfinite(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_grid)]
if str(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_value_key).startswith('signed_err_'):
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_limit = max(float(np.max(np.abs(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_finite))), 1e-18)
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmin = -full_advanced_refl_tau1_tau2_45deg_s_525um_linear_limit
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmax = full_advanced_refl_tau1_tau2_45deg_s_525um_linear_limit
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_levels = np.linspace(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmin, full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmax, 256)
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_cmap = 'plasma'
else:
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmin = float(np.min(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_finite))
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmax = float(np.max(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_finite)) + 1e-12
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_levels = np.linspace(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmin, full_advanced_refl_tau1_tau2_45deg_s_525um_linear_vmax, 256)
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_cmap = 'plasma'
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_contour = full_advanced_refl_tau1_tau2_45deg_s_525um_linear_ax.contourf(
    np.asarray(full_advanced_refl_tau1_tau2_45deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_tau1_tau2_45deg_s_525um_y_values, dtype=np.float64),
    full_advanced_refl_tau1_tau2_45deg_s_525um_linear_grid,
    levels=full_advanced_refl_tau1_tau2_45deg_s_525um_linear_levels,
    cmap=full_advanced_refl_tau1_tau2_45deg_s_525um_linear_cmap,
    extend='both',
)
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_ax.set_title(full_advanced_refl_tau1_tau2_45deg_s_525um_plot_title + ' [linear]')
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_ax.set_xlabel(full_advanced_refl_tau1_tau2_45deg_s_525um_x_label)
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_ax.set_ylabel(full_advanced_refl_tau1_tau2_45deg_s_525um_y_label)
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_cbar = full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig.colorbar(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_contour, ax=full_advanced_refl_tau1_tau2_45deg_s_525um_linear_ax)
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_cbar.set_label(metric_label(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_value_key))
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig.tight_layout()
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_png = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_linear.png"
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_pdf = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_linear.pdf"
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_png, dpi=220)
full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_pdf)

if str(full_advanced_refl_tau1_tau2_45deg_s_525um_log_value_key).startswith('signed_err_'):
    full_advanced_refl_tau1_tau2_45deg_s_525um_log_source_key = 'abs_err_' + str(full_advanced_refl_tau1_tau2_45deg_s_525um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_refl_tau1_tau2_45deg_s_525um_log_source_key = full_advanced_refl_tau1_tau2_45deg_s_525um_log_value_key
full_advanced_refl_tau1_tau2_45deg_s_525um_log_grid = full_advanced_refl_tau1_tau2_45deg_s_525um_positive_grid_for_log(full_advanced_refl_tau1_tau2_45deg_s_525um_aggregate_grid(full_advanced_refl_tau1_tau2_45deg_s_525um_log_source_key))
full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig, full_advanced_refl_tau1_tau2_45deg_s_525um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_tau1_tau2_45deg_s_525um_log_finite = full_advanced_refl_tau1_tau2_45deg_s_525um_log_grid[np.isfinite(full_advanced_refl_tau1_tau2_45deg_s_525um_log_grid)]
full_advanced_refl_tau1_tau2_45deg_s_525um_log_positive = full_advanced_refl_tau1_tau2_45deg_s_525um_log_finite[full_advanced_refl_tau1_tau2_45deg_s_525um_log_finite > 0.0]
full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmin = float(np.min(full_advanced_refl_tau1_tau2_45deg_s_525um_log_positive))
full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmax = float(np.max(full_advanced_refl_tau1_tau2_45deg_s_525um_log_positive))
if math.isclose(full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmin, full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmax):
    full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmax = full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmin * 1.01
full_advanced_refl_tau1_tau2_45deg_s_525um_log_levels = np.geomspace(full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmin, full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmax, 256)
full_advanced_refl_tau1_tau2_45deg_s_525um_log_contour = full_advanced_refl_tau1_tau2_45deg_s_525um_log_ax.contourf(
    np.asarray(full_advanced_refl_tau1_tau2_45deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_tau1_tau2_45deg_s_525um_y_values, dtype=np.float64),
    full_advanced_refl_tau1_tau2_45deg_s_525um_log_grid,
    levels=full_advanced_refl_tau1_tau2_45deg_s_525um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmin, vmax=full_advanced_refl_tau1_tau2_45deg_s_525um_log_vmax),
    extend='both',
)
full_advanced_refl_tau1_tau2_45deg_s_525um_log_ax.set_title(full_advanced_refl_tau1_tau2_45deg_s_525um_plot_title + ' [log]')
full_advanced_refl_tau1_tau2_45deg_s_525um_log_ax.set_xlabel(full_advanced_refl_tau1_tau2_45deg_s_525um_x_label)
full_advanced_refl_tau1_tau2_45deg_s_525um_log_ax.set_ylabel(full_advanced_refl_tau1_tau2_45deg_s_525um_y_label)
full_advanced_refl_tau1_tau2_45deg_s_525um_log_cbar = full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig.colorbar(full_advanced_refl_tau1_tau2_45deg_s_525um_log_contour, ax=full_advanced_refl_tau1_tau2_45deg_s_525um_log_ax)
full_advanced_refl_tau1_tau2_45deg_s_525um_log_cbar.set_label(metric_label(full_advanced_refl_tau1_tau2_45deg_s_525um_log_source_key))
full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig.tight_layout()
full_advanced_refl_tau1_tau2_45deg_s_525um_log_png = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_log.png"
full_advanced_refl_tau1_tau2_45deg_s_525um_log_pdf = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_log.pdf"
full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_log_png, dpi=220)
full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_log_pdf)

full_advanced_refl_tau1_tau2_45deg_s_525um_corr_rows = full_advanced_refl_tau1_tau2_45deg_s_525um_corr_meta['rows']
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels = sorted({str(full_advanced_refl_tau1_tau2_45deg_s_525um_row['param_i']) for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_corr_rows} | {str(full_advanced_refl_tau1_tau2_45deg_s_525um_row['param_j']) for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_corr_rows})
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_index = {full_advanced_refl_tau1_tau2_45deg_s_525um_label: full_advanced_refl_tau1_tau2_45deg_s_525um_idx for full_advanced_refl_tau1_tau2_45deg_s_525um_idx, full_advanced_refl_tau1_tau2_45deg_s_525um_label in enumerate(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels)}
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_matrix = np.full((len(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels), len(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_refl_tau1_tau2_45deg_s_525um_row in full_advanced_refl_tau1_tau2_45deg_s_525um_corr_rows:
    full_advanced_refl_tau1_tau2_45deg_s_525um_i = full_advanced_refl_tau1_tau2_45deg_s_525um_corr_index[str(full_advanced_refl_tau1_tau2_45deg_s_525um_row['param_i'])]
    full_advanced_refl_tau1_tau2_45deg_s_525um_j = full_advanced_refl_tau1_tau2_45deg_s_525um_corr_index[str(full_advanced_refl_tau1_tau2_45deg_s_525um_row['param_j'])]
    full_advanced_refl_tau1_tau2_45deg_s_525um_corr_matrix[full_advanced_refl_tau1_tau2_45deg_s_525um_i, full_advanced_refl_tau1_tau2_45deg_s_525um_j] = float(full_advanced_refl_tau1_tau2_45deg_s_525um_row['correlation'])
    full_advanced_refl_tau1_tau2_45deg_s_525um_corr_matrix[full_advanced_refl_tau1_tau2_45deg_s_525um_j, full_advanced_refl_tau1_tau2_45deg_s_525um_i] = float(full_advanced_refl_tau1_tau2_45deg_s_525um_row['correlation'])
for full_advanced_refl_tau1_tau2_45deg_s_525um_diag in range(len(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels)):
    full_advanced_refl_tau1_tau2_45deg_s_525um_corr_matrix[full_advanced_refl_tau1_tau2_45deg_s_525um_diag, full_advanced_refl_tau1_tau2_45deg_s_525um_diag] = 1.0

full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig, full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_image = full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax.imshow(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax.set_title(full_advanced_refl_tau1_tau2_45deg_s_525um_plot_title + ' [correlation]')
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax.set_xticks(range(len(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels)))
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax.set_yticks(range(len(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels)))
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_refl_tau1_tau2_45deg_s_525um_label, full_advanced_refl_tau1_tau2_45deg_s_525um_label) for full_advanced_refl_tau1_tau2_45deg_s_525um_label in full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels], rotation=45, ha='right')
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_refl_tau1_tau2_45deg_s_525um_label, full_advanced_refl_tau1_tau2_45deg_s_525um_label) for full_advanced_refl_tau1_tau2_45deg_s_525um_label in full_advanced_refl_tau1_tau2_45deg_s_525um_corr_labels])
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig.colorbar(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_image, ax=full_advanced_refl_tau1_tau2_45deg_s_525um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig.tight_layout()
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_png = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_corr.png"
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_pdf = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_corr.pdf"
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_png, dpi=220)
full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_pdf)

full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig, full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_refl_tau1_tau2_45deg_s_525um_axis, full_advanced_refl_tau1_tau2_45deg_s_525um_image_path, full_advanced_refl_tau1_tau2_45deg_s_525um_panel_title in zip(
    full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_axes,
    (full_advanced_refl_tau1_tau2_45deg_s_525um_linear_png, full_advanced_refl_tau1_tau2_45deg_s_525um_log_png, full_advanced_refl_tau1_tau2_45deg_s_525um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_refl_tau1_tau2_45deg_s_525um_image = plt.imread(str(full_advanced_refl_tau1_tau2_45deg_s_525um_image_path))
    full_advanced_refl_tau1_tau2_45deg_s_525um_axis.imshow(full_advanced_refl_tau1_tau2_45deg_s_525um_image)
    full_advanced_refl_tau1_tau2_45deg_s_525um_axis.set_title(full_advanced_refl_tau1_tau2_45deg_s_525um_panel_title)
    full_advanced_refl_tau1_tau2_45deg_s_525um_axis.axis('off')
full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig.suptitle(full_advanced_refl_tau1_tau2_45deg_s_525um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig.tight_layout()
full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_png = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_triptych.png"
full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_pdf = full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir / f"{full_advanced_refl_tau1_tau2_45deg_s_525um_study_dir.name}__section_triptych.pdf"
full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_png, dpi=220)
full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig.savefig(full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_pdf)

display(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig)
display(full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig)
display(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig)
display(full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig)
plt.close(full_advanced_refl_tau1_tau2_45deg_s_525um_linear_fig)
plt.close(full_advanced_refl_tau1_tau2_45deg_s_525um_log_fig)
plt.close(full_advanced_refl_tau1_tau2_45deg_s_525um_corr_fig)
plt.close(full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_fig)

full_advanced_refl_tau1_tau2_45deg_s_525um_plot_paths = {
    'linear_png': full_advanced_refl_tau1_tau2_45deg_s_525um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_refl_tau1_tau2_45deg_s_525um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_refl_tau1_tau2_45deg_s_525um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_refl_tau1_tau2_45deg_s_525um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_refl_tau1_tau2_45deg_s_525um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_refl_tau1_tau2_45deg_s_525um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_refl_tau1_tau2_45deg_s_525um_triptych_pdf.resolve().as_posix(),
}
full_advanced_refl_tau1_tau2_45deg_s_525um_plot_paths

### Two-Drude reflection: $\sigma_1$ vs $\sigma_2$ (45 deg, s-pol, $d_{\mathrm{sub}}=525\,\mu$m, $d_{\mathrm{epi}}=10\,\mu$m)

In [ ]:
full_advanced_refl_sigma1_sigma2_45deg_s_525um_spec = {
  "slug": "advanced_refl_sigma1_sigma2_45deg_s_525um",
  "title": "Two-Drude reflection: $\\sigma_1$ vs $\\sigma_2$ (45 deg, s-pol, $d_{\\mathrm{sub}}=525\\,\\mu$m, $d_{\\mathrm{epi}}=10\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 45.0,
  "polarization": "s",
  "map_kind": "sigma",
  "substrate_thickness_um": 525.0,
  "epi_thickness_um": 10.0,
  "sigma_range": [
    0.01,
    1.0
  ],
  "tau_fixed": 0.316,
  "x_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ],
  "y_values": [
    0.01,
    0.051250000000000004,
    0.0925,
    0.13375,
    0.17500000000000002,
    0.21625000000000003,
    0.2575,
    0.29875,
    0.34,
    0.38125000000000003,
    0.42250000000000004,
    0.46375000000000005,
    0.505,
    0.54625,
    0.5875,
    0.62875,
    0.67,
    0.71125,
    0.7525000000000001,
    0.7937500000000001,
    0.8350000000000001,
    0.8762500000000001,
    0.9175000000000001,
    0.9587500000000001,
    1.0
  ]
}
full_advanced_refl_sigma1_sigma2_45deg_s_525um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_refl_sigma1_sigma2_45deg_s_525um'
full_advanced_refl_sigma1_sigma2_45deg_s_525um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_result = run_lecture_map_from_spec(full_advanced_refl_sigma1_sigma2_45deg_s_525um_spec, output_root=full_advanced_refl_sigma1_sigma2_45deg_s_525um_output_root, profile='full')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir = Path(full_advanced_refl_sigma1_sigma2_45deg_s_525um_result['study_dir'])
full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_title = full_advanced_refl_sigma1_sigma2_45deg_s_525um_result['title']
full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_label = "$\\sigma_1$ (S/m)"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_label = "$\\sigma_2$ (S/m)"

with (full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_refl_sigma1_sigma2_45deg_s_525um_summary_handle:
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_summary_meta = json.load(full_advanced_refl_sigma1_sigma2_45deg_s_525um_summary_handle)
with (full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_handle:
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_meta = json.load(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_handle)

full_advanced_refl_sigma1_sigma2_45deg_s_525um_rows = []
with (full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_refl_sigma1_sigma2_45deg_s_525um_summary_csv:
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_reader = csv.DictReader(full_advanced_refl_sigma1_sigma2_45deg_s_525um_summary_csv)
    for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_reader:
        full_advanced_refl_sigma1_sigma2_45deg_s_525um_parsed = {}
        for full_advanced_refl_sigma1_sigma2_45deg_s_525um_key, full_advanced_refl_sigma1_sigma2_45deg_s_525um_value in full_advanced_refl_sigma1_sigma2_45deg_s_525um_row.items():
            if full_advanced_refl_sigma1_sigma2_45deg_s_525um_value is None:
                full_advanced_refl_sigma1_sigma2_45deg_s_525um_parsed[full_advanced_refl_sigma1_sigma2_45deg_s_525um_key] = full_advanced_refl_sigma1_sigma2_45deg_s_525um_value
                continue
            full_advanced_refl_sigma1_sigma2_45deg_s_525um_text = str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_value).strip()
            if full_advanced_refl_sigma1_sigma2_45deg_s_525um_text == '':
                full_advanced_refl_sigma1_sigma2_45deg_s_525um_parsed[full_advanced_refl_sigma1_sigma2_45deg_s_525um_key] = full_advanced_refl_sigma1_sigma2_45deg_s_525um_text
                continue
            try:
                full_advanced_refl_sigma1_sigma2_45deg_s_525um_parsed[full_advanced_refl_sigma1_sigma2_45deg_s_525um_key] = float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_text)
            except ValueError:
                full_advanced_refl_sigma1_sigma2_45deg_s_525um_parsed[full_advanced_refl_sigma1_sigma2_45deg_s_525um_key] = full_advanced_refl_sigma1_sigma2_45deg_s_525um_text
        full_advanced_refl_sigma1_sigma2_45deg_s_525um_rows.append(full_advanced_refl_sigma1_sigma2_45deg_s_525um_parsed)

full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key, full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key = list(full_advanced_refl_sigma1_sigma2_45deg_s_525um_summary_meta['recovery_keys'])
full_advanced_refl_sigma1_sigma2_45deg_s_525um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key}_true_minus_fit': f'signed_err_{full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key}',
    f'{full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key}_true_minus_fit': f'signed_err_{full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key}',
    f'abs_{full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key}_error': f'abs_err_{full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key}',
    f'abs_{full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key}_error': f'abs_err_{full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key}',
}
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_value_key = full_advanced_refl_sigma1_sigma2_45deg_s_525um_metric_options['data_fit']
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_value_key = full_advanced_refl_sigma1_sigma2_45deg_s_525um_metric_options['data_fit']
full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_values = sorted({float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row[full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key]) for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_rows})
full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_values = sorted({float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row[full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key]) for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_rows})

print('Available map metrics:')
for full_advanced_refl_sigma1_sigma2_45deg_s_525um_option_name, full_advanced_refl_sigma1_sigma2_45deg_s_525um_option_key in full_advanced_refl_sigma1_sigma2_45deg_s_525um_metric_options.items():
    print(f"  {full_advanced_refl_sigma1_sigma2_45deg_s_525um_option_name}: {metric_label(full_advanced_refl_sigma1_sigma2_45deg_s_525um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_value_key)}")

def full_advanced_refl_sigma1_sigma2_45deg_s_525um_aggregate_grid(value_key):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid = np.full((len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_values), len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_refl_sigma1_sigma2_45deg_s_525um_iy, full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_value in enumerate(full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_values):
        for full_advanced_refl_sigma1_sigma2_45deg_s_525um_ix, full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_value in enumerate(full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_values):
            full_advanced_refl_sigma1_sigma2_45deg_s_525um_cell = [
                float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row[value_key])
                for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_rows
                if math.isclose(float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row[full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_key]), full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row[full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_key]), full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row[value_key]))
            ]
            if full_advanced_refl_sigma1_sigma2_45deg_s_525um_cell:
                full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid[full_advanced_refl_sigma1_sigma2_45deg_s_525um_iy, full_advanced_refl_sigma1_sigma2_45deg_s_525um_ix] = float(np.mean(full_advanced_refl_sigma1_sigma2_45deg_s_525um_cell))
    return full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid

def full_advanced_refl_sigma1_sigma2_45deg_s_525um_positive_grid_for_log(grid):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_finite = full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid)]
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_positive = full_advanced_refl_sigma1_sigma2_45deg_s_525um_finite[full_advanced_refl_sigma1_sigma2_45deg_s_525um_finite > 0.0]
    if full_advanced_refl_sigma1_sigma2_45deg_s_525um_positive.size == 0:
        full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid)] = 1.0
        return full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_floor = max(float(np.min(full_advanced_refl_sigma1_sigma2_45deg_s_525um_positive)) * 0.5, 1e-18)
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid) & (full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid <= 0.0)] = full_advanced_refl_sigma1_sigma2_45deg_s_525um_floor
    return full_advanced_refl_sigma1_sigma2_45deg_s_525um_grid

full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_grid = full_advanced_refl_sigma1_sigma2_45deg_s_525um_aggregate_grid(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_value_key)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig, full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_finite = full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_grid)]
if str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_value_key).startswith('signed_err_'):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_limit = max(float(np.max(np.abs(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_finite))), 1e-18)
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmin = -full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_limit
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmax = full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_limit
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_levels = np.linspace(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmin, full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmax, 256)
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_cmap = 'plasma'
else:
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmin = float(np.min(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_finite))
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmax = float(np.max(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_finite)) + 1e-12
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_levels = np.linspace(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmin, full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_vmax, 256)
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_cmap = 'plasma'
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_contour = full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_ax.contourf(
    np.asarray(full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_values, dtype=np.float64),
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_grid,
    levels=full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_levels,
    cmap=full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_cmap,
    extend='both',
)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_ax.set_title(full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_title + ' [linear]')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_ax.set_xlabel(full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_label)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_ax.set_ylabel(full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_label)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_cbar = full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig.colorbar(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_contour, ax=full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_ax)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_cbar.set_label(metric_label(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_value_key))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_png = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_linear.png"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_pdf = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_linear.pdf"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_png, dpi=220)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_pdf)

if str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_value_key).startswith('signed_err_'):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_source_key = 'abs_err_' + str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_source_key = full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_value_key
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_grid = full_advanced_refl_sigma1_sigma2_45deg_s_525um_positive_grid_for_log(full_advanced_refl_sigma1_sigma2_45deg_s_525um_aggregate_grid(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_source_key))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig, full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_finite = full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_grid)]
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_positive = full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_finite[full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_finite > 0.0]
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmin = float(np.min(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_positive))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmax = float(np.max(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_positive))
if math.isclose(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmin, full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmax):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmax = full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmin * 1.01
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_levels = np.geomspace(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmin, full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmax, 256)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_contour = full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_ax.contourf(
    np.asarray(full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_values, dtype=np.float64),
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_grid,
    levels=full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmin, vmax=full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_vmax),
    extend='both',
)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_ax.set_title(full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_title + ' [log]')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_ax.set_xlabel(full_advanced_refl_sigma1_sigma2_45deg_s_525um_x_label)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_ax.set_ylabel(full_advanced_refl_sigma1_sigma2_45deg_s_525um_y_label)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_cbar = full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig.colorbar(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_contour, ax=full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_ax)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_cbar.set_label(metric_label(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_source_key))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_png = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_log.png"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_pdf = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_log.pdf"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_png, dpi=220)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_pdf)

full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_rows = full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_meta['rows']
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels = sorted({str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row['param_i']) for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_rows} | {str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row['param_j']) for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_rows})
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_index = {full_advanced_refl_sigma1_sigma2_45deg_s_525um_label: full_advanced_refl_sigma1_sigma2_45deg_s_525um_idx for full_advanced_refl_sigma1_sigma2_45deg_s_525um_idx, full_advanced_refl_sigma1_sigma2_45deg_s_525um_label in enumerate(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels)}
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_matrix = np.full((len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels), len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_refl_sigma1_sigma2_45deg_s_525um_row in full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_rows:
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_i = full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_index[str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row['param_i'])]
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_j = full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_index[str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row['param_j'])]
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_matrix[full_advanced_refl_sigma1_sigma2_45deg_s_525um_i, full_advanced_refl_sigma1_sigma2_45deg_s_525um_j] = float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row['correlation'])
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_matrix[full_advanced_refl_sigma1_sigma2_45deg_s_525um_j, full_advanced_refl_sigma1_sigma2_45deg_s_525um_i] = float(full_advanced_refl_sigma1_sigma2_45deg_s_525um_row['correlation'])
for full_advanced_refl_sigma1_sigma2_45deg_s_525um_diag in range(len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels)):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_matrix[full_advanced_refl_sigma1_sigma2_45deg_s_525um_diag, full_advanced_refl_sigma1_sigma2_45deg_s_525um_diag] = 1.0

full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig, full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_image = full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax.imshow(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax.set_title(full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_title + ' [correlation]')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax.set_xticks(range(len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels)))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax.set_yticks(range(len(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels)))
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_refl_sigma1_sigma2_45deg_s_525um_label, full_advanced_refl_sigma1_sigma2_45deg_s_525um_label) for full_advanced_refl_sigma1_sigma2_45deg_s_525um_label in full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels], rotation=45, ha='right')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_refl_sigma1_sigma2_45deg_s_525um_label, full_advanced_refl_sigma1_sigma2_45deg_s_525um_label) for full_advanced_refl_sigma1_sigma2_45deg_s_525um_label in full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_labels])
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig.colorbar(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_image, ax=full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_png = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_corr.png"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_pdf = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_corr.pdf"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_png, dpi=220)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_pdf)

full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig, full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_refl_sigma1_sigma2_45deg_s_525um_axis, full_advanced_refl_sigma1_sigma2_45deg_s_525um_image_path, full_advanced_refl_sigma1_sigma2_45deg_s_525um_panel_title in zip(
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_axes,
    (full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_png, full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_png, full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_image = plt.imread(str(full_advanced_refl_sigma1_sigma2_45deg_s_525um_image_path))
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_axis.imshow(full_advanced_refl_sigma1_sigma2_45deg_s_525um_image)
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_axis.set_title(full_advanced_refl_sigma1_sigma2_45deg_s_525um_panel_title)
    full_advanced_refl_sigma1_sigma2_45deg_s_525um_axis.axis('off')
full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig.suptitle(full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_png = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_triptych.png"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_pdf = full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir / f"{full_advanced_refl_sigma1_sigma2_45deg_s_525um_study_dir.name}__section_triptych.pdf"
full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_png, dpi=220)
full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig.savefig(full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_pdf)

display(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig)
display(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig)
display(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig)
display(full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig)
plt.close(full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_fig)
plt.close(full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_fig)
plt.close(full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_fig)
plt.close(full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_fig)

full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_paths = {
    'linear_png': full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_refl_sigma1_sigma2_45deg_s_525um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_refl_sigma1_sigma2_45deg_s_525um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_refl_sigma1_sigma2_45deg_s_525um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_refl_sigma1_sigma2_45deg_s_525um_triptych_pdf.resolve().as_posix(),
}
full_advanced_refl_sigma1_sigma2_45deg_s_525um_plot_paths

### Two-Drude reflection: $\tau_1$ vs $\tau_2$ (60 deg, p-pol, $d_{\mathrm{sub}}=725\,\mu$m, $d_{\mathrm{epi}}=50\,\mu$m)

In [ ]:
full_advanced_refl_tau1_tau2_60deg_p_725um_spec = {
  "slug": "advanced_refl_tau1_tau2_60deg_p_725um",
  "title": "Two-Drude reflection: $\\tau_1$ vs $\\tau_2$ (60 deg, p-pol, $d_{\\mathrm{sub}}=725\\,\\mu$m, $d_{\\mathrm{epi}}=50\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 60.0,
  "polarization": "p",
  "map_kind": "tau",
  "substrate_thickness_um": 725.0,
  "epi_thickness_um": 50.0,
  "tau_range": [
    0.1,
    1.0
  ],
  "sigma_fixed": 316.2,
  "x_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ],
  "y_values": [
    0.1,
    0.1375,
    0.175,
    0.2125,
    0.25,
    0.2875,
    0.32499999999999996,
    0.36250000000000004,
    0.4,
    0.4375,
    0.475,
    0.5125,
    0.5499999999999999,
    0.5875,
    0.625,
    0.6625,
    0.7,
    0.7374999999999999,
    0.7749999999999999,
    0.8125,
    0.85,
    0.8875,
    0.9249999999999999,
    0.9624999999999999,
    1.0
  ]
}
full_advanced_refl_tau1_tau2_60deg_p_725um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_refl_tau1_tau2_60deg_p_725um'
full_advanced_refl_tau1_tau2_60deg_p_725um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_refl_tau1_tau2_60deg_p_725um_result = run_lecture_map_from_spec(full_advanced_refl_tau1_tau2_60deg_p_725um_spec, output_root=full_advanced_refl_tau1_tau2_60deg_p_725um_output_root, profile='full')
full_advanced_refl_tau1_tau2_60deg_p_725um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir = Path(full_advanced_refl_tau1_tau2_60deg_p_725um_result['study_dir'])
full_advanced_refl_tau1_tau2_60deg_p_725um_plot_title = full_advanced_refl_tau1_tau2_60deg_p_725um_result['title']
full_advanced_refl_tau1_tau2_60deg_p_725um_x_label = "$\\tau_1$ (ps)"
full_advanced_refl_tau1_tau2_60deg_p_725um_y_label = "$\\tau_2$ (ps)"

with (full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_refl_tau1_tau2_60deg_p_725um_summary_handle:
    full_advanced_refl_tau1_tau2_60deg_p_725um_summary_meta = json.load(full_advanced_refl_tau1_tau2_60deg_p_725um_summary_handle)
with (full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_refl_tau1_tau2_60deg_p_725um_corr_handle:
    full_advanced_refl_tau1_tau2_60deg_p_725um_corr_meta = json.load(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_handle)

full_advanced_refl_tau1_tau2_60deg_p_725um_rows = []
with (full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_refl_tau1_tau2_60deg_p_725um_summary_csv:
    full_advanced_refl_tau1_tau2_60deg_p_725um_reader = csv.DictReader(full_advanced_refl_tau1_tau2_60deg_p_725um_summary_csv)
    for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_reader:
        full_advanced_refl_tau1_tau2_60deg_p_725um_parsed = {}
        for full_advanced_refl_tau1_tau2_60deg_p_725um_key, full_advanced_refl_tau1_tau2_60deg_p_725um_value in full_advanced_refl_tau1_tau2_60deg_p_725um_row.items():
            if full_advanced_refl_tau1_tau2_60deg_p_725um_value is None:
                full_advanced_refl_tau1_tau2_60deg_p_725um_parsed[full_advanced_refl_tau1_tau2_60deg_p_725um_key] = full_advanced_refl_tau1_tau2_60deg_p_725um_value
                continue
            full_advanced_refl_tau1_tau2_60deg_p_725um_text = str(full_advanced_refl_tau1_tau2_60deg_p_725um_value).strip()
            if full_advanced_refl_tau1_tau2_60deg_p_725um_text == '':
                full_advanced_refl_tau1_tau2_60deg_p_725um_parsed[full_advanced_refl_tau1_tau2_60deg_p_725um_key] = full_advanced_refl_tau1_tau2_60deg_p_725um_text
                continue
            try:
                full_advanced_refl_tau1_tau2_60deg_p_725um_parsed[full_advanced_refl_tau1_tau2_60deg_p_725um_key] = float(full_advanced_refl_tau1_tau2_60deg_p_725um_text)
            except ValueError:
                full_advanced_refl_tau1_tau2_60deg_p_725um_parsed[full_advanced_refl_tau1_tau2_60deg_p_725um_key] = full_advanced_refl_tau1_tau2_60deg_p_725um_text
        full_advanced_refl_tau1_tau2_60deg_p_725um_rows.append(full_advanced_refl_tau1_tau2_60deg_p_725um_parsed)

full_advanced_refl_tau1_tau2_60deg_p_725um_x_key, full_advanced_refl_tau1_tau2_60deg_p_725um_y_key = list(full_advanced_refl_tau1_tau2_60deg_p_725um_summary_meta['recovery_keys'])
full_advanced_refl_tau1_tau2_60deg_p_725um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_refl_tau1_tau2_60deg_p_725um_x_key}_true_minus_fit': f'signed_err_{full_advanced_refl_tau1_tau2_60deg_p_725um_x_key}',
    f'{full_advanced_refl_tau1_tau2_60deg_p_725um_y_key}_true_minus_fit': f'signed_err_{full_advanced_refl_tau1_tau2_60deg_p_725um_y_key}',
    f'abs_{full_advanced_refl_tau1_tau2_60deg_p_725um_x_key}_error': f'abs_err_{full_advanced_refl_tau1_tau2_60deg_p_725um_x_key}',
    f'abs_{full_advanced_refl_tau1_tau2_60deg_p_725um_y_key}_error': f'abs_err_{full_advanced_refl_tau1_tau2_60deg_p_725um_y_key}',
}
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_value_key = full_advanced_refl_tau1_tau2_60deg_p_725um_metric_options['data_fit']
full_advanced_refl_tau1_tau2_60deg_p_725um_log_value_key = full_advanced_refl_tau1_tau2_60deg_p_725um_metric_options['data_fit']
full_advanced_refl_tau1_tau2_60deg_p_725um_x_values = sorted({float(full_advanced_refl_tau1_tau2_60deg_p_725um_row[full_advanced_refl_tau1_tau2_60deg_p_725um_x_key]) for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_rows})
full_advanced_refl_tau1_tau2_60deg_p_725um_y_values = sorted({float(full_advanced_refl_tau1_tau2_60deg_p_725um_row[full_advanced_refl_tau1_tau2_60deg_p_725um_y_key]) for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_rows})

print('Available map metrics:')
for full_advanced_refl_tau1_tau2_60deg_p_725um_option_name, full_advanced_refl_tau1_tau2_60deg_p_725um_option_key in full_advanced_refl_tau1_tau2_60deg_p_725um_metric_options.items():
    print(f"  {full_advanced_refl_tau1_tau2_60deg_p_725um_option_name}: {metric_label(full_advanced_refl_tau1_tau2_60deg_p_725um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_refl_tau1_tau2_60deg_p_725um_log_value_key)}")

def full_advanced_refl_tau1_tau2_60deg_p_725um_aggregate_grid(value_key):
    full_advanced_refl_tau1_tau2_60deg_p_725um_grid = np.full((len(full_advanced_refl_tau1_tau2_60deg_p_725um_y_values), len(full_advanced_refl_tau1_tau2_60deg_p_725um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_refl_tau1_tau2_60deg_p_725um_iy, full_advanced_refl_tau1_tau2_60deg_p_725um_y_value in enumerate(full_advanced_refl_tau1_tau2_60deg_p_725um_y_values):
        for full_advanced_refl_tau1_tau2_60deg_p_725um_ix, full_advanced_refl_tau1_tau2_60deg_p_725um_x_value in enumerate(full_advanced_refl_tau1_tau2_60deg_p_725um_x_values):
            full_advanced_refl_tau1_tau2_60deg_p_725um_cell = [
                float(full_advanced_refl_tau1_tau2_60deg_p_725um_row[value_key])
                for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_rows
                if math.isclose(float(full_advanced_refl_tau1_tau2_60deg_p_725um_row[full_advanced_refl_tau1_tau2_60deg_p_725um_x_key]), full_advanced_refl_tau1_tau2_60deg_p_725um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_refl_tau1_tau2_60deg_p_725um_row[full_advanced_refl_tau1_tau2_60deg_p_725um_y_key]), full_advanced_refl_tau1_tau2_60deg_p_725um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_refl_tau1_tau2_60deg_p_725um_row[value_key]))
            ]
            if full_advanced_refl_tau1_tau2_60deg_p_725um_cell:
                full_advanced_refl_tau1_tau2_60deg_p_725um_grid[full_advanced_refl_tau1_tau2_60deg_p_725um_iy, full_advanced_refl_tau1_tau2_60deg_p_725um_ix] = float(np.mean(full_advanced_refl_tau1_tau2_60deg_p_725um_cell))
    return full_advanced_refl_tau1_tau2_60deg_p_725um_grid

def full_advanced_refl_tau1_tau2_60deg_p_725um_positive_grid_for_log(grid):
    full_advanced_refl_tau1_tau2_60deg_p_725um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_refl_tau1_tau2_60deg_p_725um_finite = full_advanced_refl_tau1_tau2_60deg_p_725um_grid[np.isfinite(full_advanced_refl_tau1_tau2_60deg_p_725um_grid)]
    full_advanced_refl_tau1_tau2_60deg_p_725um_positive = full_advanced_refl_tau1_tau2_60deg_p_725um_finite[full_advanced_refl_tau1_tau2_60deg_p_725um_finite > 0.0]
    if full_advanced_refl_tau1_tau2_60deg_p_725um_positive.size == 0:
        full_advanced_refl_tau1_tau2_60deg_p_725um_grid[np.isfinite(full_advanced_refl_tau1_tau2_60deg_p_725um_grid)] = 1.0
        return full_advanced_refl_tau1_tau2_60deg_p_725um_grid
    full_advanced_refl_tau1_tau2_60deg_p_725um_floor = max(float(np.min(full_advanced_refl_tau1_tau2_60deg_p_725um_positive)) * 0.5, 1e-18)
    full_advanced_refl_tau1_tau2_60deg_p_725um_grid[np.isfinite(full_advanced_refl_tau1_tau2_60deg_p_725um_grid) & (full_advanced_refl_tau1_tau2_60deg_p_725um_grid <= 0.0)] = full_advanced_refl_tau1_tau2_60deg_p_725um_floor
    return full_advanced_refl_tau1_tau2_60deg_p_725um_grid

full_advanced_refl_tau1_tau2_60deg_p_725um_linear_grid = full_advanced_refl_tau1_tau2_60deg_p_725um_aggregate_grid(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_value_key)
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig, full_advanced_refl_tau1_tau2_60deg_p_725um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_finite = full_advanced_refl_tau1_tau2_60deg_p_725um_linear_grid[np.isfinite(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_grid)]
if str(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_value_key).startswith('signed_err_'):
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_limit = max(float(np.max(np.abs(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_finite))), 1e-18)
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmin = -full_advanced_refl_tau1_tau2_60deg_p_725um_linear_limit
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmax = full_advanced_refl_tau1_tau2_60deg_p_725um_linear_limit
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_levels = np.linspace(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmin, full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmax, 256)
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_cmap = 'plasma'
else:
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmin = float(np.min(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_finite))
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmax = float(np.max(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_finite)) + 1e-12
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_levels = np.linspace(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmin, full_advanced_refl_tau1_tau2_60deg_p_725um_linear_vmax, 256)
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_cmap = 'plasma'
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_contour = full_advanced_refl_tau1_tau2_60deg_p_725um_linear_ax.contourf(
    np.asarray(full_advanced_refl_tau1_tau2_60deg_p_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_tau1_tau2_60deg_p_725um_y_values, dtype=np.float64),
    full_advanced_refl_tau1_tau2_60deg_p_725um_linear_grid,
    levels=full_advanced_refl_tau1_tau2_60deg_p_725um_linear_levels,
    cmap=full_advanced_refl_tau1_tau2_60deg_p_725um_linear_cmap,
    extend='both',
)
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_ax.set_title(full_advanced_refl_tau1_tau2_60deg_p_725um_plot_title + ' [linear]')
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_ax.set_xlabel(full_advanced_refl_tau1_tau2_60deg_p_725um_x_label)
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_ax.set_ylabel(full_advanced_refl_tau1_tau2_60deg_p_725um_y_label)
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_cbar = full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig.colorbar(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_contour, ax=full_advanced_refl_tau1_tau2_60deg_p_725um_linear_ax)
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_cbar.set_label(metric_label(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_value_key))
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig.tight_layout()
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_png = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_linear.png"
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_pdf = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_linear.pdf"
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_png, dpi=220)
full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_pdf)

if str(full_advanced_refl_tau1_tau2_60deg_p_725um_log_value_key).startswith('signed_err_'):
    full_advanced_refl_tau1_tau2_60deg_p_725um_log_source_key = 'abs_err_' + str(full_advanced_refl_tau1_tau2_60deg_p_725um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_refl_tau1_tau2_60deg_p_725um_log_source_key = full_advanced_refl_tau1_tau2_60deg_p_725um_log_value_key
full_advanced_refl_tau1_tau2_60deg_p_725um_log_grid = full_advanced_refl_tau1_tau2_60deg_p_725um_positive_grid_for_log(full_advanced_refl_tau1_tau2_60deg_p_725um_aggregate_grid(full_advanced_refl_tau1_tau2_60deg_p_725um_log_source_key))
full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig, full_advanced_refl_tau1_tau2_60deg_p_725um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_tau1_tau2_60deg_p_725um_log_finite = full_advanced_refl_tau1_tau2_60deg_p_725um_log_grid[np.isfinite(full_advanced_refl_tau1_tau2_60deg_p_725um_log_grid)]
full_advanced_refl_tau1_tau2_60deg_p_725um_log_positive = full_advanced_refl_tau1_tau2_60deg_p_725um_log_finite[full_advanced_refl_tau1_tau2_60deg_p_725um_log_finite > 0.0]
full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmin = float(np.min(full_advanced_refl_tau1_tau2_60deg_p_725um_log_positive))
full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmax = float(np.max(full_advanced_refl_tau1_tau2_60deg_p_725um_log_positive))
if math.isclose(full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmin, full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmax):
    full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmax = full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmin * 1.01
full_advanced_refl_tau1_tau2_60deg_p_725um_log_levels = np.geomspace(full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmin, full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmax, 256)
full_advanced_refl_tau1_tau2_60deg_p_725um_log_contour = full_advanced_refl_tau1_tau2_60deg_p_725um_log_ax.contourf(
    np.asarray(full_advanced_refl_tau1_tau2_60deg_p_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_tau1_tau2_60deg_p_725um_y_values, dtype=np.float64),
    full_advanced_refl_tau1_tau2_60deg_p_725um_log_grid,
    levels=full_advanced_refl_tau1_tau2_60deg_p_725um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmin, vmax=full_advanced_refl_tau1_tau2_60deg_p_725um_log_vmax),
    extend='both',
)
full_advanced_refl_tau1_tau2_60deg_p_725um_log_ax.set_title(full_advanced_refl_tau1_tau2_60deg_p_725um_plot_title + ' [log]')
full_advanced_refl_tau1_tau2_60deg_p_725um_log_ax.set_xlabel(full_advanced_refl_tau1_tau2_60deg_p_725um_x_label)
full_advanced_refl_tau1_tau2_60deg_p_725um_log_ax.set_ylabel(full_advanced_refl_tau1_tau2_60deg_p_725um_y_label)
full_advanced_refl_tau1_tau2_60deg_p_725um_log_cbar = full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig.colorbar(full_advanced_refl_tau1_tau2_60deg_p_725um_log_contour, ax=full_advanced_refl_tau1_tau2_60deg_p_725um_log_ax)
full_advanced_refl_tau1_tau2_60deg_p_725um_log_cbar.set_label(metric_label(full_advanced_refl_tau1_tau2_60deg_p_725um_log_source_key))
full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig.tight_layout()
full_advanced_refl_tau1_tau2_60deg_p_725um_log_png = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_log.png"
full_advanced_refl_tau1_tau2_60deg_p_725um_log_pdf = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_log.pdf"
full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_log_png, dpi=220)
full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_log_pdf)

full_advanced_refl_tau1_tau2_60deg_p_725um_corr_rows = full_advanced_refl_tau1_tau2_60deg_p_725um_corr_meta['rows']
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels = sorted({str(full_advanced_refl_tau1_tau2_60deg_p_725um_row['param_i']) for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_corr_rows} | {str(full_advanced_refl_tau1_tau2_60deg_p_725um_row['param_j']) for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_corr_rows})
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_index = {full_advanced_refl_tau1_tau2_60deg_p_725um_label: full_advanced_refl_tau1_tau2_60deg_p_725um_idx for full_advanced_refl_tau1_tau2_60deg_p_725um_idx, full_advanced_refl_tau1_tau2_60deg_p_725um_label in enumerate(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels)}
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_matrix = np.full((len(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels), len(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_refl_tau1_tau2_60deg_p_725um_row in full_advanced_refl_tau1_tau2_60deg_p_725um_corr_rows:
    full_advanced_refl_tau1_tau2_60deg_p_725um_i = full_advanced_refl_tau1_tau2_60deg_p_725um_corr_index[str(full_advanced_refl_tau1_tau2_60deg_p_725um_row['param_i'])]
    full_advanced_refl_tau1_tau2_60deg_p_725um_j = full_advanced_refl_tau1_tau2_60deg_p_725um_corr_index[str(full_advanced_refl_tau1_tau2_60deg_p_725um_row['param_j'])]
    full_advanced_refl_tau1_tau2_60deg_p_725um_corr_matrix[full_advanced_refl_tau1_tau2_60deg_p_725um_i, full_advanced_refl_tau1_tau2_60deg_p_725um_j] = float(full_advanced_refl_tau1_tau2_60deg_p_725um_row['correlation'])
    full_advanced_refl_tau1_tau2_60deg_p_725um_corr_matrix[full_advanced_refl_tau1_tau2_60deg_p_725um_j, full_advanced_refl_tau1_tau2_60deg_p_725um_i] = float(full_advanced_refl_tau1_tau2_60deg_p_725um_row['correlation'])
for full_advanced_refl_tau1_tau2_60deg_p_725um_diag in range(len(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels)):
    full_advanced_refl_tau1_tau2_60deg_p_725um_corr_matrix[full_advanced_refl_tau1_tau2_60deg_p_725um_diag, full_advanced_refl_tau1_tau2_60deg_p_725um_diag] = 1.0

full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig, full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_image = full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax.imshow(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax.set_title(full_advanced_refl_tau1_tau2_60deg_p_725um_plot_title + ' [correlation]')
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax.set_xticks(range(len(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels)))
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax.set_yticks(range(len(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels)))
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_refl_tau1_tau2_60deg_p_725um_label, full_advanced_refl_tau1_tau2_60deg_p_725um_label) for full_advanced_refl_tau1_tau2_60deg_p_725um_label in full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels], rotation=45, ha='right')
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_refl_tau1_tau2_60deg_p_725um_label, full_advanced_refl_tau1_tau2_60deg_p_725um_label) for full_advanced_refl_tau1_tau2_60deg_p_725um_label in full_advanced_refl_tau1_tau2_60deg_p_725um_corr_labels])
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig.colorbar(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_image, ax=full_advanced_refl_tau1_tau2_60deg_p_725um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig.tight_layout()
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_png = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_corr.png"
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_pdf = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_corr.pdf"
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_png, dpi=220)
full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_pdf)

full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig, full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_refl_tau1_tau2_60deg_p_725um_axis, full_advanced_refl_tau1_tau2_60deg_p_725um_image_path, full_advanced_refl_tau1_tau2_60deg_p_725um_panel_title in zip(
    full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_axes,
    (full_advanced_refl_tau1_tau2_60deg_p_725um_linear_png, full_advanced_refl_tau1_tau2_60deg_p_725um_log_png, full_advanced_refl_tau1_tau2_60deg_p_725um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_refl_tau1_tau2_60deg_p_725um_image = plt.imread(str(full_advanced_refl_tau1_tau2_60deg_p_725um_image_path))
    full_advanced_refl_tau1_tau2_60deg_p_725um_axis.imshow(full_advanced_refl_tau1_tau2_60deg_p_725um_image)
    full_advanced_refl_tau1_tau2_60deg_p_725um_axis.set_title(full_advanced_refl_tau1_tau2_60deg_p_725um_panel_title)
    full_advanced_refl_tau1_tau2_60deg_p_725um_axis.axis('off')
full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig.suptitle(full_advanced_refl_tau1_tau2_60deg_p_725um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig.tight_layout()
full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_png = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_triptych.png"
full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_pdf = full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir / f"{full_advanced_refl_tau1_tau2_60deg_p_725um_study_dir.name}__section_triptych.pdf"
full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_png, dpi=220)
full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig.savefig(full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_pdf)

display(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig)
display(full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig)
display(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig)
display(full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig)
plt.close(full_advanced_refl_tau1_tau2_60deg_p_725um_linear_fig)
plt.close(full_advanced_refl_tau1_tau2_60deg_p_725um_log_fig)
plt.close(full_advanced_refl_tau1_tau2_60deg_p_725um_corr_fig)
plt.close(full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_fig)

full_advanced_refl_tau1_tau2_60deg_p_725um_plot_paths = {
    'linear_png': full_advanced_refl_tau1_tau2_60deg_p_725um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_refl_tau1_tau2_60deg_p_725um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_refl_tau1_tau2_60deg_p_725um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_refl_tau1_tau2_60deg_p_725um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_refl_tau1_tau2_60deg_p_725um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_refl_tau1_tau2_60deg_p_725um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_refl_tau1_tau2_60deg_p_725um_triptych_pdf.resolve().as_posix(),
}
full_advanced_refl_tau1_tau2_60deg_p_725um_plot_paths

### Two-Drude reflection: $\sigma_1$ vs $\sigma_2$ (60 deg, s-pol, $d_{\mathrm{sub}}=725\,\mu$m, $d_{\mathrm{epi}}=50\,\mu$m)

In [ ]:
full_advanced_refl_sigma1_sigma2_60deg_s_725um_spec = {
  "slug": "advanced_refl_sigma1_sigma2_60deg_s_725um",
  "title": "Two-Drude reflection: $\\sigma_1$ vs $\\sigma_2$ (60 deg, s-pol, $d_{\\mathrm{sub}}=725\\,\\mu$m, $d_{\\mathrm{epi}}=50\\,\\mu$m)",
  "mode": "reflection",
  "angle_deg": 60.0,
  "polarization": "s",
  "map_kind": "sigma",
  "substrate_thickness_um": 725.0,
  "epi_thickness_um": 50.0,
  "sigma_range": [
    100.0,
    1000.0
  ],
  "tau_fixed": 0.316,
  "x_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ],
  "y_values": [
    100.0,
    137.5,
    175.0,
    212.5,
    250.0,
    287.5,
    325.0,
    362.5,
    400.0,
    437.5,
    475.0,
    512.5,
    550.0,
    587.5,
    625.0,
    662.5,
    700.0,
    737.5,
    775.0,
    812.5,
    850.0,
    887.5,
    925.0,
    962.5,
    1000.0
  ]
}
full_advanced_refl_sigma1_sigma2_60deg_s_725um_output_root = repo_root / 'docs' / 'lecture_build' / 'notebook_section_runs' / 'full_advanced_refl_sigma1_sigma2_60deg_s_725um'
full_advanced_refl_sigma1_sigma2_60deg_s_725um_output_root.mkdir(parents=True, exist_ok=True)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_result = run_lecture_map_from_spec(full_advanced_refl_sigma1_sigma2_60deg_s_725um_spec, output_root=full_advanced_refl_sigma1_sigma2_60deg_s_725um_output_root, profile='full')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_result['study_dir']

This plotting cell is section-local. You can rewrite this block without changing the other advanced study plots.

In [ ]:
full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir = Path(full_advanced_refl_sigma1_sigma2_60deg_s_725um_result['study_dir'])
full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_title = full_advanced_refl_sigma1_sigma2_60deg_s_725um_result['title']
full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_label = "$\\sigma_1$ (S/m)"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_label = "$\\sigma_2$ (S/m)"

with (full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / 'study_summary.json').open('r', encoding='utf-8') as full_advanced_refl_sigma1_sigma2_60deg_s_725um_summary_handle:
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_summary_meta = json.load(full_advanced_refl_sigma1_sigma2_60deg_s_725um_summary_handle)
with (full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / 'averaged_correlation.json').open('r', encoding='utf-8') as full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_handle:
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_meta = json.load(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_handle)

full_advanced_refl_sigma1_sigma2_60deg_s_725um_rows = []
with (full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / 'study_summary.csv').open('r', newline='', encoding='utf-8') as full_advanced_refl_sigma1_sigma2_60deg_s_725um_summary_csv:
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_reader = csv.DictReader(full_advanced_refl_sigma1_sigma2_60deg_s_725um_summary_csv)
    for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_reader:
        full_advanced_refl_sigma1_sigma2_60deg_s_725um_parsed = {}
        for full_advanced_refl_sigma1_sigma2_60deg_s_725um_key, full_advanced_refl_sigma1_sigma2_60deg_s_725um_value in full_advanced_refl_sigma1_sigma2_60deg_s_725um_row.items():
            if full_advanced_refl_sigma1_sigma2_60deg_s_725um_value is None:
                full_advanced_refl_sigma1_sigma2_60deg_s_725um_parsed[full_advanced_refl_sigma1_sigma2_60deg_s_725um_key] = full_advanced_refl_sigma1_sigma2_60deg_s_725um_value
                continue
            full_advanced_refl_sigma1_sigma2_60deg_s_725um_text = str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_value).strip()
            if full_advanced_refl_sigma1_sigma2_60deg_s_725um_text == '':
                full_advanced_refl_sigma1_sigma2_60deg_s_725um_parsed[full_advanced_refl_sigma1_sigma2_60deg_s_725um_key] = full_advanced_refl_sigma1_sigma2_60deg_s_725um_text
                continue
            try:
                full_advanced_refl_sigma1_sigma2_60deg_s_725um_parsed[full_advanced_refl_sigma1_sigma2_60deg_s_725um_key] = float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_text)
            except ValueError:
                full_advanced_refl_sigma1_sigma2_60deg_s_725um_parsed[full_advanced_refl_sigma1_sigma2_60deg_s_725um_key] = full_advanced_refl_sigma1_sigma2_60deg_s_725um_text
        full_advanced_refl_sigma1_sigma2_60deg_s_725um_rows.append(full_advanced_refl_sigma1_sigma2_60deg_s_725um_parsed)

full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key, full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key = list(full_advanced_refl_sigma1_sigma2_60deg_s_725um_summary_meta['recovery_keys'])
full_advanced_refl_sigma1_sigma2_60deg_s_725um_metric_options = {
    'data_fit': 'data_fit',
    'weighted_data_fit': 'weighted_data_fit',
    f'{full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key}_true_minus_fit': f'signed_err_{full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key}',
    f'{full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key}_true_minus_fit': f'signed_err_{full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key}',
    f'abs_{full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key}_error': f'abs_err_{full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key}',
    f'abs_{full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key}_error': f'abs_err_{full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key}',
}
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_value_key = full_advanced_refl_sigma1_sigma2_60deg_s_725um_metric_options['data_fit']
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_value_key = full_advanced_refl_sigma1_sigma2_60deg_s_725um_metric_options['data_fit']
full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_values = sorted({float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row[full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key]) for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_rows})
full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_values = sorted({float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row[full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key]) for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_rows})

print('Available map metrics:')
for full_advanced_refl_sigma1_sigma2_60deg_s_725um_option_name, full_advanced_refl_sigma1_sigma2_60deg_s_725um_option_key in full_advanced_refl_sigma1_sigma2_60deg_s_725um_metric_options.items():
    print(f"  {full_advanced_refl_sigma1_sigma2_60deg_s_725um_option_name}: {metric_label(full_advanced_refl_sigma1_sigma2_60deg_s_725um_option_key)}")
print(f"Current linear map: {metric_label(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_value_key)}")
print(f"Current log map: {metric_label(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_value_key)}")

def full_advanced_refl_sigma1_sigma2_60deg_s_725um_aggregate_grid(value_key):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid = np.full((len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_values), len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_values)), np.nan, dtype=np.float64)
    for full_advanced_refl_sigma1_sigma2_60deg_s_725um_iy, full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_value in enumerate(full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_values):
        for full_advanced_refl_sigma1_sigma2_60deg_s_725um_ix, full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_value in enumerate(full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_values):
            full_advanced_refl_sigma1_sigma2_60deg_s_725um_cell = [
                float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row[value_key])
                for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_rows
                if math.isclose(float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row[full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_key]), full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_value, rel_tol=0.0, abs_tol=1e-12)
                and math.isclose(float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row[full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_key]), full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_value, rel_tol=0.0, abs_tol=1e-12)
                and np.isfinite(float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row[value_key]))
            ]
            if full_advanced_refl_sigma1_sigma2_60deg_s_725um_cell:
                full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid[full_advanced_refl_sigma1_sigma2_60deg_s_725um_iy, full_advanced_refl_sigma1_sigma2_60deg_s_725um_ix] = float(np.mean(full_advanced_refl_sigma1_sigma2_60deg_s_725um_cell))
    return full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid

def full_advanced_refl_sigma1_sigma2_60deg_s_725um_positive_grid_for_log(grid):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid = np.asarray(grid, dtype=np.float64).copy()
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_finite = full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid)]
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_positive = full_advanced_refl_sigma1_sigma2_60deg_s_725um_finite[full_advanced_refl_sigma1_sigma2_60deg_s_725um_finite > 0.0]
    if full_advanced_refl_sigma1_sigma2_60deg_s_725um_positive.size == 0:
        full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid)] = 1.0
        return full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_floor = max(float(np.min(full_advanced_refl_sigma1_sigma2_60deg_s_725um_positive)) * 0.5, 1e-18)
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid) & (full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid <= 0.0)] = full_advanced_refl_sigma1_sigma2_60deg_s_725um_floor
    return full_advanced_refl_sigma1_sigma2_60deg_s_725um_grid

full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_grid = full_advanced_refl_sigma1_sigma2_60deg_s_725um_aggregate_grid(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_value_key)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig, full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_finite = full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_grid)]
if str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_value_key).startswith('signed_err_'):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_limit = max(float(np.max(np.abs(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_finite))), 1e-18)
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmin = -full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_limit
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmax = full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_limit
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_levels = np.linspace(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmin, full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmax, 256)
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_cmap = 'plasma'
else:
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmin = float(np.min(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_finite))
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmax = float(np.max(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_finite)) + 1e-12
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_levels = np.linspace(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmin, full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_vmax, 256)
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_cmap = 'plasma'
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_contour = full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_ax.contourf(
    np.asarray(full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_values, dtype=np.float64),
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_grid,
    levels=full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_levels,
    cmap=full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_cmap,
    extend='both',
)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_ax.set_title(full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_title + ' [linear]')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_ax.set_xlabel(full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_label)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_ax.set_ylabel(full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_label)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_cbar = full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig.colorbar(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_contour, ax=full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_ax)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_cbar.set_label(metric_label(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_value_key))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_png = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_linear.png"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_pdf = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_linear.pdf"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_png, dpi=220)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_pdf)

if str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_value_key).startswith('signed_err_'):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_source_key = 'abs_err_' + str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_value_key)[len('signed_err_'):]
else:
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_source_key = full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_value_key
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_grid = full_advanced_refl_sigma1_sigma2_60deg_s_725um_positive_grid_for_log(full_advanced_refl_sigma1_sigma2_60deg_s_725um_aggregate_grid(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_source_key))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig, full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_ax = plt.subplots(figsize=(5.8, 4.6))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_finite = full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_grid[np.isfinite(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_grid)]
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_positive = full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_finite[full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_finite > 0.0]
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmin = float(np.min(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_positive))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmax = float(np.max(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_positive))
if math.isclose(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmin, full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmax):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmax = full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmin * 1.01
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_levels = np.geomspace(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmin, full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmax, 256)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_contour = full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_ax.contourf(
    np.asarray(full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_values, dtype=np.float64),
    np.asarray(full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_values, dtype=np.float64),
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_grid,
    levels=full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_levels,
    cmap='plasma',
    norm=mcolors.LogNorm(vmin=full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmin, vmax=full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_vmax),
    extend='both',
)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_ax.set_title(full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_title + ' [log]')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_ax.set_xlabel(full_advanced_refl_sigma1_sigma2_60deg_s_725um_x_label)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_ax.set_ylabel(full_advanced_refl_sigma1_sigma2_60deg_s_725um_y_label)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_cbar = full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig.colorbar(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_contour, ax=full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_ax)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_cbar.set_label(metric_label(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_source_key))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_png = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_log.png"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_pdf = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_log.pdf"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_png, dpi=220)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_pdf)

full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_rows = full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_meta['rows']
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels = sorted({str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row['param_i']) for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_rows} | {str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row['param_j']) for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_rows})
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_index = {full_advanced_refl_sigma1_sigma2_60deg_s_725um_label: full_advanced_refl_sigma1_sigma2_60deg_s_725um_idx for full_advanced_refl_sigma1_sigma2_60deg_s_725um_idx, full_advanced_refl_sigma1_sigma2_60deg_s_725um_label in enumerate(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels)}
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_matrix = np.full((len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels), len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels)), np.nan, dtype=np.float64)
for full_advanced_refl_sigma1_sigma2_60deg_s_725um_row in full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_rows:
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_i = full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_index[str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row['param_i'])]
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_j = full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_index[str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row['param_j'])]
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_matrix[full_advanced_refl_sigma1_sigma2_60deg_s_725um_i, full_advanced_refl_sigma1_sigma2_60deg_s_725um_j] = float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row['correlation'])
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_matrix[full_advanced_refl_sigma1_sigma2_60deg_s_725um_j, full_advanced_refl_sigma1_sigma2_60deg_s_725um_i] = float(full_advanced_refl_sigma1_sigma2_60deg_s_725um_row['correlation'])
for full_advanced_refl_sigma1_sigma2_60deg_s_725um_diag in range(len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels)):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_matrix[full_advanced_refl_sigma1_sigma2_60deg_s_725um_diag, full_advanced_refl_sigma1_sigma2_60deg_s_725um_diag] = 1.0

full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig, full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax = plt.subplots(figsize=(4.8, 4.2))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_image = full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax.imshow(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_matrix, cmap='plasma', vmin=-1.0, vmax=1.0)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax.set_title(full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_title + ' [correlation]')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax.set_xticks(range(len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels)))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax.set_yticks(range(len(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels)))
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax.set_xticklabels([PARAMETER_LABELS.get(full_advanced_refl_sigma1_sigma2_60deg_s_725um_label, full_advanced_refl_sigma1_sigma2_60deg_s_725um_label) for full_advanced_refl_sigma1_sigma2_60deg_s_725um_label in full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels], rotation=45, ha='right')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax.set_yticklabels([PARAMETER_LABELS.get(full_advanced_refl_sigma1_sigma2_60deg_s_725um_label, full_advanced_refl_sigma1_sigma2_60deg_s_725um_label) for full_advanced_refl_sigma1_sigma2_60deg_s_725um_label in full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_labels])
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig.colorbar(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_image, ax=full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_ax, label=r'$\rho_{ij}$')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_png = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_corr.png"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_pdf = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_corr.pdf"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_png, dpi=220)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_pdf)

full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig, full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_axes = plt.subplots(1, 3, figsize=(14.2, 4.5))
for full_advanced_refl_sigma1_sigma2_60deg_s_725um_axis, full_advanced_refl_sigma1_sigma2_60deg_s_725um_image_path, full_advanced_refl_sigma1_sigma2_60deg_s_725um_panel_title in zip(
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_axes,
    (full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_png, full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_png, full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_png),
    ('Linear scale', 'Log scale', 'Average fit correlation'),
):
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_image = plt.imread(str(full_advanced_refl_sigma1_sigma2_60deg_s_725um_image_path))
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_axis.imshow(full_advanced_refl_sigma1_sigma2_60deg_s_725um_image)
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_axis.set_title(full_advanced_refl_sigma1_sigma2_60deg_s_725um_panel_title)
    full_advanced_refl_sigma1_sigma2_60deg_s_725um_axis.axis('off')
full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig.suptitle(full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_title, fontsize=14, fontweight='bold', y=1.02)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig.tight_layout()
full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_png = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_triptych.png"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_pdf = full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir / f"{full_advanced_refl_sigma1_sigma2_60deg_s_725um_study_dir.name}__section_triptych.pdf"
full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_png, dpi=220)
full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig.savefig(full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_pdf)

display(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig)
display(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig)
display(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig)
display(full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig)
plt.close(full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_fig)
plt.close(full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_fig)
plt.close(full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_fig)
plt.close(full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_fig)

full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_paths = {
    'linear_png': full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_png.resolve().as_posix(),
    'linear_pdf': full_advanced_refl_sigma1_sigma2_60deg_s_725um_linear_pdf.resolve().as_posix(),
    'log_png': full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_png.resolve().as_posix(),
    'log_pdf': full_advanced_refl_sigma1_sigma2_60deg_s_725um_log_pdf.resolve().as_posix(),
    'corr_png': full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_png.resolve().as_posix(),
    'corr_pdf': full_advanced_refl_sigma1_sigma2_60deg_s_725um_corr_pdf.resolve().as_posix(),
    'triptych_png': full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_png.resolve().as_posix(),
    'triptych_pdf': full_advanced_refl_sigma1_sigma2_60deg_s_725um_triptych_pdf.resolve().as_posix(),
}
full_advanced_refl_sigma1_sigma2_60deg_s_725um_plot_paths